# XAUUSD — single canonical Colab notebook

This notebook contains all scientific stages and writes their modules at runtime. Only this one notebook is distributed as scientific source code.

The archived Excel results in `outputs/` are preserved byte-for-byte from their historical exports. The present notebook has label and source-metadata normalization for publication; **a complete numerical rerun of this exact edited notebook has not been performed**. The existing outputs must not be described as newly generated by this edited notebook.

## Inputs

Place `OHLCV15MIN.csv`, `DTWEXBGS.csv`, and `VIXCLS.csv` in `MyDrive/xauusd_data` or upload them when prompted. The economic calendar file is optional; see `data/README.md` in the repository.

## Running

Open in Google Colab, use a CPU runtime, and run all cells. Results and checkpoints go to `MyDrive/XAUUSD_COLAB_RUN` by default. A new execution must use its own output folder rather than overwrite the archived historical results.


In [ ]:
# Colab setup, input data and environment checks
import os
import sys
import json
import hashlib
import shutil
import subprocess
import importlib.metadata
import math
from pathlib import Path

# Input data: put these CSV files in one folder of YOUR Google Drive (or upload them when asked).
DATA_FOLDER = "/content/drive/MyDrive/xauusd_data"
REQUIRED_FILES = ("OHLCV15MIN.csv", "DTWEXBGS.csv", "VIXCLS.csv")
OPTIONAL_MACRO_FILE = "economic_calendar_publication_master.csv"

# Where results and checkpoints are written. Reuse the same folder to resume a run.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/XAUUSD_COLAB_RUN"
LOCAL_PROJECT_DIR = "/content/xauusd_scientific_pipeline"

COLAB_MAX_CPU_WORKERS = 3

for key in tuple(os.environ):
    if (key.startswith("XAUUSD_STAGE6") or key.startswith("XAUUSD_STAGE7")
        or key.startswith("XAUUSD_STAGE8") or key in {
        "XAUUSD_N_SURROGATES", "XAUUSD_N_IAAFT_SURROGATES", "XAUUSD_MS_SEARCH_REPS",
        "XAUUSD_SAVE_DFA_YEAR_FIGS"}):
        os.environ.pop(key, None)

for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "ruptures", "openpyxl", "tabulate", "joblib", "scipy",
                "statsmodels", "matplotlib", "numba", "psutil"], check=True)

try:
    from google.colab import drive, files
except ImportError as exc:
    raise RuntimeError("This notebook is meant to run in Google Colab.") from exc

drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = Path(LOCAL_PROJECT_DIR)
DATA_DIR = PROJECT_DIR / "input_data"
DRIVE_ROOT = Path(DRIVE_OUTPUT_DIR)
CHECKPOINTS = DRIVE_ROOT / "checkpoints"
RESULTS = DRIVE_ROOT / "results"
CONTROL_DIR = DRIVE_ROOT / "control"
for folder in (PROJECT_DIR, DATA_DIR, DRIVE_ROOT, CHECKPOINTS, RESULTS, CONTROL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SCIENTIFIC_PACKAGES = ("numpy", "pandas", "scipy", "statsmodels", "numba",
                       "ruptures", "joblib", "openpyxl", "matplotlib", "tabulate")
def environment_details():
    return {
        "python": sys.version.split()[0],
        "packages": {name: importlib.metadata.version(name) for name in SCIENTIFIC_PACKAGES},
    }
environment_lock_path = CONTROL_DIR / "scientific_environment_lock.json"
environment_snapshot = environment_details()
if environment_lock_path.exists():
    previous_environment = json.loads(environment_lock_path.read_text(encoding="utf-8"))
    if previous_environment != environment_snapshot:
        raise RuntimeError(
            "Colab changed Python or a scientific dependency since this run. "
            "Checkpoints should not be mixed across environments. Restore the "
            f"versions in {environment_lock_path} or choose a NEW DRIVE_OUTPUT_DIR. "
            f"Expected {previous_environment}; got {environment_snapshot}"
        )
else:
    tmp=environment_lock_path.with_name(environment_lock_path.name+".partial")
    tmp.write_text(json.dumps(environment_snapshot, indent=2), encoding="utf-8")
    os.replace(tmp,environment_lock_path)
print(f"Package versions saved to {environment_lock_path}")

# joblib scratch space stays on the local disk, not on Drive
LOCAL_SCRATCH = PROJECT_DIR / "joblib_scratch"
LOCAL_SCRATCH.mkdir(parents=True,exist_ok=True)
os.environ["JOBLIB_TEMP_FOLDER"] = str(LOCAL_SCRATCH)

import psutil
def effective_cpu_count():
    counts=[os.cpu_count() or 1]
    try: counts.append(len(os.sched_getaffinity(0)))
    except (AttributeError, OSError): pass
    try:
        q,p=Path("/sys/fs/cgroup/cpu.max").read_text().strip().split()
        if q != "max": counts.append(max(1,math.floor(int(q)/int(p))))
    except (FileNotFoundError, ValueError, OSError): pass
    return max(1,min(counts))

def cgroup_ram_total():
    limits=[psutil.virtual_memory().total]
    for fname in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            v=Path(fname).read_text().strip()
            if v.isdigit() and 0 < int(v) < 1 << 48: limits.append(int(v))
        except OSError: pass
    return min(limits)

CPU_CAPACITY = effective_cpu_count()
RAM_TOTAL_GIB = cgroup_ram_total()/1024**3
recommended_workers = min(COLAB_MAX_CPU_WORKERS, CPU_CAPACITY,
                          3 if RAM_TOTAL_GIB >= 11 else 2 if RAM_TOTAL_GIB >= 7 else 1)
worker_plan_path = CONTROL_DIR / "cpu_worker_plan.json"
if worker_plan_path.exists():
    plan=json.loads(worker_plan_path.read_text(encoding="utf-8"))
    n_workers=int(plan["workers"])
    if n_workers>recommended_workers:
        raise RuntimeError(
            f"This session safely supports {recommended_workers} workers but the scientific run "
            f"was started with {n_workers}. Use a runtime with sufficient CPU/RAM or "
            "a NEW DRIVE_OUTPUT_DIR; do not silently change the parallelism mid-run."
        )
else:
    n_workers=recommended_workers
    plan={"workers":n_workers,"max_requested":COLAB_MAX_CPU_WORKERS,
          "effective_cpu_at_start":CPU_CAPACITY,"ram_gib_at_start":round(RAM_TOTAL_GIB,2)}
    tmp=worker_plan_path.with_name(worker_plan_path.name+".partial")
    tmp.write_text(json.dumps(plan,indent=2),encoding="utf-8")
    os.replace(tmp,worker_plan_path)
print(f"[CPU] CPU effective={CPU_CAPACITY}; RAM limit={RAM_TOTAL_GIB:.1f} GiB; "
      f"workers={n_workers}; ")

# Point config.py's checkpoint folder at Drive so it survives Colab resets
cp_link = PROJECT_DIR / "checkpoints"
if cp_link.is_symlink() and cp_link.resolve() != CHECKPOINTS.resolve():
    raise RuntimeError(f"Unexpected checkpoint symlink: {cp_link} -> {cp_link.resolve()}")
if cp_link.exists() and not cp_link.is_symlink():
    raise RuntimeError(f"{cp_link} is not a symlink. Do not mix local and Drive checkpoints.")
if not cp_link.exists() and not cp_link.is_symlink():
    cp_link.symlink_to(CHECKPOINTS, target_is_directory=True)

# Data files are copied to a fixed local path with a fixed modification time,
# so the checkpoint fingerprints stay the same when a run is resumed.
FIXED_MTIME_NS = 1_700_000_000 * 1_000_000_000


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fp:
        for chunk in iter(lambda: fp.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


source_dir = Path(DATA_FOLDER)
missing = [f for f in REQUIRED_FILES if not (source_dir / f).is_file() and not (DATA_DIR / f).is_file()]
if missing:
    print(f"Not found in {source_dir}: {missing}")
    print("Select the missing CSV files from your computer.")
    uploaded = files.upload()
    for name, content in uploaded.items():
        (DATA_DIR / name).write_bytes(content)
    missing = [f for f in REQUIRED_FILES if not (source_dir / f).is_file() and not (DATA_DIR / f).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing input files: {missing}")

file_info = {}
for filename in (*REQUIRED_FILES, OPTIONAL_MACRO_FILE):
    src = source_dir / filename
    target = DATA_DIR / filename
    if src.is_file():
        shutil.copyfile(src, target)
    if not target.is_file():
        continue
    os.utime(target, ns=(FIXED_MTIME_NS, FIXED_MTIME_NS))
    file_info[filename] = {
        "size_bytes": target.stat().st_size,
        "sha256": sha256_file(target),
        "local_absolute_path": str(target.resolve()),
        "local_mtime_ns": target.stat().st_mtime_ns,
    }
    print(f"{filename}: {target.stat().st_size:,} bytes, SHA256={file_info[filename]['sha256']}")

os.environ["XAUUSD_OHLCV_PATH"] = str(DATA_DIR / "OHLCV15MIN.csv")
os.environ["XAUUSD_USD_PATH"] = str(DATA_DIR / "DTWEXBGS.csv")
os.environ["XAUUSD_VIX_PATH"] = str(DATA_DIR / "VIXCLS.csv")
os.environ["XAUUSD_OUTPUT_DIR"] = str(RESULTS)
os.environ["XAUUSD_N_JOBS"] = str(n_workers)
if OPTIONAL_MACRO_FILE in file_info:
    os.environ["XAUUSD_MACRO_CALENDAR_PATH"] = str(DATA_DIR / OPTIONAL_MACRO_FILE)
else:
    os.environ.pop("XAUUSD_MACRO_CALENDAR_PATH", None)

print(f"Results: {RESULTS}")
print(f"Checkpoints: {CHECKPOINTS}")


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/config.py
# -*- coding: utf-8 -*-
"""Shared configuration and utilities for the XAUUSD pipeline."""
from __future__ import annotations

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import sys
import json
import hashlib
import datetime as _dt
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from scipy.stats.mstats import winsorize
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression
import statsmodels.api as sm
import ruptures as rpt
from joblib import Parallel, delayed

PIPELINE_VERSION = "XAUUSD_PIPELINE_v1"
BASE_DIR = Path(__file__).resolve().parent
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = BASE_DIR / "input_data"
OHLCV_PATH = Path(os.getenv("XAUUSD_OHLCV_PATH", str(DATASET_DIR / "OHLCV15MIN.csv")))
USD_PATH = Path(os.getenv("XAUUSD_USD_PATH", str(DATASET_DIR / "DTWEXBGS.csv")))
VIX_PATH = Path(os.getenv("XAUUSD_VIX_PATH", str(DATASET_DIR / "VIXCLS.csv")))
OUTPUT_DIR = Path(os.getenv("XAUUSD_OUTPUT_DIR", str(BASE_DIR / "results")))

FIG_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "stage_logs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

FINAL_EXCEL = OUTPUT_DIR / "statistical_results.xlsx"
DAILY_CSV = BASE_DIR / "daily.csv"
ROLLING_CSV = BASE_DIR / "rolling_df.csv"
STATUS_JSON = CHECKPOINT_DIR / "pipeline_status.json"

STUDY_END_DATE = pd.Timestamp("2026-07-31 23:59:59")
RANDOM_SEED = 42
ROLLING_WINDOWS = (30, 45, 60, 90, 120)
PRIMARY_WINDOW = 60
DFA_R2_MIN = 0.95
EXPECTED_BAR_MINUTES = 15
YZ_MULTIDAY_WINDOW = 22
EXTREME_RETURN_THRESHOLD = 0.0020

PELT_MIN_SIZES = (15, 30, 60, 90)
PELT_PENALTY_MULTIPLIERS = (0.5, 0.75, 1.0, 1.5, 2.0, 3.0)
PELT_CLUSTER_TOL_DAYS = 15
PELT_CONSENSUS_SUPPORT = 0.50
MS_SEARCH_REPS = int(os.getenv("XAUUSD_MS_SEARCH_REPS", "20"))

OOS_N_FOLDS = 4
OOS_FIRST_TEST_FRACTION = 0.60
MICROSTRUCTURE_WINDOWS = (60, 90)

MF_Q_VALUES = (-5, -3, -1, 0, 1, 2, 3, 5)
MF_VARIANCE_FLOOR = 1e-12
MF_FLOOR_SENSITIVITY = (1e-10, 1e-12, 1e-14)
N_SURROGATES = int(os.getenv("XAUUSD_N_SURROGATES", "99"))
N_IAAFT_SURROGATES = int(os.getenv("XAUUSD_N_IAAFT_SURROGATES", "39"))
IAAFT_MAX_ITER = 100
IAAFT_CONVERGENCE_TOL = 1e-8
SURROGATE_BATCH_SIZE = 4

N_JOBS = int(os.getenv("XAUUSD_N_JOBS", "4"))
N_JOBS = max(1, min(N_JOBS, os.cpu_count() or 1))
JOBLIB_VERBOSE = int(os.getenv("XAUUSD_JOBLIB_VERBOSE", "0"))

DATA_VENDOR = os.getenv("XAUUSD_DATA_VENDOR", "Dukascopy")
SOURCE_TIMEZONE = os.getenv("XAUUSD_SOURCE_TIMEZONE", "EST+07")
SESSION_DEFINITION = os.getenv(
    "XAUUSD_SESSION_DEFINITION",
    "calendar_day_from_source_file_timestamp"
)

SAVE_APPENDIX_DFA_FIGURES = os.getenv("XAUUSD_SAVE_DFA_YEAR_FIGS", "0") == "1"

class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()
    def flush(self):
        for s in self.streams:
            s.flush()

@contextmanager
def stage_logging(stage_name: str):
    log_path = LOG_DIR / f"{stage_name}.log"
    original = sys.stdout
    f = open(log_path, "a", encoding="utf-8")
    sys.stdout = _Tee(original, f)
    try:
        print("\n" + "=" * 100, flush=True)
        print(f"[{stage_name}] start: {_dt.datetime.now().isoformat(timespec='seconds')}", flush=True)
        print(f"Pipeline: {PIPELINE_VERSION} | N_JOBS={N_JOBS}", flush=True)
        yield log_path
        print(f"[{stage_name}] end: {_dt.datetime.now().isoformat(timespec='seconds')}", flush=True)
    finally:
        sys.stdout = original
        f.flush(); f.close()


def _source_signature(path: Path):
    try:
        st = path.stat()
        return {"path": str(path.resolve()), "size": st.st_size, "mtime_ns": st.st_mtime_ns}
    except FileNotFoundError:
        return {"path": str(path), "missing": True}


def _hash_dict(d):
    raw = json.dumps(d, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def config_fingerprint(scope="all") -> str:
    base1 = {
        "version": PIPELINE_VERSION,
        "ohlcv": _source_signature(OHLCV_PATH),
        "study_end": str(STUDY_END_DATE),
        "expected_bar_minutes": EXPECTED_BAR_MINUTES,
        "yz_multiday_window": YZ_MULTIDAY_WINDOW,
        "extreme_threshold": EXTREME_RETURN_THRESHOLD,
    }
    if scope == "stage1": return _hash_dict(base1)
    base2 = {**base1, "pelt_min_sizes": PELT_MIN_SIZES, "pelt_penalties": PELT_PENALTY_MULTIPLIERS,
             "pelt_tol": PELT_CLUSTER_TOL_DAYS, "pelt_support": PELT_CONSENSUS_SUPPORT,
             "ms_search_reps": MS_SEARCH_REPS, "random_seed": RANDOM_SEED}
    if scope == "stage2": return _hash_dict(base2)
    base3 = {**base2, "rolling_windows": ROLLING_WINDOWS, "primary_window": PRIMARY_WINDOW,
             "dfa_r2_min": DFA_R2_MIN, "oos_folds": OOS_N_FOLDS, "oos_first": OOS_FIRST_TEST_FRACTION,
             "usd": _source_signature(USD_PATH), "vix": _source_signature(VIX_PATH)}
    if scope == "stage3": return _hash_dict(base3)
    base4 = {**base3, "mf_q": MF_Q_VALUES, "mf_floor": MF_VARIANCE_FLOOR,
             "mf_floor_sens": MF_FLOOR_SENSITIVITY, "n_shuffle": N_SURROGATES, "n_iaaft": N_IAAFT_SURROGATES,
             "iaaft_iter": IAAFT_MAX_ITER, "iaaft_tol": IAAFT_CONVERGENCE_TOL,
             "micro_windows": MICROSTRUCTURE_WINDOWS}
    return _hash_dict(base4)

def _infer_scope(name):
    if name.startswith("stage1"): return "stage1"
    if name.startswith("stage2"): return "stage2"
    if name.startswith("stage3"): return "stage3"
    return "stage4"

def rolling_fingerprint():
    return config_fingerprint("stage3")[:12]

def surrogate_fingerprint():
    d={"stage3":config_fingerprint("stage3"),"mf_q":MF_Q_VALUES,"mf_floor":MF_VARIANCE_FLOOR,
       "iaaft_iter":IAAFT_MAX_ITER,"iaaft_tol":IAAFT_CONVERGENCE_TOL}
    return _hash_dict(d)[:12]

def save_checkpoint(name: str, payload: dict):
    path = CHECKPOINT_DIR / f"{name}.pkl"
    scope = _infer_scope(name)
    wrapper = {
        "pipeline_version": PIPELINE_VERSION,
        "config_fingerprint": config_fingerprint(scope),
        "saved_at": _dt.datetime.now().isoformat(timespec="seconds"),
        "payload": payload,
    }
    pd.to_pickle(wrapper, path)
    return path


def load_checkpoint(name: str, strict: bool = True) -> dict:
    path = CHECKPOINT_DIR / f"{name}.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Required checkpoint does not exist: {path}")
    wrapper = pd.read_pickle(path)
    if strict:
        if wrapper.get("pipeline_version") != PIPELINE_VERSION:
            raise RuntimeError(f"Checkpoint {name} belongs to a different pipeline version.")
        if wrapper.get("config_fingerprint") != config_fingerprint(_infer_scope(name)):
            raise RuntimeError(
                f"Checkpoint {name} no longer matches current data/configuration. "
                "Rerun from the corresponding stage or use --force in the launcher."
            )
    return wrapper["payload"]


def checkpoint_exists(name: str) -> bool:
    return (CHECKPOINT_DIR / f"{name}.pkl").exists()


def save_stage_tables(stage_name: str, tables: dict):
    return save_checkpoint(f"{stage_name}_tables", {"tables": tables})


def load_stage_tables(stage_name: str) -> dict:
    return load_checkpoint(f"{stage_name}_tables")["tables"]


def mark_stage(stage_num: int, status: str, detail: str = ""):
    data = {}
    if STATUS_JSON.exists():
        try:
            data = json.loads(STATUS_JSON.read_text(encoding="utf-8"))
        except Exception:
            data = {}
    data[str(stage_num)] = {
        "status": status,
        "detail": detail,
        "timestamp": _dt.datetime.now().isoformat(timespec="seconds"),
        "version": PIPELINE_VERSION,
    }
    STATUS_JSON.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def newey_west_corr_pvalue(x, y, maxlags=120):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(maxlags)})
    r = np.corrcoef(x, y)[0, 1]
    return r, model.pvalues[1], model


def fdr_table_from_pairs(pairs, family_name, alpha=0.05):
    if not pairs:
        return pd.DataFrame(columns=["Family", "Test", "p_HAC", "p_adjusted_FDR", "Significant_after_FDR"])
    labels = [lbl for lbl, _ in pairs]
    raw_p = np.asarray([p for _, p in pairs], dtype=float)
    valid = np.isfinite(raw_p)
    out = pd.DataFrame({"Family": family_name, "Test": labels, "p_HAC": raw_p})
    out["p_adjusted_FDR"] = np.nan; out["Significant_after_FDR"] = False
    if valid.any():
        reject, p_adj, _, _ = multipletests(raw_p[valid], alpha=alpha, method="fdr_bh")
        out.loc[valid, "p_adjusted_FDR"] = p_adj
        out.loc[valid, "Significant_after_FDR"] = reject
    return out


def stationarity_report(series, name, regression="c"):
    s = pd.Series(series).dropna().astype(float)
    if len(s) < 30 or s.nunique() < 3:
        return {"Series": name, "N": len(s), "ADF_stat": np.nan, "ADF_p": np.nan,
                "KPSS_stat": np.nan, "KPSS_p": np.nan,
                "ADF_stationary(p<0.05)": False, "KPSS_stationary(p>0.05)": False}
    if len(s) > 50_000:
        adf_stat, adf_p, *_ = adfuller(s, regression=regression, autolag=None, maxlag=20)
    else:
        adf_stat, adf_p, *_ = adfuller(s, regression=regression, autolag="AIC")
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_stat, kpss_p, *_ = kpss(s, regression=regression, nlags="auto")
    return {"Series": name, "N": len(s), "ADF_stat": adf_stat, "ADF_p": adf_p,
            "KPSS_stat": kpss_stat, "KPSS_p": kpss_p,
            "ADF_stationary(p<0.05)": bool(adf_p < 0.05),
            "KPSS_stationary(p>0.05)": bool(kpss_p > 0.05)}


def robust_discrepancy_check(x, y, name_x, name_y, diff_threshold=0.10, winsor_limits=(0.05,0.05)):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y); x, y = x[mask], y[mask]
    r_p, _ = pearsonr(x, y); r_s, _ = spearmanr(x, y)
    if abs(r_p-r_s) < diff_threshold:
        return None
    xw = np.asarray(winsorize(x, limits=winsor_limits)); yw = np.asarray(winsorize(y, limits=winsor_limits))
    r_pw, p_pw = pearsonr(xw, yw)
    X = sm.add_constant(x)
    huber = sm.RLM(y, X, M=sm.robust.norms.HuberT()).fit()
    print(f"  [Robustness] {name_x} vs {name_y}: Pearson={r_p:+.3f}, Spearman={r_s:+.3f}, "
          f"winsor={r_pw:+.3f} (p={p_pw:.4g}), beta_Huber={huber.params[1]:+.4g}", flush=True)
    return {"r_pearson": r_p, "r_spearman": r_s, "r_pearson_winsor": r_pw, "beta_huber": huber.params[1]}


def by_day(series):
    return series.groupby(series.index.normalize())


def yang_zhang(group):
    n = len(group)
    if n < 3: return np.nan
    k = 0.34/(1.34+(n+1)/(n-1))
    sigma_oc = group["LogOC"].var(); sigma_co = group["LogCO"].var(); sigma_rs = group["RS_term"].mean()
    yz_bar = np.sqrt(np.maximum(sigma_oc+k*sigma_co+(1-k)*sigma_rs, 0))
    return yz_bar*np.sqrt(n)


def yang_zhang_multiday(df_ohlc_daily, window=22):
    o,h,l,c = df_ohlc_daily["Open"],df_ohlc_daily["High"],df_ohlc_daily["Low"],df_ohlc_daily["Close"]
    log_oc = np.log(o/c.shift(1)); log_co = np.log(c/o)
    rs = np.log(h/c)*np.log(h/o)+np.log(l/c)*np.log(l/o)
    n=window; k=0.34/(1.34+(n+1)/(n-1))
    return np.sqrt(np.maximum(log_oc.rolling(n).var()+k*log_co.rolling(n).var()+(1-k)*rs.rolling(n).mean(),0))


def bipower_variation(x):
    arr=np.asarray(x,dtype=float); arr=arr[np.isfinite(arr)]; n=len(arr)
    if n<2: return np.nan
    return (np.pi/2)*(n/(n-1))*np.sum(np.abs(arr[1:])*np.abs(arr[:-1]))


def annualize_rms(series, annualization_days):
    s=pd.Series(series).dropna().astype(float)
    return np.sqrt(np.mean(np.square(s))*annualization_days)*100 if len(s) else np.nan

def _pelt_one_base(series_name, s, min_size, multipliers):
    s=pd.Series(s).dropna().astype(float)
    if len(s)<2*min_size: return [], []
    y=s.values; base=np.log(len(y))*np.var(y)
    algo=rpt.Pelt(model="l2", min_size=int(min_size), jump=1).fit(y)
    summary=[]; records=[]
    for mult in multipliers:
        bkps=algo.predict(pen=base*mult)[:-1]
        dates=[s.index[i] for i in bkps if i<len(s)]
        spec=f"{series_name}|min{min_size}|pen{mult:g}"
        summary.append({"Series":series_name,"Min_size_days":min_size,"Multiplicador_penalizacion":mult,
                        "N_changepoints":len(dates),"Dates":", ".join(str(pd.Timestamp(d).date()) for d in dates) if dates else "(none)"})
        records.extend({"Date":pd.Timestamp(d),"Series":series_name,"Min_size_days":min_size,
                        "Multiplicador_penalizacion":mult,"Spec_ID":spec} for d in dates)
    return summary, records


def pelt_multispec_grid_parallel(series_dict, min_sizes=PELT_MIN_SIZES, multipliers=PELT_PENALTY_MULTIPLIERS, n_jobs=N_JOBS):
    tasks=[(name,s,m) for name,s in series_dict.items() for m in min_sizes]
    res=Parallel(n_jobs=n_jobs, backend="loky", verbose=JOBLIB_VERBOSE)(
        delayed(_pelt_one_base)(name,s,m,multipliers) for name,s,m in tasks)
    summary=[]; records=[]
    for a,b in res: summary.extend(a); records.extend(b)
    return pd.DataFrame(summary), pd.DataFrame(records)


def cluster_changepoint_records(records_df, total_specs, series_names, tolerance_days=15, support_threshold=0.50):
    if records_df is None or len(records_df)==0:
        return pd.DataFrame(columns=["Date_consensus","N_detections","N_specs_support","Support_pct","N_series_support","Robust"])
    recs=records_df.sort_values("Date").to_dict("records"); clusters=[]
    for rec in recs:
        d=pd.Timestamp(rec["Date"]); best_idx=None; best_dist=None
        for j,cl in enumerate(clusters):
            center=sorted(cl["dates"])[len(cl["dates"])//2]; dist=abs((d-center).days)
            if dist<=tolerance_days and (best_dist is None or dist<best_dist): best_idx,best_dist=j,dist
        if best_idx is None: clusters.append({"dates":[d],"records":[rec]})
        else: clusters[best_idx]["dates"].append(d); clusters[best_idx]["records"].append(rec)
    rows=[]; n_series_total=len(series_names)
    for cl in clusters:
        ds=sorted(cl["dates"]); center=ds[len(ds)//2]; specs={r["Spec_ID"] for r in cl["records"]}; ss={r["Series"] for r in cl["records"]}
        support=len(specs)/total_specs if total_specs else np.nan
        row={"Date_consensus":center,"Date_min":ds[0],"Date_max":ds[-1],"N_detections":len(cl["records"]),
             "N_specs_support":len(specs),"Support_pct":support,"N_series_support":len(ss),
             "Robust":bool(len(ss)==n_series_total and support>=support_threshold)}
        for sname in series_names:
            row[f"Specs_{sname}"]=len({r["Spec_ID"] for r in cl["records"] if r["Series"]==sname})
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["Robust","Support_pct","Date_consensus"],ascending=[False,False,True]).reset_index(drop=True)


def markov_switching_regime(vol_series, ar_order=1):
    s=pd.Series(vol_series).dropna().astype(float)
    np.random.seed(RANDOM_SEED)
    mod=MarkovAutoregression(s,k_regimes=2,order=ar_order,trend="c",switching_variance=True)
    res=mod.fit(disp=False,search_reps=MS_SEARCH_REPS)
    probs=np.asarray(res.smoothed_marginal_probabilities,dtype=float); y=s.iloc[ar_order:].values
    weighted=[]
    for i in range(probs.shape[1]):
        w=probs[:,i]; weighted.append(float(np.sum(w*y)/np.sum(w)))
    high=int(np.argmax(weighted)); p=pd.Series(probs[:,high],index=s.index[ar_order:],name="Prob_Regime_High")
    return res,p,high,weighted


def find_high_regime_episodes(p_series,threshold=0.5,min_duration=10):
    above=(p_series>threshold).values; episodes=[]; i=0; n=len(above)
    while i<n:
        if above[i]:
            j=i
            while j<n and above[j]: j+=1
            if j-i>=min_duration: episodes.append((p_series.index[i],p_series.index[j-1],j-i))
            i=j
        else: i+=1
    return episodes

def _linear_detrend_variances(segs):
    segs=np.asarray(segs,dtype=float); s=segs.shape[1]
    t=np.arange(s,dtype=float); tc=t-t.mean(); denom=np.dot(tc,tc)
    means=segs.mean(axis=1)
    slopes=((segs-means[:,None])@tc)/denom
    fitted=means[:,None]+slopes[:,None]*tc[None,:]
    return np.mean((segs-fitted)**2,axis=1)


def _segment_variances(profile,s,order=1):
    n_seg=len(profile)//s
    if n_seg<2: return None
    f=profile[:n_seg*s].reshape(n_seg,s); b=profile[len(profile)-n_seg*s:][::-1].reshape(n_seg,s)
    if order==1:
        return np.concatenate([_linear_detrend_variances(f),_linear_detrend_variances(b)])
    t=np.arange(s); A=np.vander(t,order+1)
    cf,*_=np.linalg.lstsq(A,f.T,rcond=None); cb,*_=np.linalg.lstsq(A,b.T,rcond=None)
    rf=f-(A@cf).T; rb=b-(A@cb).T
    return np.concatenate([np.mean(rf**2,axis=1),np.mean(rb**2,axis=1)])


def dfa(x,min_window=10,max_window=None,n_scales=20,order=1):
    x=np.asarray(x,dtype=float); x=x[np.isfinite(x)]; n=len(x)
    if n<100: return np.nan,np.nan,None,None
    if max_window is None: max_window=n//4
    scales=np.unique(np.logspace(np.log10(min_window),np.log10(max_window),n_scales).astype(int))
    scales=scales[scales>=max(order+2,min_window)]
    if len(scales)<4: return np.nan,np.nan,None,None
    profile=np.cumsum(x-x.mean()); F=np.full(len(scales),np.nan)
    for i,s in enumerate(scales):
        vars_=_segment_variances(profile,int(s),order)
        if vars_ is not None: F[i]=np.sqrt(np.mean(vars_))
    valid=(F>0)&np.isfinite(F)
    if valid.sum()<4: return np.nan,np.nan,scales,F
    lx=np.log(scales[valid]); ly=np.log(F[valid]); slope,intercept=np.polyfit(lx,ly,1); pred=slope*lx+intercept
    den=np.sum((ly-ly.mean())**2); r2=1-np.sum((ly-pred)**2)/den if den>0 else np.nan
    return slope,r2,scales,F


def dfa_biscale(x,min_window=10,max_window=None,n_scales=32,order=1,min_points_per_segment=6,min_segments_per_scale=4):
    x=np.asarray(x,dtype=float); x=x[np.isfinite(x)]; n=len(x)
    if n<200: return None
    if max_window is None: max_window=max(min_window+1,n//min_segments_per_scale)
    scales=np.unique(np.logspace(np.log10(min_window),np.log10(max_window),n_scales).astype(int))
    scales=scales[scales>=max(order+2,min_window)]
    if len(scales)<2*min_points_per_segment: return None
    profile=np.cumsum(x-x.mean()); F=np.full(len(scales),np.nan)
    for i,s in enumerate(scales):
        vars_=_segment_variances(profile,int(s),order)
        if vars_ is not None and (n//s)>=min_segments_per_scale: F[i]=np.sqrt(np.mean(vars_))
    valid=(F>0)&np.isfinite(F); sv,Fv=scales[valid],F[valid]
    if len(sv)<2*min_points_per_segment: return None
    lx,ly=np.log(sv),np.log(Fv); npts=len(lx); m0,b0=np.polyfit(lx,ly,1); p0=m0*lx+b0
    ssr0=max(float(np.sum((ly-p0)**2)),np.finfo(float).tiny); bic0=npts*np.log(ssr0/npts)+2*np.log(npts)
    best=None
    for split in range(min_points_per_segment,npts-min_points_per_segment+1):
        x1,y1=lx[:split],ly[:split]; x2,y2=lx[split:],ly[split:]
        m1,b1=np.polyfit(x1,y1,1); m2,b2=np.polyfit(x2,y2,1); p1=m1*x1+b1; p2=m2*x2+b2
        ssr=max(float(np.sum((y1-p1)**2)+np.sum((y2-p2)**2)),np.finfo(float).tiny); bic2=npts*np.log(ssr/npts)+5*np.log(npts)
        if best is None or bic2<best["bic_two_lines"]:
            def r2(obs,pred):
                den=np.sum((obs-obs.mean())**2); return 1-np.sum((obs-pred)**2)/den if den>0 else np.nan
            best={"alpha_short":m1,"alpha_long":m2,"r2_short":r2(y1,p1),"r2_long":r2(y2,p2),
                  "break_scale_bars":float(np.sqrt(sv[split-1]*sv[split])),"n_short":len(x1),"n_long":len(x2),
                  "ssr_one_line":ssr0,"ssr_two_lines":ssr,"bic_one_line":bic0,"bic_two_lines":bic2,
                  "delta_bic_favor_two_lines":bic0-bic2,"alpha_single":m0,"scales":sv,"F":Fv,"split":split}
    return best

def _rolling_point(i,w,dates,idx_values,logret,yz,rv,bv,jump,min_window_points):
    end_date=dates[i]; start_date=dates[i-w+1]; next_day=end_date+np.timedelta64(1,"D")
    sl=slice(i-w+1,i+1)
    yz_mean=float(np.nanmean(yz[sl])); rv_mean=float(np.nanmean(rv[sl])); bv_mean=float(np.nanmean(bv[sl])); jump_sum=float(np.nansum(jump[sl]))
    a=np.searchsorted(idx_values,start_date,side="left"); b=np.searchsorted(idx_values,next_day,side="left")
    rets=np.asarray(logret[a:b],dtype=float); rets=rets[np.isfinite(rets)]
    if len(rets)>=min_window_points: H,r2,_,_=dfa(rets)
    else: H,r2=np.nan,np.nan
    return end_date,yz_mean,rv_mean,bv_mean,jump_sum,H,r2,len(rets)


def rolling_window_parallel(df,daily,w,min_window_points=500,n_jobs=N_JOBS):
    dates=daily.index.to_numpy(); idx=df.index.values; logret=df["LogRet"].to_numpy(dtype=float)
    yz=daily["Yang_Zhang"].to_numpy(dtype=float); rv=daily["RV"].to_numpy(dtype=float)
    bv=daily["Bipower_Var"].to_numpy(dtype=float); jump=daily["Jump_Component"].to_numpy(dtype=float)
    indices=range(w-1,len(dates))
    rows=Parallel(n_jobs=n_jobs,backend="loky",batch_size="auto",max_nbytes="10M",mmap_mode="r",verbose=JOBLIB_VERBOSE)(
        delayed(_rolling_point)(i,w,dates,idx,logret,yz,rv,bv,jump,min_window_points) for i in indices)
    cols=["Date",f"YZ_{w}d",f"RV_{w}d",f"BV_{w}d",f"Jump_{w}d",f"H_{w}d",f"R2_{w}d",f"Nret_{w}d"]
    return pd.DataFrame(rows,columns=cols).set_index("Date")


def rolling_vol_and_dfa_parallel_resume(df,daily,windows=ROLLING_WINDOWS,n_jobs=N_JOBS):
    pieces=[]
    for w in windows:
        cp=CHECKPOINT_DIR/f"rolling_{w}d_{rolling_fingerprint()}.pkl"
        if cp.exists():
            part=pd.read_pickle(cp)
            print(f"  [resume] rolling {w}d loaded: {len(part):,} rows",flush=True)
        else:
            print(f"  [rolling] computing {w}d with {n_jobs} processes...",flush=True)
            part=rolling_window_parallel(df,daily,w,n_jobs=n_jobs)
            pd.to_pickle(part,cp)
            print(f"  [checkpoint] {cp.name} saved",flush=True)
        pieces.append(part)
    out=pd.concat(pieces,axis=1)
    for w in windows: out[f"D_H_{w}d"]=(out[f"H_{w}d"]-0.5).abs()
    return out

def load_fred_series(path,rename_to,end_date=None):
    raw=pd.read_csv(path,na_values=["."])
    if "observation_date" not in raw.columns: raise ValueError(f"FRED CSV missing observation_date: {path}")
    raw["observation_date"]=pd.to_datetime(raw["observation_date"]); raw=raw.set_index("observation_date")
    cols=list(raw.columns)
    if not cols: raise ValueError(f"FRED CSV contains no values: {path}")
    s=pd.to_numeric(raw[cols[0]],errors="coerce").rename(rename_to).dropna().sort_index()
    if end_date is not None: s=s[s.index<=end_date]
    return s


def build_har_features(rv_series):
    rv=rv_series.copy(); feat=pd.DataFrame(index=rv.index)
    feat["RV_d"]=rv.shift(1); feat["RV_w"]=rv.shift(1).rolling(5).mean(); feat["RV_m"]=rv.shift(1).rolling(22).mean(); feat["RV_target"]=rv
    return feat.dropna()


def purged_walk_forward_corr(data,x_col,y_col,window_days,n_folds=OOS_N_FOLDS,first_test_fraction=OOS_FIRST_TEST_FRACTION):
    d=data[[x_col,y_col]].dropna().copy(); n=len(d); rows=[]
    if n<200: return pd.DataFrame(rows)
    edges=np.linspace(first_test_fraction,1.0,n_folds+1)
    for fold in range(n_folds):
        ts=int(np.floor(edges[fold]*n)); te=n if fold==n_folds-1 else int(np.floor(edges[fold+1]*n)); tr_end=max(0,ts-int(window_days))
        train=d.iloc[:tr_end]; test=d.iloc[ts:te]
        if len(train)<60 or len(test)<30: continue
        rt,pt,_=newey_west_corr_pvalue(train[x_col],train[y_col],maxlags=window_days); rr,pr,_=newey_west_corr_pvalue(test[x_col],test[y_col],maxlags=window_days)
        rows.append({"Fold":fold+1,"Window_days":window_days,"N_train":len(train),"N_purge":ts-tr_end,"N_test":len(test),
                     "Train_end":train.index.max(),"Test_start":test.index.min(),"Test_end":test.index.max(),
                     "r_train":rt,"p_train_HAC":pt,"r_test":rr,"p_test_HAC":pr,"Sign_equal":bool(np.sign(rt)==np.sign(rr))})
    return pd.DataFrame(rows)

def mf_dfa(x,q_values=MF_Q_VALUES,min_window=10,max_window=None,n_scales=20,order=1,variance_floor=MF_VARIANCE_FLOOR,min_segments_per_scale=4):
    x=np.asarray(x,dtype=float); x=x[np.isfinite(x)]; n=len(x)
    if n<200: return None
    if max_window is None: max_window=max(min_window+1,n//min_segments_per_scale)
    scales=np.unique(np.logspace(np.log10(min_window),np.log10(max_window),n_scales).astype(int)); scales=scales[scales>=max(order+2,min_window)]
    if len(scales)<6: return None
    profile=np.cumsum(x-x.mean()); rows=[]
    for q in q_values:
        Fq=np.full(len(scales),np.nan)
        for i,s in enumerate(scales):
            if n//s<min_segments_per_scale: continue
            vars_=_segment_variances(profile,int(s),order)
            if vars_ is None: continue
            va=np.maximum(vars_,variance_floor)
            Fq[i]=np.exp(0.5*np.mean(np.log(va))) if q==0 else np.mean(va**(q/2.0))**(1.0/q)
        valid=(Fq>0)&np.isfinite(Fq)
        if valid.sum()<5: rows.append({"q":q,"h(q)":np.nan,"R2_hq":np.nan,"N_escalas":int(valid.sum())}); continue
        lx=np.log(scales[valid]); ly=np.log(Fq[valid]); m,b=np.polyfit(lx,ly,1); pred=m*lx+b; den=np.sum((ly-ly.mean())**2)
        rows.append({"q":q,"h(q)":m,"R2_hq":1-np.sum((ly-pred)**2)/den if den>0 else np.nan,"N_escalas":int(valid.sum())})
    out=pd.DataFrame(rows).sort_values("q").reset_index(drop=True); mask=out["h(q)"].notna()
    if mask.sum()>=3:
        qv=out.loc[mask,"q"].to_numpy(float); hv=out.loc[mask,"h(q)"].to_numpy(float); tau=qv*hv-1.0; alpha=np.gradient(tau,qv); f=qv*alpha-tau
        out.loc[mask,"tau(q)"]=tau; out.loc[mask,"alpha"]=alpha; out.loc[mask,"f(alpha)"]=f
    else: out["tau(q)"]=np.nan; out["alpha"]=np.nan; out["f(alpha)"]=np.nan
    return out


def mfdfa_range(result_df):
    vals=result_df["h(q)"].dropna().to_numpy(float)
    return float(vals.max()-vals.min()) if len(vals)>=2 else np.nan


def iaaft_surrogate(x,rng,max_iter=IAAFT_MAX_ITER,tol=IAAFT_CONVERGENCE_TOL):
    x=np.asarray(x,dtype=float); x=x[np.isfinite(x)]
    if len(x)<8: return x.copy(),np.nan
    sx=np.sort(x); target=np.abs(np.fft.rfft(x)); denom=np.linalg.norm(target); y=rng.permutation(x); prev=np.inf; err=np.nan
    for _ in range(max_iter):
        Y=np.fft.rfft(y); phases=np.exp(1j*np.angle(Y)); ys=np.fft.irfft(target*phases,n=len(x)); order=np.argsort(ys,kind="mergesort"); ynew=np.empty_like(ys); ynew[order]=sx; y=ynew
        cur=np.abs(np.fft.rfft(y)); err=np.linalg.norm(cur-target)/denom if denom>0 else np.nan
        if np.isfinite(err) and np.isfinite(prev) and abs(prev-err)<=tol*max(prev,1e-15): break
        prev=err
    return y,err


def _one_shuffle(i,x,mf_kwargs):
    rng=np.random.default_rng(RANDOM_SEED+10_000+i); res=mf_dfa(rng.permutation(x),**mf_kwargs); hr=mfdfa_range(res) if res is not None else np.nan
    return {"Surrogate_ID":i+1,"h_range_shuffled":hr}


def _one_iaaft(i,x,mf_kwargs):
    rng=np.random.default_rng(RANDOM_SEED+100_000+i); xs,err=iaaft_surrogate(x,rng); res=mf_dfa(xs,**mf_kwargs); hr=mfdfa_range(res) if res is not None else np.nan
    av=res["alpha"].dropna().to_numpy(float) if res is not None and "alpha" in res else np.array([]); da=float(av.max()-av.min()) if len(av)>=2 else np.nan
    return {"Surrogate_ID":i+1,"h_range_IAAFT":hr,"Delta_alpha_IAAFT":da,"Error_spectral_rel_L2":err}


def _resume_surrogates(kind,x,n_total,batch_size=SURROGATE_BATCH_SIZE,n_jobs=N_JOBS,**mf_kwargs):
    cp=CHECKPOINT_DIR/f"mfdfa_{kind}_progress_{surrogate_fingerprint()}.pkl"; df_done=pd.read_pickle(cp) if cp.exists() else pd.DataFrame(); done=set(df_done.get("Surrogate_ID",pd.Series(dtype=int)).astype(int).tolist())
    missing=[i for i in range(n_total) if (i+1) not in done]
    print(f"  [{kind}] completed={len(done)}/{n_total}; missing={len(missing)}",flush=True)
    fn=_one_shuffle if kind=="shuffle" else _one_iaaft
    for k in range(0,len(missing),batch_size):
        ids=missing[k:k+batch_size]
        rows=Parallel(n_jobs=min(n_jobs,len(ids)),backend="loky",verbose=JOBLIB_VERBOSE)(delayed(fn)(i,x,mf_kwargs) for i in ids)
        df_done=pd.concat([df_done,pd.DataFrame(rows)],ignore_index=True).drop_duplicates("Surrogate_ID",keep="last").sort_values("Surrogate_ID")
        pd.to_pickle(df_done,cp)
        print(f"  [{kind}] checkpoint {len(df_done)}/{n_total}",flush=True)
    return df_done.reset_index(drop=True)


def mfdfa_shuffle_resume(x,n_surrogates=N_SURROGATES,**mf_kwargs):
    return _resume_surrogates("shuffle",np.asarray(x,float),n_surrogates,**mf_kwargs)


def mfdfa_iaaft_resume(x,n_surrogates=N_IAAFT_SURROGATES,**mf_kwargs):
    return _resume_surrogates("iaaft",np.asarray(x,float),n_surrogates,**mf_kwargs)

def resample_ohlc(df_source,rule):
    agg={"Open":"first","High":"max","Low":"min","Close":"last"}
    return df_source[["Open","High","Low","Close"]].resample(rule).agg(agg).dropna()


def realized_daily_from_frequency(df_freq,expected_minutes=30):
    d=df_freq.copy(); delta=d.index.to_series().diff().dt.total_seconds().div(60.0); good=np.isclose(delta.values,expected_minutes,rtol=0,atol=1e-9)
    d["LogRet_Global"]=np.log(d["Close"]/d["Close"].shift(1)); d["LogRet"]=d["LogRet_Global"].where(good)
    dr=d.dropna(subset=["LogRet"]).copy(); rv=by_day(dr["LogRet"]).apply(lambda x:np.sqrt(np.sum(np.asarray(x,float)**2))); bv=by_day(dr["LogRet"]).apply(bipower_variation)
    daily=pd.DataFrame({"RV":rv,"Bipower_Var":bv}); daily["Jump_Component"]=np.maximum(daily["RV"]**2-daily["Bipower_Var"],0); daily["BV_to_RV2_Ratio_Pct"]=np.where(daily["RV"]**2>0,daily["Bipower_Var"]/daily["RV"]**2*100,np.nan)
    return d,daily


def _rolling_freq_point(i,w,dates,idx,lr,j,min_window_points):
    end=dates[i]; start=dates[i-w+1]; nxt=end+np.timedelta64(1,"D")
    js=float(np.nansum(j[i-w+1:i+1])); a=np.searchsorted(idx,start,"left"); b=np.searchsorted(idx,nxt,"left")
    r=lr[a:b]; r=r[np.isfinite(r)]
    H,r2,_,_=dfa(r) if len(r)>=min_window_points else (np.nan,np.nan,None,None)
    return end,js,H,r2,len(r)

def rolling_jump_h_frequency(df_freq,daily_freq,windows=MICROSTRUCTURE_WINDOWS,min_window_points=250,n_jobs=N_JOBS):
    dates=daily_freq.index.to_numpy(); idx=df_freq.index.values; lr=df_freq["LogRet"].to_numpy(float); j=daily_freq["Jump_Component"].to_numpy(float)
    outs=[]
    for w in windows:
        rows=Parallel(n_jobs=n_jobs,backend="loky",batch_size="auto",max_nbytes="10M",mmap_mode="r",verbose=JOBLIB_VERBOSE)(
            delayed(_rolling_freq_point)(i,w,dates,idx,lr,j,min_window_points) for i in range(w-1,len(dates)))
        outs.append(pd.DataFrame(rows,columns=["Date",f"Jump30_{w}d",f"H30_{w}d",f"R2_30_{w}d",f"Nret30_{w}d"]).set_index("Date"))
    return pd.concat(outs,axis=1)

def safe_sheet_name(name,used):
    clean=name
    for ch in [":","\\","/","?","*","[","]"]: clean=clean.replace(ch,"_")
    clean=clean[:31]; base=clean; i=1
    while clean in used:
        suffix=f"_{i}"; clean=base[:31-len(suffix)]+suffix; i+=1
    used.add(clean); return clean


def export_all_tables(tables,path=FINAL_EXCEL):
    used=set()
    with pd.ExcelWriter(path,engine="openpyxl") as writer:
        for name,df in tables.items():
            sheet=safe_sheet_name(name,used)
            if df is None or len(df)==0: df=pd.DataFrame({"Note":["No data / block not executed"]})
            df.to_excel(writer,sheet_name=sheet,index=False)
    return path


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage1_data.py
# -*- coding: utf-8 -*-
"""Stage 1: data loading, QA and daily estimators."""
from config import *
from tabulate import tabulate

STAGE = "stage1_data"


def main(force=False):
    if checkpoint_exists(STAGE) and not force:
        print(f"[{STAGE}] Checkpoint already exists. Use --force to recompute.")
        return

    with stage_logging(STAGE):
        mark_stage(1, "running")
        np.random.seed(RANDOM_SEED)
        tables = {}

        if not OHLCV_PATH.exists():
            raise FileNotFoundError(f"OHLCV file not found: {OHLCV_PATH}")
        print(f"Loading dataset: {OHLCV_PATH}", flush=True)
        required = ["Date", "Time", "Open", "High", "Low", "Close"]
        df = pd.read_csv(OHLCV_PATH)
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        for c in ["Open", "High", "Low", "Close"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        n_raw = len(df)
        df["Date"] = df["Date"].astype(str); df["Time"] = df["Time"].astype(str)
        df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%Y%m%d %H:%M:%S", errors="coerce")
        n_bad_dt = int(df["Datetime"].isna().sum())
        n_bad_num = int(df[["Open", "High", "Low", "Close"]].isna().any(axis=1).sum())
        if n_bad_dt or n_bad_num:
            raise ValueError(f"Critical QA: invalid timestamps={n_bad_dt}, nonnumeric/NaN OHLC={n_bad_num}")
        df = df.set_index("Datetime").sort_index()

        n_dup = int(df.index.duplicated(keep=False).sum())
        nonpositive = (df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1)
        invalid = ((df["High"] < df[["Open", "Close", "Low"]].max(axis=1)) |
                   (df["Low"] > df[["Open", "Close", "High"]].min(axis=1)) |
                   (df["High"] < df["Low"]))
        if n_dup:
            raise ValueError(f"Critical QA: {n_dup} rows with duplicate timestamps")
        if nonpositive.any():
            raise ValueError(f"Critical QA: {int(nonpositive.sum())} rows with nonpositive prices (<= 0)")
        if invalid.any():
            raise ValueError(f"Critical QA: {int(invalid.sum())} rows violate OHLC geometry")

        n_before = len(df)
        df = df[df.index <= STUDY_END_DATE].copy()
        n_dropped = n_before - len(df)
        gap_minutes = df.index.to_series().diff().dt.total_seconds().div(60.0)
        candles_per_day = df["Close"].groupby(df.index.normalize()).size()
        median_candles = float(candles_per_day.median())
        low_days = candles_per_day[candles_per_day < 0.75 * median_candles]

        qa = pd.DataFrame([
            {"Metric": "Raw rows read", "Value": n_raw},
            {"Metric": "Rows after STUDY_END_DATE", "Value": len(df)},
            {"Metric": "Duplicate timestamps", "Value": n_dup},
            {"Metric": "Invalid OHLC rows", "Value": int(invalid.sum())},
            {"Metric": "Non-positive prices", "Value": int(nonpositive.sum())},
            {"Metric": "Distinct days", "Value": int(candles_per_day.size)},
            {"Metric": "Median bars/day", "Value": median_candles},
            {"Metric": "Bars/day p05", "Value": float(candles_per_day.quantile(0.05))},
            {"Metric": "Bars/day p95", "Value": float(candles_per_day.quantile(0.95))},
            {"Metric": "Days below 75% of median", "Value": int(len(low_days))},
            {"Metric": "Gaps over 30 minutes", "Value": int((gap_minutes > 30).sum())},
            {"Metric": "Gaps over 6 hours", "Value": int((gap_minutes > 360).sum())},
            {"Metric": "Gaps over 24 hours", "Value": int((gap_minutes > 1440).sum())},
        ])
        tables["QA_Intraday_Data"] = qa
        tables["QA_Daily_Coverage"] = candles_per_day.rename("N_bars").to_frame().reset_index().rename(columns={"Datetime":"Date","index":"Date"})
        print(f"Range: {df.index.min()} -> {df.index.max()} | {len(df):,} bars | excluded by date={n_dropped}")
        if DATA_VENDOR == "PENDING_DOCUMENTATION" or SOURCE_TIMEZONE == "PENDING_DOCUMENTATION":
            print("[QA-METADATA] provider/timezone still pending documentation.")

        # Returns: only exactly equally spaced intervals.
        df["LogRet_Global"] = np.log(df["Close"] / df["Close"].shift(1))
        delta = df.index.to_series().diff().dt.total_seconds().div(60.0)
        good = np.isclose(delta.values, EXPECTED_BAR_MINUTES, rtol=0.0, atol=1e-9)
        df["LogRet"] = df["LogRet_Global"].where(good)
        n_nonstandard = int((delta.notna() & ~pd.Series(good, index=df.index)).sum())
        qa = pd.concat([qa, pd.DataFrame([{"Metric": f"Intervals != {EXPECTED_BAR_MINUTES}min excluded", "Value": n_nonstandard}])], ignore_index=True)
        tables["QA_Intraday_Data"] = qa
        print(f"Returns excluded because interval !={EXPECTED_BAR_MINUTES}m: {n_nonstandard:,}")

        df["LogHL"] = np.log(df["High"] / df["Low"])
        df["LogCO"] = np.log(df["Close"] / df["Open"])
        df["LogOC"] = np.log(df["Open"] / df["Close"].shift(1))
        df["LogHC"] = np.log(df["High"] / df["Close"])
        df["LogHO"] = np.log(df["High"] / df["Open"])
        df["LogLC"] = np.log(df["Low"] / df["Close"])
        df["LogLO"] = np.log(df["Low"] / df["Open"])
        df["GK_term"] = 0.5*df["LogHL"]**2 - (2*np.log(2)-1)*df["LogCO"]**2
        df["RS_term"] = df["LogHC"]*df["LogHO"] + df["LogLC"]*df["LogLO"]
        df_ret = df.dropna(subset=["LogRet"]).copy()

        n_candles = by_day(df["Close"]).apply(len)
        n_returns = by_day(df_ret["LogRet"]).apply(len)
        N_CANDLES_MEDIAN = int(n_candles.median()); N_RETURNS_MEDIAN = int(n_returns.median())
        n_years = (df.index.max() - df.index.min()).days / 365.25
        ANNUALIZATION_DAYS = df.index.normalize().nunique() / n_years
        print(f"Median bars/day={N_CANDLES_MEDIAN}; valid returns/day={N_RETURNS_MEDIAN}; days/year={ANNUALIZATION_DAYS:.2f}")

        daily = pd.DataFrame()
        daily["RV"] = by_day(df_ret["LogRet"]).apply(lambda x: np.sqrt(np.sum(np.asarray(x,float)**2)))
        daily["Parkinson"] = by_day(df["LogHL"]).apply(lambda x: np.sqrt(np.maximum(np.sum(x**2)/(4*np.log(2)),0)))
        daily["Garman_Klass"] = by_day(df["GK_term"]).apply(lambda x: np.sqrt(np.maximum(np.sum(x),0)))
        daily["Rogers_Satchell"] = by_day(df["RS_term"]).apply(lambda x: np.sqrt(np.maximum(np.sum(x),0)))
        daily["Yang_Zhang"] = df.groupby(df.index.normalize()).apply(yang_zhang)
        daily_ohlc = df.groupby(df.index.normalize()).agg(Open=("Open","first"),High=("High","max"),Low=("Low","min"),Close=("Close","last"))
        daily[f"Yang_Zhang_MultiDia_{YZ_MULTIDAY_WINDOW}d"] = yang_zhang_multiday(daily_ohlc, YZ_MULTIDAY_WINDOW)
        yz_comp = daily[["Yang_Zhang", f"Yang_Zhang_MultiDia_{YZ_MULTIDAY_WINDOW}d"]].dropna()
        if len(yz_comp) >= 30:
            r,_ = pearsonr(yz_comp.iloc[:,0], yz_comp.iloc[:,1])
            print(f"Intraday YZ vs conventional {YZ_MULTIDAY_WINDOW}d: r={r:+.3f} n={len(yz_comp)}")
            tables["YZ_Intraday_Multiday_Robust"] = yz_comp.reset_index().rename(columns={"Datetime":"Date","index":"Date"})

        daily["Bipower_Var"] = by_day(df_ret["LogRet"]).apply(bipower_variation)
        daily["Jump_Component"] = np.maximum(daily["RV"]**2 - daily["Bipower_Var"], 0)
        daily["Jump_Variation_Proxy"] = daily["Jump_Component"]
        daily["BV_to_RV2_Ratio_Pct"] = np.where(daily["RV"]**2>0, daily["Bipower_Var"]/daily["RV"]**2*100, np.nan)
        daily["Diffusion_Pct"] = daily["BV_to_RV2_Ratio_Pct"]
        daily["Kurtosis"] = by_day(df_ret["LogRet"]).apply(lambda x: x.kurtosis())
        daily["Skewness"] = by_day(df_ret["LogRet"]).apply(lambda x: x.skew())
        daily["Extreme_Candles"] = by_day(df_ret["LogRet"]).apply(lambda x: np.sum(np.abs(x)>EXTREME_RETURN_THRESHOLD))
        daily.index = pd.to_datetime(daily.index); daily["Year"] = daily.index.year

        summary = daily.groupby("Year").agg(
            Days=("RV","count"),
            RV=("RV",lambda x: annualize_rms(x,ANNUALIZATION_DAYS)),
            Parkinson=("Parkinson",lambda x: annualize_rms(x,ANNUALIZATION_DAYS)),
            Garman_Klass=("Garman_Klass",lambda x: annualize_rms(x,ANNUALIZATION_DAYS)),
            Rogers_Satchell=("Rogers_Satchell",lambda x: annualize_rms(x,ANNUALIZATION_DAYS)),
            Yang_Zhang=("Yang_Zhang",lambda x: annualize_rms(x,ANNUALIZATION_DAYS)),
        ).reset_index()
        summary_extra = daily.groupby("Year").agg(
            Kurtosis_Mean=("Kurtosis","mean"), Skewness_Mean=("Skewness","mean"),
            Jumps_Extreme=("Extreme_Candles","sum"), BV_RV2_Ratio_Pct=("BV_to_RV2_Ratio_Pct","mean"),
            RV_Max_Daily=("RV",lambda x: x.max()*np.sqrt(ANNUALIZATION_DAYS)*100),
            RV_Median=("RV",lambda x: x.median()*np.sqrt(ANNUALIZATION_DAYS)*100),
        ).reset_index()
        tables["Annual_Calendar_Context"] = summary.merge(summary_extra,on="Year")
        print(tabulate(summary.tail(8),headers="keys",tablefmt="simple",showindex=False,floatfmt=".2f"))

        input_sha = file_sha256(OHLCV_PATH)
        payload = {
            "df": df,
            "daily": daily,
            "N_CANDLES_MEDIAN": N_CANDLES_MEDIAN,
            "N_RETURNS_MEDIAN": N_RETURNS_MEDIAN,
            "ANNUALIZATION_DAYS": ANNUALIZATION_DAYS,
            "input_sha": input_sha,
            "n_nonstandard_intervals": n_nonstandard,
        }
        save_checkpoint(STAGE, payload)
        save_stage_tables(STAGE, tables)
        daily.to_csv(DAILY_CSV)
        mark_stage(1, "complete", f"{len(df):,} bars; {len(daily):,} days")
        print(f"Stage 1 checkpoint saved. SHA256 OHLCV={input_sha[:16]}...")


if __name__ == "__main__":
    import argparse
    p=argparse.ArgumentParser(); p.add_argument("--force",action="store_true"); args=p.parse_args()
    main(force=args.force)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage2_regime.py
# -*- coding: utf-8 -*-
"""Stage 2: regime changes (PELT, MS-AR) and DFA by segment."""
from config import *
from tabulate import tabulate
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

STAGE="stage2_regime"


def _annual_dfa_one(year, values):
    H,r2,scales,F=dfa(values)
    return {"Year":year,"N_returns_intraday":int(np.isfinite(values).sum()),"H_DFA":H,"R2_scaling":r2}


def _segment_dfa_one(regime_id, days, vals):
    H,r2,_,_=dfa(vals); bs=dfa_biscale(vals,min_window=10)
    out={"Regime_ID":int(regime_id),"Days":int(days),"N_returns_intraday":int(np.isfinite(vals).sum()),"H_DFA_one_line":H,"R2_one_line":r2}
    if bs is not None:
        out.update({"alpha_short":bs["alpha_short"],"R2_short":bs["r2_short"],"alpha_long":bs["alpha_long"],"R2_long":bs["r2_long"],
                    "break_scale_bars":bs["break_scale_bars"],"Delta_BIC_favor_two_lines":bs["delta_bic_favor_two_lines"]})
    return out


def main(force=False):
    if checkpoint_exists(STAGE) and not force:
        print(f"[{STAGE}] Checkpoint already exists. Use --force to recompute."); return
    s1=load_checkpoint("stage1_data"); df=s1["df"]; daily=s1["daily"].copy(); ann=s1["ANNUALIZATION_DAYS"]; nret=s1["N_RETURNS_MEDIAN"]
    with stage_logging(STAGE):
        mark_stage(2,"running"); tables={}
        print("PELT: parallel grid of 2 series x 4 min_size values, reusing each fit for 6 penalties.")
        series={"log_RV":np.log(daily["RV"].replace(0,np.nan)).dropna(),"log_YZ_intraday":np.log(daily["Yang_Zhang"].replace(0,np.nan)).dropna()}
        grid,records=pelt_multispec_grid_parallel(series,n_jobs=N_JOBS)
        total=len(series)*len(PELT_MIN_SIZES)*len(PELT_PENALTY_MULTIPLIERS)
        consensus=cluster_changepoint_records(records,total,list(series.keys()),PELT_CLUSTER_TOL_DAYS,PELT_CONSENSUS_SUPPORT)
        robust=sorted(pd.to_datetime(consensus.loc[consensus["Robust"],"Date_consensus"].tolist())) if len(consensus) else []
        tables["PELT_Multispecification_Grid"]=grid; tables["PELT_All_Detections"]=records; tables["PELT_Consensus_Changepoints"]=consensus
        print(tabulate(grid[["Series","Min_size_days","Multiplicador_penalizacion","N_changepoints"]],headers="keys",tablefmt="simple",showindex=False))
        print(f"PELT robust consensus: {len(robust)} changepoints")

        ms_ok=True
        try:
            _,p_high,_,means=markov_switching_regime(series["log_RV"],1); episodes=find_high_regime_episodes(p_high,0.5,10)
        except Exception as exc:
            ms_ok=False; p_high=pd.Series(dtype=float); means=[np.nan,np.nan]; episodes=[]; print(f"[MS-AR] failure: {exc}")
        tables["MS_AR_High_Regime_Prob"]=pd.DataFrame({"Date":p_high.index,"Prob_Regime_High":p_high.values})
        tables["MS_AR_High_Vol_Episodes"]=pd.DataFrame([{"Start":a,"End":b,"Duration_obs":d} for a,b,d in episodes]) if episodes else pd.DataFrame(columns=["Start","End","Duration_obs"])
        if ms_ok: print(f"MS-AR weighted means: low={min(means):.6f}, high={max(means):.6f}; episodes={len(episodes)}")

        boundaries=[]
        for a,b,_ in episodes: boundaries.extend([(pd.Timestamp(a),"Start"),(pd.Timestamp(b),"End")])
        crows=[]; brows=[]
        if robust and boundaries:
            for d in robust:
                ds=[abs((pd.Timestamp(d)-bd).days) for bd,_ in boundaries]; j=int(np.argmin(ds)); bd,t=boundaries[j]
                crows.append({"Changepoint_PELT_robust":d,"Boundary_MSAR_more_nearest":bd,"Type_boundary_MSAR":t,"Separation_days":ds[j],"Matching_15d":ds[j]<=PELT_CLUSTER_TOL_DAYS})
            for bd,t in boundaries:
                ds=[abs((bd-pd.Timestamp(d)).days) for d in robust]; j=int(np.argmin(ds))
                brows.append({"Boundary_MSAR":bd,"Type_boundary_MSAR":t,"Changepoint_PELT_more_nearest":robust[j],"Separation_days":ds[j],"Covered_by_PELT_15d":ds[j]<=PELT_CLUSTER_TOL_DAYS})
        cdf=pd.DataFrame(crows); bdf=pd.DataFrame(brows); tables["PELT_MSAR_Agreement"]=cdf; tables["MSAR_Coverage_by_PELT"]=bdf
        if len(cdf): print(f"Agreement PELT->MSAR: {cdf['Matching_15d'].mean():.1%}; MSAR->PELT: {bdf['Covered_by_PELT_15d'].mean():.1%}")

        break_arr=np.asarray(robust,dtype="datetime64[ns]")
        daily["Regime_ID"]=np.searchsorted(break_arr,daily.index.values,side="right") if len(break_arr) else 0
        daily["Regime_ID"]=daily["Regime_ID"].astype("Int64")
        bounds=[]
        for rid,g in daily.groupby("Regime_ID"):
            bounds.append({"Regime_ID":int(rid),"Date_Start":g.index.min().date(),"Date_End":g.index.max().date(),"Days":len(g)})
        bounds_df=pd.DataFrame(bounds)
        reg=daily.groupby("Regime_ID").agg(
            Days=("RV","count"),RV=("RV",lambda x:annualize_rms(x,ann)),Parkinson=("Parkinson",lambda x:annualize_rms(x,ann)),
            Garman_Klass=("Garman_Klass",lambda x:annualize_rms(x,ann)),Rogers_Satchell=("Rogers_Satchell",lambda x:annualize_rms(x,ann)),Yang_Zhang=("Yang_Zhang",lambda x:annualize_rms(x,ann))).reset_index()
        tables["Volatility_Consensus_Segments"]=bounds_df.merge(reg,on=["Regime_ID","Days"])
        print(tabulate(bounds_df,headers="keys",tablefmt="simple",showindex=False))

        fig,ax=plt.subplots(figsize=(12,5)); ax.plot(daily.index,daily["RV"],lw=.8,label="Daily RV")
        for d in robust: ax.axvline(d,ls="--",lw=1,alpha=.8)
        ax.set_title("Daily RV and robust PELT consensus changepoints"); ax.legend(); fig.tight_layout(); fig.savefig(FIG_DIR/"changepoints_consensus_RV.png",dpi=150); plt.close(fig)

        lag1=float(df["LogRet"].autocorr(lag=1)); print(f"Lag-1 autocorr LogRet={lag1:+.4f}")

        # Parallel annual DFA.
        annual_tasks=[(int(y),g["LogRet"].values) for y,g in df.groupby(df.index.year)]
        drows=Parallel(n_jobs=N_JOBS,backend="loky",verbose=JOBLIB_VERBOSE)(delayed(_annual_dfa_one)(y,v) for y,v in annual_tasks)
        dfa_summary=pd.DataFrame(drows); tables["DFA_Annual_Context"]=dfa_summary
        print(tabulate(dfa_summary,headers="keys",tablefmt="simple",showindex=False,floatfmt=".4f"))

        bs=dfa_biscale(df["LogRet"].values,min_window=10); sens=[]
        if bs is not None:
            print(f"Two-scale DFA: alpha_short={bs['alpha_short']:.4f}; alpha_long={bs['alpha_long']:.4f}; crossover~{bs['break_scale_bars']:.0f} bars (~{bs['break_scale_bars']/max(nret,1):.1f} days); DeltaBIC={bs['delta_bic_favor_two_lines']:+.2f}")
            tables["DFA_TwoScale_Full_Sample"]=pd.DataFrame([{k:v for k,v in bs.items() if k not in ("scales","F","split")}])
        for mw in (10,12,15):
            z=dfa_biscale(df["LogRet"].values,min_window=mw)
            if z is not None:
                sens.append({"min_window":mw,"alpha_short":z["alpha_short"],"R2_short":z["r2_short"],"alpha_long":z["alpha_long"],"R2_long":z["r2_long"],"break_scale_bars":z["break_scale_bars"],"break_scale_approx_days":z["break_scale_bars"]/max(nret,1),"Delta_BIC_favor_two_lines":z["delta_bic_favor_two_lines"],"n_short":z["n_short"],"n_long":z["n_long"]})
        tables["DFA_TwoScale_Sensitivity"]=pd.DataFrame(sens)

        # Parallel segment-level DFA.
        seg_tasks=[]
        for _,row in bounds_df.iterrows():
            start=pd.Timestamp(row["Date_Start"]); end=pd.Timestamp(row["Date_End"])+pd.Timedelta(days=1)
            vals=df.loc[(df.index>=start)&(df.index<end),"LogRet"].to_numpy(dtype=float)
            seg_tasks.append((int(row["Regime_ID"]),int(row["Days"]),vals))
        segrows=Parallel(n_jobs=N_JOBS,backend="loky",max_nbytes="10M",mmap_mode="r",verbose=JOBLIB_VERBOSE)(delayed(_segment_dfa_one)(rid,days,vals) for rid,days,vals in seg_tasks)
        tables["DFA_by_Consensus_Segment"]=pd.DataFrame(segrows)

        payload={"daily":daily,"robust_break_dates":robust,"regime_bounds_df":bounds_df,"lag1_autocorr":lag1,
                 "pelt_consensus_df":consensus,"ms_episodes":episodes,"regime_weighted_means":means}
        save_checkpoint(STAGE,payload); save_stage_tables(STAGE,tables); daily.to_csv(DAILY_CSV)
        mark_stage(2,"complete",f"{len(robust)} robust changepoints; {len(bounds_df)} segments")
        print("Stage 2 saved. daily.csv enriched with Regime_ID.")

if __name__=="__main__":
    import argparse
    p=argparse.ArgumentParser(); p.add_argument("--force",action="store_true"); a=p.parse_args(); main(a.force)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage3_rolling.py
# -*- coding: utf-8 -*-
"""Stage 3: rolling DFA, correlations, HAR and OOS."""
from config import *
from tabulate import tabulate
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

STAGE="stage3_rolling"


def main(force=False):
    if checkpoint_exists(STAGE) and not force:
        print(f"[{STAGE}] Checkpoint already exists. Use --force to recompute."); return
    if force:
        for p in CHECKPOINT_DIR.glob("rolling_*d_*.pkl"):
            p.unlink()
    s1=load_checkpoint("stage1_data"); s2=load_checkpoint("stage2_regime")
    df=s1["df"]; daily=s2["daily"].copy()
    with stage_logging(STAGE):
        mark_stage(3,"running"); tables={}
        print(f"Multi-horizon rolling with {N_JOBS} processes. Each window is checkpointed separately.")
        rolling=rolling_vol_and_dfa_parallel_resume(df,daily,ROLLING_WINDOWS,N_JOBS)
        rolling.to_csv(ROLLING_CSV)
        print(f"Rolling generated: {rolling.shape[0]:,} rows x {rolling.shape[1]} cols")

        # Main figure.
        fig,ax1=plt.subplots(figsize=(12,5)); ax1.plot(rolling.index,rolling[f"Jump_{PRIMARY_WINDOW}d"],lw=1); ax1.set_ylabel(f"Jump proxy {PRIMARY_WINDOW}d")
        ax2=ax1.twinx(); ax2.plot(rolling.index,rolling[f"D_H_{PRIMARY_WINDOW}d"],lw=1,alpha=.8); ax2.set_ylabel(f"|H-0.5| {PRIMARY_WINDOW}d"); fig.tight_layout(); fig.savefig(FIG_DIR/"rolling_Jump_vs_DH_primary.png",dpi=150); plt.close(fig)

        # Actual stationarity of central variables.
        stats=[stationarity_report(df["LogRet"],"LogRet intraday (15min)")]
        for w in ROLLING_WINDOWS:
            for col in (f"H_{w}d",f"D_H_{w}d",f"Jump_{w}d",f"YZ_{w}d"):
                stats.append(stationarity_report(rolling[col],col))
        stat_df=pd.DataFrame(stats); tables["Stationarity_ADF_KPSS"]=stat_df
        print(tabulate(stat_df,headers="keys",tablefmt="simple",showindex=False,floatfmt=".4f"))

        # Central association + FDR families.
        primary_pairs=[]; yz_pairs=[]; change_pairs=[]; corr_rows=[]
        for w in ROLLING_WINDOWS:
            cols=[f"H_{w}d",f"D_H_{w}d",f"YZ_{w}d",f"Jump_{w}d",f"R2_{w}d"]
            sub=rolling[cols].dropna().copy(); n0=len(sub); sub=sub[sub[f"R2_{w}d"]>=DFA_R2_MIN]
            if len(sub)<30: continue
            jump=sub[f"Jump_{w}d"]; dh=sub[f"D_H_{w}d"]; H=sub[f"H_{w}d"]; yz=sub[f"YZ_{w}d"]
            rp,_=pearsonr(jump,dh); _,pp,_=newey_west_corr_pvalue(jump,dh,w); rs,_=spearmanr(jump,dh)
            _,ps,_=newey_west_corr_pvalue(pd.Series(jump).rank().values,pd.Series(dh).rank().values,w)
            ry,_=pearsonr(yz,H); _,py,_=newey_west_corr_pvalue(yz,H,w)
            d=pd.DataFrame({"dJump":jump.diff(),"dDH":dh.diff()}).dropna(); rd,pdlt,_=newey_west_corr_pvalue(d["dJump"],d["dDH"],max(1,min(w,30))) if len(d)>=30 else (np.nan,np.nan,None)
            robust_discrepancy_check(jump,dh,f"Jump_{w}d",f"D_H_{w}d")
            primary_pairs += [(f"Jump_DH_{w}d_Pearson",pp),(f"Jump_DH_{w}d_Spearman",ps)]; yz_pairs.append((f"YZ_H_{w}d_Pearson",py)); change_pairs.append((f"DeltaJump_DeltaDH_{w}d",pdlt))
            corr_rows.append({"Window_days":w,"N_obs":len(sub),"N_excluded_R2":n0-len(sub),"Pearson_r_Jump_DH":rp,"Pearson_p_HAC":pp,"Spearman_rho_Jump_DH":rs,"Spearman_p_HAC_rangos":ps,"Delta_r":rd,"Delta_p_HAC":pdlt,"YZ_H_r":ry,"YZ_H_p_HAC":py})
            print(f"{w}d: Jump-DH Pearson={rp:+.3f} pHAC={pp:.4g}; Spearman={rs:+.3f} pHAC={ps:.4g}; delta ={rd:+.3f} p={pdlt:.4g}")
        corr_df=pd.DataFrame(corr_rows); tables["MultiHorizon_Correlations"]=corr_df
        fdr_primary=fdr_table_from_pairs(primary_pairs,"Primary_Jump_vs_DH"); fdr_yz=fdr_table_from_pairs(yz_pairs,"Secondary_YZ_vs_H"); fdr_changes=fdr_table_from_pairs(change_pairs,"Robustness_First_Differences")

        if len(corr_df):
            fig,ax=plt.subplots(figsize=(7,5)); ax.plot(corr_df["Window_days"],corr_df["Pearson_r_Jump_DH"],"o-",label="Pearson"); ax.plot(corr_df["Window_days"],corr_df["Spearman_rho_Jump_DH"],"s--",label="Spearman"); ax.axhline(0,ls=":"); ax.legend(); ax.set_xlabel("Window (days)"); ax.set_ylabel("corr Jump vs |H-.5|"); fig.tight_layout(); fig.savefig(FIG_DIR/"perfil_horizon_Jump_DH.png",dpi=150); plt.close(fig)

        # Macro: same sample / same horizon.
        macro_rows=[]; macro_detail=[]; macro_pairs=[]; extra_stats=[]
        broad=pd.Series(dtype=float); vix=pd.Series(dtype=float)
        if USD_PATH.exists() and VIX_PATH.exists():
            broad=load_fred_series(USD_PATH,"Broad_USD_DTWEXBGS",STUDY_END_DATE); vix=load_fred_series(VIX_PATH,"VIX",STUDY_END_DATE)
            extra_stats += [stationarity_report(broad,"Broad_USD_DTWEXBGS level"),stationarity_report(vix,"VIX level")]
            ba=broad.reindex(daily.index).ffill(limit=5); va=vix.reindex(daily.index).ffill(limit=5)
            for w in ROLLING_WINDOWS:
                ur=np.log(ba/ba.shift(w)).rename("BroadUSD_Return_w"); vm=va.rolling(w,min_periods=max(10,int(.8*w))).mean().rename("VIX_Mean_w")
                extra_stats += [stationarity_report(ur,f"BroadUSD log-return {w}d"),stationarity_report(vm,f"VIX mean {w}d")]
                sub=rolling[[f"D_H_{w}d",f"Jump_{w}d",f"R2_{w}d"]].dropna(); sub=sub[sub[f"R2_{w}d"]>=DFA_R2_MIN]
                mrg=pd.concat([sub[f"D_H_{w}d"].rename("dist_H"),sub[f"Jump_{w}d"].rename("Jump"),ur,vm],axis=1).dropna()
                if len(mrg)<60: continue
                m0=sm.OLS(mrg["dist_H"],sm.add_constant(mrg[["Jump"]])).fit(cov_type="HAC",cov_kwds={"maxlags":w})
                m1=sm.OLS(mrg["dist_H"],sm.add_constant(mrg[["Jump","BroadUSD_Return_w","VIX_Mean_w"]])).fit(cov_type="HAC",cov_kwds={"maxlags":w})
                z=(mrg-mrg.mean())/mrg.std(ddof=0).replace(0,np.nan); z=z.dropna(); beta_std=np.nan
                if len(z)>=60:
                    mz=sm.OLS(z["dist_H"],sm.add_constant(z[["Jump","BroadUSD_Return_w","VIX_Mean_w"]])).fit(cov_type="HAC",cov_kwds={"maxlags":w}); beta_std=mz.params["Jump"]
                macro_rows.append({"Window_days":w,"N_sample_common":int(m1.nobs),"Date_start":mrg.index.min(),"Date_end":mrg.index.max(),"Beta_Jump_M0_same_sample":m0.params["Jump"],"p_Jump_M0_HAC":m0.pvalues["Jump"],"Beta_Jump_M1_controls":m1.params["Jump"],"p_Jump_M1_HAC":m1.pvalues["Jump"],"Beta_std_Jump_M1":beta_std,"R2_M0":m0.rsquared,"R2_M1":m1.rsquared})
                macro_pairs.append((f"Jump_controlled_macro_{w}d",m1.pvalues["Jump"]))
                for mn,mdl in [("M0_same_sample",m0),("M1_controls_horizon",m1)]:
                    ci=mdl.conf_int()
                    for term in mdl.params.index:
                        macro_detail.append({"Window_days":w,"Model":mn,"Variable":term,"Coef":mdl.params[term],"Std_Err_HAC":mdl.bse[term],"z_t":mdl.tvalues[term],"p_value":mdl.pvalues[term],"IC95_inf":ci.loc[term,0],"IC95_sup":ci.loc[term,1],"N_obs":int(mdl.nobs),"R2":mdl.rsquared})
                print(f"Macro {w}d: M0 pJump={m0.pvalues['Jump']:.4g}; M1 pJump={m1.pvalues['Jump']:.4g}")
        else:
            print("[Macro] DTWEXBGS/VIXCLS missing; block skipped")
        if extra_stats: tables["Stationarity_ADF_KPSS"]=pd.concat([stat_df,pd.DataFrame(extra_stats)],ignore_index=True)
        tables["Macro_M0_vs_M1_Same_Sample"]=pd.DataFrame(macro_rows); tables["Macro_Regression_Detail"]=pd.DataFrame(macro_detail)
        fdr_macro=fdr_table_from_pairs(macro_pairs,"Robustness_Macro_Jump_controlled")
        fdr_all=pd.concat([fdr_primary,fdr_yz,fdr_changes,fdr_macro],ignore_index=True); tables["FDR_Predefined_Families"]=fdr_all
        print("Primary FDR:"); print(tabulate(fdr_primary,headers="keys",tablefmt="simple",showindex=False,floatfmt=".4f"))

        # HAR with no look-ahead.
        hname=f"H_{PRIMARY_WINDOW}d_lag1"; hlag=rolling[f"H_{PRIMARY_WINDOW}d"].shift(1).rename(hname); har=build_har_features(daily["RV"]).join(hlag).dropna(); har_rows=[]
        if len(har)>=200:
            split=int(len(har)*.8); tr,te=har.iloc[:split],har.iloc[split:]
            mb=sm.OLS(tr["RV_target"],sm.add_constant(tr[["RV_d","RV_w","RV_m"]])).fit(); me=sm.OLS(tr["RV_target"],sm.add_constant(tr[["RV_d","RV_w","RV_m",hname]])).fit()
            pb=mb.predict(sm.add_constant(te[["RV_d","RV_w","RV_m"]],has_constant="add")); pe=me.predict(sm.add_constant(te[["RV_d","RV_w","RV_m",hname]],has_constant="add"))
            eb=te["RV_target"]-pb; ee=te["RV_target"]-pe; rb=float(np.sqrt(np.mean(eb**2))); re=float(np.sqrt(np.mean(ee**2))); ld=(eb**2-ee**2).values
            lm=sm.OLS(ld,np.ones((len(ld),1))).fit(cov_type="HAC",cov_kwds={"maxlags":5}); lp=float(lm.pvalues[0])
            print(f"HAR holdout: RMSE base={rb:.6f}, +H(t-1)={re:.6f}, improvement={(rb-re)/rb*100:+.3f}%, p_loss={lp:.4g}")
            for mn,mdl,rmse in [("HAR_base",mb,rb),("HAR_more_H_lag1",me,re)]:
                for term in mdl.params.index: har_rows.append({"Model":mn,"Variable":term,"Coef":mdl.params[term],"p_value_in_sample":mdl.pvalues[term],"RMSE_holdout":rmse,"LossDiff_p_HAC":lp})
        tables["HAR_RV_OOS_No_Lookahead"]=pd.DataFrame(har_rows)

        # Purged multi-fold OOS.
        of=[]; osum=[]; oy=[]
        for w in ROLLING_WINDOWS:
            sub=rolling[[f"Jump_{w}d",f"D_H_{w}d",f"H_{w}d",f"YZ_{w}d",f"R2_{w}d"]].dropna(); sub=sub[sub[f"R2_{w}d"]>=DFA_R2_MIN]
            c=purged_walk_forward_corr(sub,f"Jump_{w}d",f"D_H_{w}d",w)
            if len(c):
                of.append(c); osum.append({"Window_days":w,"N_folds_valid":len(c),"Median_r_test":float(c["r_test"].median()),"Fraction_r_test_negative":float((c["r_test"]<0).mean()),"Fraction_p_test_less_than_0.10":float((c["p_test_HAC"]<.10).mean()),"Fraction_sign_train_test_equal":float(c["Sign_equal"].mean())})
                print(f"OOS {w}d: median r_test={c['r_test'].median():+.3f}; negative sign={(c['r_test']<0).mean():.0%}")
            y=purged_walk_forward_corr(sub,f"YZ_{w}d",f"H_{w}d",w)
            if len(y): y["Relation"]="YZ_vs_H_secondary"; oy.append(y)
        tables["OOS_Purged_Jump_DH_Folds"]=pd.concat(of,ignore_index=True) if of else pd.DataFrame(); tables["OOS_Purged_Jump_DH_Summary"]=pd.DataFrame(osum); tables["OOS_Purged_YZ_H_Folds"]=pd.concat(oy,ignore_index=True) if oy else pd.DataFrame()

        payload={"rolling_df":rolling,"correlation_df":corr_df,"fdr_primary":fdr_primary,"fdr_all":fdr_all,
                 "macro_available":bool(USD_PATH.exists() and VIX_PATH.exists())}
        save_checkpoint(STAGE,payload); save_stage_tables(STAGE,tables); rolling.to_csv(ROLLING_CSV)
        mark_stage(3,"complete",f"rolling {ROLLING_WINDOWS}; {len(corr_df)} horizons valid")
        print("Stage 3 saved. Stage 4 can be started in a subsequent session.")

if __name__=="__main__":
    import argparse
    p=argparse.ArgumentParser(); p.add_argument("--force",action="store_true"); a=p.parse_args(); main(a.force)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage4_multifractal.py
# -*- coding: utf-8 -*-
"""Stage 4: MF-DFA with surrogates and final export."""
from config import *
from tabulate import tabulate
import platform
import importlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

STAGE="stage4_multifractal"


def _pkg_version(name):
    try:
        return getattr(importlib.import_module(name),"__version__","unknown")
    except Exception:
        return "not available"


def main(force=False):
    if checkpoint_exists(STAGE) and not force:
        print(f"[{STAGE}] Final checkpoint already exists. Use --force to regenerate it."); return
    if force:
        for p in CHECKPOINT_DIR.glob("mfdfa_*_progress_*.pkl"):
            p.unlink()
    s1=load_checkpoint("stage1_data"); s2=load_checkpoint("stage2_regime"); s3=load_checkpoint("stage3_rolling")
    df=s1["df"]; daily=s2["daily"].copy(); rolling=s3["rolling_df"]; corr=s3["correlation_df"]
    with stage_logging(STAGE):
        mark_stage(4,"running"); tables={}
        x=df["LogRet"].to_numpy(dtype=float); x=x[np.isfinite(x)]
        print(f"Original MF-DFA on {len(x):,} valid returns.")
        mf=mf_dfa(x)
        if mf is None:
            raise RuntimeError("MF-DFA could not be estimated.")
        hr=mfdfa_range(mf); av=mf["alpha"].dropna().to_numpy(float); da=float(av.max()-av.min()) if len(av)>=2 else np.nan
        print(tabulate(mf,headers="keys",tablefmt="simple",showindex=False,floatfmt=".4f"))
        print(f"h(q) range={hr:.4f}; Delta_alpha={da:.4f}")

        floors=[]
        for floor in MF_FLOOR_SENSITIVITY:
            z=mf_dfa(x,variance_floor=floor); a=z["alpha"].dropna().to_numpy(float) if z is not None else np.array([])
            floors.append({"Variance_floor":floor,"h_q_range":mfdfa_range(z) if z is not None else np.nan,"Delta_alpha":float(a.max()-a.min()) if len(a)>=2 else np.nan})
        floor_df=pd.DataFrame(floors); tables["MFDFA_hq_Spectrum"]=mf.assign(h_q_range=hr,Delta_alpha=da); tables["MFDFA_Floor_Sensitivity"]=floor_df

        # Shuffles: parallel batches + granular checkpoint.
        print(f"\nShuffle test: {N_SURROGATES} surrogates, batches={SURROGATE_BATCH_SIZE}, processes={N_JOBS}")
        sh=mfdfa_shuffle_resume(x,N_SURROGATES)
        shr=sh["h_range_shuffled"].dropna().to_numpy(float)
        p_sh=(np.sum(shr>=hr)+1)/(len(shr)+1) if len(shr) else np.nan; q95=float(np.percentile(shr,95)) if len(shr) else np.nan
        print(f"Shuffle: mean={np.nanmean(shr):.4f}; p95={q95:.4f}; p_emp={p_sh:.4f}; n={len(shr)}")
        tables["MFDFA_Surrogates_Shuffle"]=sh

        # IAAFT: parallel batches + granular checkpoint.
        print(f"\nIAAFT: {N_IAAFT_SURROGATES} surrogates, max_iter={IAAFT_MAX_ITER}, processes={N_JOBS}")
        ia=mfdfa_iaaft_resume(x,N_IAAFT_SURROGATES)
        iar=ia["h_range_IAAFT"].dropna().to_numpy(float)
        p_ia=(np.sum(iar>=hr)+1)/(len(iar)+1) if len(iar) else np.nan; q95ia=float(np.percentile(iar,95)) if len(iar) else np.nan
        spec_med=float(ia["Error_spectral_rel_L2"].median()) if len(ia) else np.nan; spec_max=float(ia["Error_spectral_rel_L2"].max()) if len(ia) else np.nan
        print(f"IAAFT: mean h_range={np.nanmean(iar):.4f}; p95={q95ia:.4f}; p_emp={p_ia:.4f}; n={len(iar)}")
        print(f"IAAFT spectral fidelity: median L2 error={spec_med:.3e}; max={spec_max:.3e}")
        tables["MFDFA_Surrogates_IAAFT"]=ia
        summary=pd.DataFrame([{
            "h_range_original":hr,"Delta_alpha_original":da,
            "shuffle_mean":float(np.nanmean(shr)) if len(shr) else np.nan,"shuffle_p95":q95,"shuffle_p_emp":p_sh,"N_shuffle_valid":len(shr),
            "IAAFT_mean":float(np.nanmean(iar)) if len(iar) else np.nan,"IAAFT_p95":q95ia,"IAAFT_p_emp":p_ia,"N_IAAFT_valid":len(iar),
            "IAAFT_error_spectral_median":spec_med,"IAAFT_error_spectral_max":spec_max,
            "Inference_shuffle":"marginal alone insufficient" if np.isfinite(p_sh) and p_sh<.05 else "not distinguishable from marginal",
            "Inference_IAAFT":"additional nonlinear temporal structure" if np.isfinite(p_ia) and p_ia<.05 else "not distinguishable from linear+marginal null",
        }]); tables["MFDFA_Surrogates_Summary"]=summary

        # MF-DFA figure.
        fig,ax=plt.subplots(figsize=(6,5)); ax.plot(mf["q"],mf["h(q)"],"o-"); ax.axhline(.5,ls=":"); ax.set_xlabel("q"); ax.set_ylabel("h(q)"); ax.set_title("MF-DFA: h(q) spectrum"); fig.tight_layout(); fig.savefig(FIG_DIR/"MFDFA_spectrum_hq.png",dpi=150); plt.close(fig)

        print("\n15m vs 30m frequency robustness: recomputing Jump-DH.")
        d30raw=resample_ohlc(df,"30min"); d30,day30=realized_daily_from_frequency(d30raw,30)
        comp=pd.DataFrame({"BV_RV2_Ratio_15min_mean":[daily["BV_to_RV2_Ratio_Pct"].mean()],"BV_RV2_Ratio_30min_mean":[day30["BV_to_RV2_Ratio_Pct"].mean()],"Jump_15min_mean":[daily["Jump_Component"].mean()],"Jump_30min_mean":[day30["Jump_Component"].mean()]})
        cj=pd.concat([daily["Jump_Component"].rename("Jump15"),day30["Jump_Component"].rename("Jump30")],axis=1).dropna()
        if len(cj)>=30:
            r,p=pearsonr(cj["Jump15"],cj["Jump30"]); comp["Corr_daily_Jump15_Jump30"]=r; comp["p_corr_daily"]=p
        roll30=rolling_jump_h_frequency(d30,day30,MICROSTRUCTURE_WINDOWS,n_jobs=N_JOBS)
        rows=[]
        for w in MICROSTRUCTURE_WINDOWS:
            sub=roll30[[f"Jump30_{w}d",f"H30_{w}d",f"R2_30_{w}d"]].dropna(); sub=sub[sub[f"R2_30_{w}d"]>=DFA_R2_MIN].copy(); sub["D_H30"]=(sub[f"H30_{w}d"]-.5).abs()
            if len(sub)>=30:
                r30,p30,_=newey_west_corr_pvalue(sub[f"Jump30_{w}d"],sub["D_H30"],w); rho,_=spearmanr(sub[f"Jump30_{w}d"],sub["D_H30"]); _,ps,_=newey_west_corr_pvalue(sub[f"Jump30_{w}d"].rank(),sub["D_H30"].rank(),w)
            else: r30=p30=rho=ps=np.nan
            base=corr.loc[corr["Window_days"]==w]
            r15=float(base.iloc[0]["Pearson_r_Jump_DH"]) if len(base) else np.nan; p15=float(base.iloc[0]["Pearson_p_HAC"]) if len(base) else np.nan; s15=float(base.iloc[0]["Spearman_rho_Jump_DH"]) if len(base) else np.nan
            rows.append({"Window_days":w,"Pearson_r_15min":r15,"p_HAC_15min":p15,"Spearman_15min":s15,"Pearson_r_30min":r30,"p_HAC_30min":p30,"Spearman_30min":rho,"p_HAC_rangos_30min":ps,"Same_sign_Pearson":bool(np.sign(r15)==np.sign(r30)) if np.isfinite(r15) and np.isfinite(r30) else False,"N_30min":len(sub)})
            print(f"{w}d: r15={r15:+.3f}; r30={r30:+.3f}; p30={p30:.4g}")
        tables["Frequency_Robustness_Summary"]=comp; tables["Jump_DH_Frequency_Robustness"]=pd.DataFrame(rows)

        # Final metadata.
        input_sha=s1["input_sha"]; usd_sha=file_sha256(USD_PATH) if USD_PATH.exists() else "UNAVAILABLE"; vix_sha=file_sha256(VIX_PATH) if VIX_PATH.exists() else "UNAVAILABLE"
        meta=[
            {"Parameter":"Methodological version","Value":PIPELINE_VERSION},
            {"Parameter":"Final execution date","Value":pd.Timestamp.now().isoformat()},
            {"Parameter":"CPU workers joblib","Value":N_JOBS},
            {"Parameter":"BLAS threads by worker","Value":"1 (OMP/MKL/OpenBLAS/NUMEXPR)"},
            {"Parameter":"OHLCV file","Value":str(OHLCV_PATH)}, {"Parameter":"SHA256 OHLCV","Value":input_sha},
            {"Parameter":"XAUUSD data provider","Value":DATA_VENDOR},{"Parameter":"Source time zone","Value":SOURCE_TIMEZONE},{"Parameter":"Session/day","Value":SESSION_DEFINITION},
            {"Parameter":"Date range","Value":f"{df.index.min()} -> {df.index.max()}"},{"Parameter":"STUDY_END_DATE","Value":str(STUDY_END_DATE)},
            {"Parameter":"Loaded bars","Value":len(df)},{"Parameter":"Days","Value":len(daily)},
            {"Parameter":"Median bars/day","Value":s1["N_CANDLES_MEDIAN"]},{"Parameter":"Median returns/day","Value":s1["N_RETURNS_MEDIAN"]},
            {"Parameter":"Effective trading days/year","Value":s1["ANNUALIZATION_DAYS"]},{"Parameter":"Rolling windows","Value":str(ROLLING_WINDOWS)},
            {"Parameter":"PELT min_size","Value":str(PELT_MIN_SIZES)},{"Parameter":"PELT penalties","Value":str(PELT_PENALTY_MULTIPLIERS)},{"Parameter":"Robust PELT change points","Value":len(s2["robust_break_dates"])},
            {"Parameter":"MS-AR search_reps","Value":MS_SEARCH_REPS},{"Parameter":"Random seed","Value":RANDOM_SEED},
            {"Parameter":"MF-DFA q","Value":str(MF_Q_VALUES)},{"Parameter":"N shuffles","Value":N_SURROGATES},{"Parameter":"N IAAFT","Value":N_IAAFT_SURROGATES},
            {"Parameter":"IAAFT max_iter/tol","Value":f"{IAAFT_MAX_ITER}/{IAAFT_CONVERGENCE_TOL}"},{"Parameter":"Median IAAFT spectral error","Value":spec_med},
            {"Parameter":"OOS folds","Value":OOS_N_FOLDS},{"Parameter":"OOS purge","Value":"equal to window length"},
            {"Parameter":"Series USD","Value":"FRED DTWEXBGS Nominal Broad U.S. Dollar Index"},{"Parameter":"SHA256 DTWEXBGS","Value":usd_sha},{"Parameter":"SHA256 VIX","Value":vix_sha},
            {"Parameter":"Python","Value":platform.python_version()},{"Parameter":"pandas","Value":_pkg_version("pandas")},{"Parameter":"numpy","Value":_pkg_version("numpy")},{"Parameter":"scipy","Value":_pkg_version("scipy")},{"Parameter":"statsmodels","Value":_pkg_version("statsmodels")},{"Parameter":"ruptures","Value":_pkg_version("ruptures")},{"Parameter":"joblib","Value":_pkg_version("joblib")},
        ]
        tables["Parameters_Reproducibility"]=pd.DataFrame(meta)

        # Merge all four stages and produce final Excel.
        all_tables={}
        for name in ("stage1_data","stage2_regime","stage3_rolling"):
            all_tables.update(load_stage_tables(name))
        all_tables.update(tables)
        path=export_all_tables(all_tables,FINAL_EXCEL)
        daily.to_csv(DAILY_CSV); rolling.to_csv(ROLLING_CSV)
        save_checkpoint(STAGE,{"mfdfa_result":mf,"mfdfa_summary":summary,"final_excel":str(path)})
        save_stage_tables(STAGE,tables); mark_stage(4,"complete",f"Excel final: {path}")
        print(f"\nPipeline completed. Final Excel: {path}")
        print(f"If IAAFT/shuffle is interrupted, completed surrogates are retained on rerun in {CHECKPOINT_DIR}.")

if __name__=="__main__":
    import argparse
    p=argparse.ArgumentParser(); p.add_argument("--force",action="store_true"); a=p.parse_args(); main(a.force)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage5_mechanism.py
# -*- coding: utf-8 -*-
"""Stage 5: BNS jump test and event study."""
from __future__ import annotations

import os
import json
import math
import hashlib
from pathlib import Path

from config import *
from scipy.special import gamma
from scipy.stats import norm, fisher_exact, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from tabulate import tabulate

STAGE = "stage5_mechanism_v3"
STAGE5_VERSION = "XAUUSD_STAGE5_BNS_EVENT_v3_2026-09-11"

BNS_FDR_Q = 0.05
BNS_STRICT_RAW_P = 0.001
MIN_RETURNS_PER_DAY = 40
MIN_ADJACENT_PAIRS = 30
MIN_ADJACENT_TRIPLETS = 20
EVENT_WINDOWS = (30, 60)       # trading days before and after
PRIMARY_EVENT_WINDOW = 30
JUMP_EPISODE_GAP_DAYS = 2
MATCH_PREVOL_DAYS = 20
DFA_EVENT_R2_MIN = DFA_R2_MIN
RANDOMIZATION_REPS = 20000
BOOTSTRAP_REPS = 10000
WILD_CLUSTER_REPS = 49999
STAGE5_SEED = 20260911

STAGE5_EXCEL = OUTPUT_DIR / "stage5_mechanism_v3_results.xlsx"


def _fingerprint_stage5(input_sha: str) -> str:
    d = {
        "stage5_version": STAGE5_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "input_sha": input_sha,
        "expected_bar_minutes": EXPECTED_BAR_MINUTES,
        "bns_fdr_q": BNS_FDR_Q,
        "bns_strict_raw_p": BNS_STRICT_RAW_P,
        "min_returns_day": MIN_RETURNS_PER_DAY,
        "event_windows": EVENT_WINDOWS,
        "jump_episode_gap_days": JUMP_EPISODE_GAP_DAYS,
        "selection": "all_FDR_jump_days_max_cardinality_then_max_sum_Z_nonoverlap",
        "placebo_match": "same_year_preRV_DHpre_no_replacement",
        "match_prevol_days": MATCH_PREVOL_DAYS,
        "dfa_r2_min": DFA_EVENT_R2_MIN,
    }
    return hashlib.sha256(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:16]


def _save_stage5_cp(path: Path, payload: dict, fingerprint: str):
    pd.to_pickle({
        "stage5_version": STAGE5_VERSION,
        "fingerprint": fingerprint,
        "saved_at": pd.Timestamp.now().isoformat(),
        "payload": payload,
    }, path)


def _load_stage5_cp(path: Path, fingerprint: str):
    if not path.exists():
        return None
    obj = pd.read_pickle(path)
    if obj.get("stage5_version") != STAGE5_VERSION or obj.get("fingerprint") != fingerprint:
        return None
    return obj.get("payload")


def _bns_one_day(g: pd.DataFrame) -> dict:
    g = g.sort_index()
    r = pd.to_numeric(g["LogRet"], errors="coerce").to_numpy(float)
    ts = g.index.view("i8")
    finite = np.isfinite(r)
    M = int(finite.sum())
    if M < MIN_RETURNS_PER_DAY:
        return {"N_returns": M}

    step_ns = int(pd.Timedelta(minutes=EXPECTED_BAR_MINUTES).value)
    consecutive_1 = (ts[1:] - ts[:-1]) == step_ns
    pair_mask = finite[1:] & finite[:-1] & consecutive_1
    P = int(pair_mask.sum())

    if len(r) >= 3:
        consecutive_2 = ((ts[2:] - ts[1:-1]) == step_ns) & ((ts[1:-1] - ts[:-2]) == step_ns)
        tri_mask = finite[2:] & finite[1:-1] & finite[:-2] & consecutive_2
        T = int(tri_mask.sum())
    else:
        tri_mask = np.zeros(0, dtype=bool); T = 0

    if P < MIN_ADJACENT_PAIRS or T < MIN_ADJACENT_TRIPLETS:
        return {"N_returns": M, "N_pairs": P, "N_triplets": T}

    rv = float(np.sum(r[finite] ** 2))
    if not np.isfinite(rv) or rv <= 0:
        return {"N_returns": M, "N_pairs": P, "N_triplets": T}

    # mu_1 = E|N(0,1)| = sqrt(2/pi)
    mu1 = math.sqrt(2.0 / math.pi)
    bv_raw = float(np.sum(np.abs(r[1:][pair_mask]) * np.abs(r[:-1][pair_mask])))
    bv = (mu1 ** -2) * (M / P) * bv_raw

    mu43 = (2.0 ** (2.0 / 3.0)) * gamma(7.0 / 6.0) / gamma(0.5)
    tri_prod = (np.abs(r[2:][tri_mask]) ** (4.0 / 3.0) *
                np.abs(r[1:-1][tri_mask]) ** (4.0 / 3.0) *
                np.abs(r[:-2][tri_mask]) ** (4.0 / 3.0))
    tq = M * (mu43 ** -3) * (M / T) * float(np.sum(tri_prod))

    omega = (mu1 ** -4) + 2.0 * (mu1 ** -2) - 5.0
    if not np.isfinite(bv) or bv <= 0 or not np.isfinite(tq) or tq <= 0:
        return {"N_returns": M, "N_pairs": P, "N_triplets": T,
                "RV_var": rv, "BV": bv, "TQ": tq}

    denom = math.sqrt(omega * max(1.0, tq / (bv ** 2)))
    z = math.sqrt(M) * (1.0 - bv / rv) / denom if denom > 0 else np.nan
    p_one = float(norm.sf(z)) if np.isfinite(z) else np.nan
    jump_var = max(rv - bv, 0.0)
    return {
        "N_returns": M, "N_pairs": P, "N_triplets": T,
        "RV_var": rv, "BV": bv, "TQ": tq,
        "JumpVar_raw": jump_var,
        "JumpShare_RV": jump_var / rv if rv > 0 else np.nan,
        "Z_BNS": z, "p_BNS_one_sided": p_one,
    }


def compute_bns_daily(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for date, g in df.groupby(df.index.normalize(), sort=True):
        d = _bns_one_day(g)
        d["Date"] = pd.Timestamp(date)
        rows.append(d)
    out = pd.DataFrame(rows).set_index("Date").sort_index()
    for col in ("p_BNS_one_sided", "JumpVar_raw", "Z_BNS"):
        if col not in out.columns:
            out[col] = np.nan
    out["p_BNS_FDR"] = np.nan
    out["Jump_BNS_FDR"] = False
    valid = out["p_BNS_one_sided"].notna()
    if valid.any():
        reject, padj, _, _ = multipletests(out.loc[valid, "p_BNS_one_sided"].values,
                                           alpha=BNS_FDR_Q, method="fdr_bh")
        out.loc[valid, "p_BNS_FDR"] = padj
        out.loc[valid, "Jump_BNS_FDR"] = reject
    out["Jump_BNS_p0001"] = out["p_BNS_one_sided"] < BNS_STRICT_RAW_P
    out["JumpVar_BNS_FDR"] = np.where(out["Jump_BNS_FDR"], out["JumpVar_raw"], 0.0)
    return out


def _returns_for_date_span(df, start_day, end_day_inclusive):
    a = pd.Timestamp(start_day)
    b = pd.Timestamp(end_day_inclusive) + pd.Timedelta(days=1)
    x = pd.to_numeric(df.loc[(df.index >= a) & (df.index < b), "LogRet"], errors="coerce").to_numpy(float)
    return x[np.isfinite(x)]


def _event_metrics_one(pos: int, L: int, dates: pd.DatetimeIndex, df: pd.DataFrame,
                       daily: pd.DataFrame, bns: pd.DataFrame) -> dict | None:
    if pos < L or pos + L >= len(dates):
        return None
    event_date = pd.Timestamp(dates[pos])
    pre_dates = dates[pos-L:pos]
    post_dates = dates[pos+1:pos+1+L]
    if len(pre_dates) != L or len(post_dates) != L:
        return None

    rpre = _returns_for_date_span(df, pre_dates[0], pre_dates[-1])
    rpost = _returns_for_date_span(df, post_dates[0], post_dates[-1])
    Hpre, R2pre, _, _ = dfa(rpre)
    Hpost, R2post, _, _ = dfa(rpost)
    if not (np.isfinite(Hpre) and np.isfinite(Hpost) and np.isfinite(R2pre) and np.isfinite(R2post)):
        return None

    pre_rv_slice = daily["RV"].reindex(pre_dates).dropna().tail(MATCH_PREVOL_DAYS)
    pre_rv = float(pre_rv_slice.mean()) if len(pre_rv_slice) else np.nan
    z = bns.at[event_date, "Z_BNS"] if event_date in bns.index else np.nan
    jv = bns.at[event_date, "JumpVar_raw"] if event_date in bns.index else np.nan
    return {
        "Date": event_date, "Pos": int(pos), "L": int(L),
        "H_pre": Hpre, "R2_pre": R2pre, "D_H_pre": abs(Hpre - 0.5), "Nret_pre": len(rpre),
        "H_post": Hpost, "R2_post": R2post, "D_H_post": abs(Hpost - 0.5), "Nret_post": len(rpost),
        "Delta_H": Hpost - Hpre,
        "Delta_D_H": abs(Hpost - 0.5) - abs(Hpre - 0.5),
        "Pre_RV_mean": pre_rv,
        "Z_BNS": z, "JumpVar_raw": jv,
    }


def _positions_far_from_jump(jump_positions: np.ndarray, candidate_pos: int, radius: int) -> bool:
    if len(jump_positions) == 0:
        return True
    return bool(np.min(np.abs(jump_positions - candidate_pos)) > radius)


def build_jump_episodes(bns: pd.DataFrame, dates: pd.DatetimeIndex):
    pos_map = {pd.Timestamp(d): i for i, d in enumerate(dates)}
    jump_dates = [pd.Timestamp(d) for d in bns.index[bns["Jump_BNS_FDR"].fillna(False)] if pd.Timestamp(d) in pos_map]
    jump_positions = np.array(sorted(pos_map[d] for d in jump_dates), dtype=int)
    if len(jump_positions) == 0:
        return pd.DataFrame(), jump_positions

    clusters = [[int(jump_positions[0])]]
    for p in jump_positions[1:]:
        p = int(p)
        if p - clusters[-1][-1] <= JUMP_EPISODE_GAP_DAYS:
            clusters[-1].append(p)
        else:
            clusters.append([p])

    rows = []
    for eid, g in enumerate(clusters):
        best = sorted(
            g,
            key=lambda p: (
                -float(bns.at[pd.Timestamp(dates[p]), "Z_BNS"]),
                p,
            )
        )[0]
        rows.append({
            "Episode_ID": eid,
            "Episode_Start": pd.Timestamp(dates[g[0]]),
            "Episode_End": pd.Timestamp(dates[g[-1]]),
            "N_jump_days": len(g),
            "Representative_Pos": int(best),
            "Representative_Date": pd.Timestamp(dates[best]),
            "Representative_Z_BNS": float(bns.at[pd.Timestamp(dates[best]), "Z_BNS"]),
        })
    return pd.DataFrame(rows), jump_positions


def _max_cardinality_nonoverlap(candidates: list[tuple[int, float]], L: int) -> list[int]:
    if not candidates:
        return []
    cand = sorted((int(p), float(z)) for p, z in candidates)
    pos = np.array([p for p, _ in cand], dtype=int)
    n = len(cand)

    # compatible predecessor of each candidate
    pred = np.empty(n, dtype=int)
    for i in range(n):
        pred[i] = int(np.searchsorted(pos, pos[i] - 2 * L, side="left") - 1)

    # DP over the first i candidates: tuple (count, sum_z)
    dp_count = np.zeros(n + 1, dtype=int)
    dp_z = np.zeros(n + 1, dtype=float)
    take = np.zeros(n, dtype=bool)

    for i in range(1, n + 1):
        p, z = cand[i - 1]
        j = pred[i - 1] + 1
        c_take = dp_count[j] + 1
        z_take = dp_z[j] + z
        c_skip = dp_count[i - 1]
        z_skip = dp_z[i - 1]
        choose_take = (c_take > c_skip) or (c_take == c_skip and z_take > z_skip + 1e-15)
        if choose_take:
            dp_count[i] = c_take
            dp_z[i] = z_take
            take[i - 1] = True
        else:
            dp_count[i] = c_skip
            dp_z[i] = z_skip

    selected = []
    i = n
    while i > 0:
        p, z = cand[i - 1]
        j = pred[i - 1] + 1
        c_take = dp_count[j] + 1
        z_take = dp_z[j] + z
        if dp_count[i] == c_take and abs(dp_z[i] - z_take) <= 1e-10:
            selected.append(p)
            i = j
        else:
            i -= 1
    return sorted(selected)


def select_nonoverlap_jump_positions(bns: pd.DataFrame, dates: pd.DatetimeIndex, L: int):
    episodes, jump_positions = build_jump_episodes(bns, dates)
    pos_map = {pd.Timestamp(d): i for i, d in enumerate(dates)}
    candidates = []
    for d in bns.index[bns["Jump_BNS_FDR"].fillna(False)]:
        d = pd.Timestamp(d)
        if d not in pos_map:
            continue
        p = int(pos_map[d])
        if p < L or p + L >= len(dates):
            continue
        z = float(bns.at[d, "Z_BNS"])
        if np.isfinite(z):
            candidates.append((p, z))
    selected = _max_cardinality_nonoverlap(candidates, L)
    diag = {
        "N_jump_days_FDR": int(len(jump_positions)),
        "N_episodes_descriptive": int(len(episodes)),
        "N_jump_candidates_with_margins": int(len(candidates)),
        "N_selected_nonoverlap": int(len(selected)),
    }
    return selected, jump_positions, episodes, diag


def _eligible_placebo_positions(dates, daily, bns, rolling, L, jump_positions):
    out = []
    hcol, r2col = f"H_{L}d", f"R2_{L}d"
    if hcol not in rolling.columns or r2col not in rolling.columns:
        raise KeyError(f"Stage 3 does not contain {hcol}/{r2col}; cannot match H_pre.")
    rindex = pd.DatetimeIndex(rolling.index).normalize()
    rmap = {pd.Timestamp(d): i for i, d in enumerate(rindex)}

    for p in range(L, len(dates) - L):
        d = pd.Timestamp(dates[p])
        if d not in bns.index or not np.isfinite(bns.at[d, "p_BNS_one_sided"]):
            continue
        if bool(bns.at[d, "Jump_BNS_FDR"]):
            continue
        if not _positions_far_from_jump(jump_positions, p, JUMP_EPISODE_GAP_DAYS):
            continue

        prev_d = pd.Timestamp(dates[p - 1])
        if prev_d not in rmap:
            continue
        rr = rolling.iloc[rmap[prev_d]]
        hpre = float(rr[hcol]) if np.isfinite(rr[hcol]) else np.nan
        r2pre = float(rr[r2col]) if np.isfinite(rr[r2col]) else np.nan
        if not np.isfinite(hpre) or not np.isfinite(r2pre) or r2pre < DFA_EVENT_R2_MIN:
            continue

        pre = daily["RV"].iloc[p - MATCH_PREVOL_DAYS:p].dropna()
        if len(pre) < max(10, MATCH_PREVOL_DAYS // 2):
            continue
        prv = float(pre.mean())
        out.append({
            "Pos": int(p), "Date": d, "Year": int(d.year),
            "Pre_RV_mean": prv,
            "log_Pre_RV": float(np.log(max(prv, np.finfo(float).tiny))),
            "H_pre_proxy": hpre,
            "D_H_pre_proxy": abs(hpre - 0.5),
            "H_side_proxy": int(hpre >= 0.5),
            "R2_pre_proxy": r2pre,
        })
    return pd.DataFrame(out)


def _robust_scale(x):
    x = pd.Series(x).dropna().to_numpy(float)
    if len(x) < 2:
        return 1.0
    med = np.median(x)
    mad = 1.4826 * np.median(np.abs(x - med))
    if not np.isfinite(mad) or mad <= 1e-12:
        mad = np.std(x, ddof=0)
    return float(mad if np.isfinite(mad) and mad > 1e-12 else 1.0)


def match_placebos(treat_df: pd.DataFrame, placebo_pool: pd.DataFrame, L: int):
    if len(treat_df) == 0 or len(placebo_pool) == 0:
        return pd.DataFrame()

    pool = placebo_pool.copy()
    rv_scale = _robust_scale(pool["log_Pre_RV"])
    dh_scale = _robust_scale(pool["D_H_pre_proxy"])
    used_pos = set()
    rows = []

    # reduce the risk of exhausting their best controls.
    order = treat_df.assign(_ext=treat_df["D_H_pre"]).sort_values(["_ext", "Date"], ascending=[False, True])
    for _, tr in order.iterrows():
        yr = int(pd.Timestamp(tr["Date"]).year)
        tr_log = float(np.log(max(tr["Pre_RV_mean"], np.finfo(float).tiny)))
        tr_h = float(tr["H_pre"])
        tr_dh = float(tr["D_H_pre"])
        tr_side = int(tr_h >= 0.5)

        base = pool[~pool["Pos"].isin(used_pos)].copy()
        if len(base) == 0:
            break

        tier_defs = [
            ("same_year", base[base["Year"] == yr]),
            ("plusminus1y", base[np.abs(base["Year"] - yr) <= 1]),
            ("all_years", base),
        ]
        tier_name, cand = next(((name, c.copy()) for name, c in tier_defs if len(c)), ("all_years", base.copy()))

        same_side = cand[cand["H_side_proxy"] == tr_side]
        side_relaxed = False
        if len(same_side):
            cand = same_side
        else:
            side_relaxed = True

        cand["dist_match"] = (
            np.abs(cand["log_Pre_RV"] - tr_log) / rv_scale +
            np.abs(cand["D_H_pre_proxy"] - tr_dh) / dh_scale
        )

        spacing_relaxed = False
        if used_pos:
            cp = cand["Pos"].to_numpy(int)
            good = np.ones(len(cand), dtype=bool)
            for up in used_pos:
                good &= np.abs(cp - int(up)) > 2 * L
            if good.any():
                cand = cand.loc[good]
            else:
                spacing_relaxed = True

        best = cand.sort_values(["dist_match", "Date"]).iloc[0]
        pp = int(best["Pos"])
        used_pos.add(pp)
        rows.append({
            "Treatment_Date": tr["Date"], "Treatment_Pos": int(tr["Pos"]),
            "Treatment_Year": yr,
            "Placebo_Date": best["Date"], "Placebo_Pos": pp,
            "Match_Time_Tier": tier_name,
            "Treatment_Pre_RV": float(tr["Pre_RV_mean"]),
            "Placebo_Pre_RV": float(best["Pre_RV_mean"]),
            "Treatment_log_PreRV": tr_log,
            "Placebo_log_PreRV": float(best["log_Pre_RV"]),
            "AbsDiff_log_PreRV": abs(float(best["log_Pre_RV"]) - tr_log),
            "Treatment_H_pre": tr_h,
            "Placebo_H_pre_proxy": float(best["H_pre_proxy"]),
            "AbsDiff_H_pre_proxy": abs(float(best["H_pre_proxy"]) - tr_h),
            "Treatment_D_H_pre": tr_dh,
            "Placebo_D_H_pre_proxy": float(best["D_H_pre_proxy"]),
            "AbsDiff_D_H_pre_proxy": abs(float(best["D_H_pre_proxy"]) - tr_dh),
            "Same_H_side_proxy": bool(int(best["H_side_proxy"]) == tr_side),
            "H_side_constraint_relaxed": bool(side_relaxed),
            "Match_distance": float(best["dist_match"]),
            "Placebo_spacing_relaxed": bool(spacing_relaxed),
        })
    return pd.DataFrame(rows)


def _cluster_se_mean(y: np.ndarray, clusters: np.ndarray) -> float:
    y = np.asarray(y, float)
    clusters = np.asarray(clusters)
    ok = np.isfinite(y)
    y, clusters = y[ok], clusters[ok]
    n = len(y)
    uniq = pd.unique(clusters)
    G = len(uniq)
    if n < 2 or G < 2:
        return np.nan
    mu = float(np.mean(y))
    score_sq = 0.0
    for g in uniq:
        score_sq += float(np.sum(y[clusters == g] - mu)) ** 2
    # CR1 for an intercept-only regression: X'X=n.
    var = (G / (G - 1.0)) * score_sq / (n ** 2)
    return float(np.sqrt(max(var, 0.0)))


def wild_cluster_bootstrap_mean_test(df: pd.DataFrame, value_col: str, cluster_col: str,
                                     alternative="less", reps=WILD_CLUSTER_REPS,
                                     seed=STAGE5_SEED):
    d = df[[value_col, cluster_col]].dropna().copy()
    if len(d) < 2:
        return np.nan, 0, np.nan, np.nan
    y = d[value_col].to_numpy(float)
    cl = d[cluster_col].to_numpy()
    uniq = pd.unique(cl)
    G = len(uniq)
    if G < 2:
        return np.nan, G, float(np.mean(y)), np.nan
    mu = float(np.mean(y))
    se = _cluster_se_mean(y, cl)
    if not np.isfinite(se) or se <= 0:
        return np.nan, G, mu, np.nan
    t_obs = mu / se
    g_index = {g:i for i,g in enumerate(uniq)}
    gi = np.array([g_index[g] for g in cl], dtype=int)

    def t_from_weights(w):
        ys = y * w[gi]
        ses = _cluster_se_mean(ys, cl)
        if not np.isfinite(ses) or ses <= 0:
            return np.nan
        return float(np.mean(ys) / ses)

    if G <= 16:
        B = 1 << G
        tstar = np.empty(B, float)
        for mask in range(B):
            w = np.array([1.0 if (mask >> j) & 1 else -1.0 for j in range(G)])
            tstar[mask] = t_from_weights(w)
    else:
        rng = np.random.default_rng(seed)
        tstar = np.empty(reps, float)
        for b in range(reps):
            w = rng.choice((-1.0, 1.0), size=G)
            tstar[b] = t_from_weights(w)
    tstar = tstar[np.isfinite(tstar)]
    if len(tstar) == 0:
        return np.nan, G, mu, t_obs
    if alternative == "less":
        p = (np.sum(tstar <= t_obs) + 1) / (len(tstar) + 1)
    elif alternative == "greater":
        p = (np.sum(tstar >= t_obs) + 1) / (len(tstar) + 1)
    else:
        p = (np.sum(np.abs(tstar) >= abs(t_obs)) + 1) / (len(tstar) + 1)
    return float(p), G, mu, float(t_obs)


def cluster_signflip_test(df: pd.DataFrame, value_col: str, cluster_col: str,
                          alternative="less", seed=STAGE5_SEED):
    if len(df) == 0 or value_col not in df.columns or cluster_col not in df.columns:
        return np.nan, 0
    cm = df.groupby(cluster_col)[value_col].mean().dropna()
    return signflip_test(cm.values, alternative=alternative, seed=seed), int(len(cm))


def cluster_bootstrap_mean_ci(df: pd.DataFrame, value_col: str, cluster_col: str,
                              reps=BOOTSTRAP_REPS, seed=STAGE5_SEED):
    d = df[[value_col, cluster_col]].dropna()
    clusters = list(d[cluster_col].unique())
    if len(clusters) < 2:
        return (np.nan, np.nan)
    vals = {c: d.loc[d[cluster_col] == c, value_col].to_numpy(float) for c in clusters}
    rng = np.random.default_rng(seed)
    means = np.empty(reps)
    for i in range(reps):
        chosen = rng.choice(clusters, size=len(clusters), replace=True)
        sample = np.concatenate([vals[c] for c in chosen])
        means[i] = np.mean(sample)
    return tuple(np.quantile(means, [0.025, 0.975]))

def signflip_test(values, alternative="less", reps=RANDOMIZATION_REPS, seed=STAGE5_SEED):
    x = np.asarray(values, float); x = x[np.isfinite(x)]
    n = len(x)
    if n == 0: return np.nan
    obs = float(np.mean(x))
    if n <= 16:
        total = 1 << n; stats = np.empty(total)
        for mask in range(total):
            signs = np.array([1.0 if (mask >> j) & 1 else -1.0 for j in range(n)])
            stats[mask] = np.mean(x * signs)
    else:
        rng = np.random.default_rng(seed)
        # Batches avoid creating a huge matrix.
        parts = []
        left = reps
        while left > 0:
            m = min(left, 2000)
            signs = rng.choice((-1.0, 1.0), size=(m, n))
            parts.append((signs * x).mean(axis=1)); left -= m
        stats = np.concatenate(parts)
    if alternative == "less":
        return float((np.sum(stats <= obs) + 1) / (len(stats) + 1))
    if alternative == "greater":
        return float((np.sum(stats >= obs) + 1) / (len(stats) + 1))
    return float((np.sum(np.abs(stats) >= abs(obs)) + 1) / (len(stats) + 1))


def bootstrap_mean_ci(values, reps=BOOTSTRAP_REPS, seed=STAGE5_SEED):
    x = np.asarray(values, float); x = x[np.isfinite(x)]
    if len(x) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = np.empty(reps)
    for i in range(reps):
        means[i] = np.mean(rng.choice(x, size=len(x), replace=True))
    return tuple(np.quantile(means, [0.025, 0.975]))


def _smd(a, b):
    a = pd.Series(a).dropna().to_numpy(float); b = pd.Series(b).dropna().to_numpy(float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    sp = np.sqrt((va + vb) / 2.0)
    if not np.isfinite(sp) or sp <= 1e-15:
        return 0.0 if abs(np.mean(a)-np.mean(b)) <= 1e-15 else np.nan
    return float((np.mean(a) - np.mean(b)) / sp)


def matching_balance_table(treat: pd.DataFrame, pool: pd.DataFrame, pairs: pd.DataFrame, L: int):
    rows=[]
    if len(treat) and len(pool):
        rows.append({
            "Window_days":L, "Moment":"BEFORE_matching_vs_pool",
            "N_treatment":len(treat), "N_control":len(pool),
            "Mean_log_PreRV_treat":float(np.log(treat["Pre_RV_mean"]).mean()),
            "Mean_log_PreRV_control":float(pool["log_Pre_RV"].mean()),
            "SMD_log_PreRV":_smd(np.log(treat["Pre_RV_mean"]),pool["log_Pre_RV"]),
            "Mean_D_H_pre_treat":float(treat["D_H_pre"].mean()),
            "Mean_D_H_pre_control":float(pool["D_H_pre_proxy"].mean()),
            "SMD_D_H_pre":_smd(treat["D_H_pre"],pool["D_H_pre_proxy"]),
        })
    if len(pairs):
        rows.append({
            "Window_days":L, "Moment":"AFTER_paired_matching",
            "N_treatment":len(pairs), "N_control":len(pairs),
            "Mean_log_PreRV_treat":float(pairs["Treatment_log_PreRV"].mean()),
            "Mean_log_PreRV_control":float(pairs["Placebo_log_PreRV"].mean()),
            "SMD_log_PreRV":_smd(pairs["Treatment_log_PreRV"],pairs["Placebo_log_PreRV"]),
            "Mean_D_H_pre_treat":float(pairs["Treatment_D_H_pre"].mean()),
            "Mean_D_H_pre_control":float(pairs["Placebo_D_H_pre_actual"].mean()),
            "SMD_D_H_pre":_smd(pairs["Treatment_D_H_pre"],pairs["Placebo_D_H_pre_actual"]),
            "Mean_absdiff_log_PreRV":float(pairs["AbsDiff_log_PreRV"].mean()),
            "Mean_absdiff_D_H_pre":float(pairs["AbsDiff_D_H_pre_actual"].mean()),
            "Frac_same_H_side":float(pairs["Same_H_side_actual"].mean()),
        })
    return pd.DataFrame(rows)

def find_macro_calendar(explicit_path=None):
    env = os.getenv("XAUUSD_MACRO_CALENDAR_PATH")
    candidates = []
    if explicit_path: candidates.append(Path(explicit_path))
    if env: candidates.append(Path(env))
    candidates += [
        BASE_DIR / "economic_calendar_publication_master.csv",
        BASE_DIR / "economic_calendar_debug.csv",
        BASE_DIR.parent / "economic_calendar_publication_master.csv",
        BASE_DIR.parent / "economic_calendar_debug.csv",
    ]
    for p in candidates:
        if p.exists(): return p
    return None


def load_macro_calendar(path: Path):
    c = pd.read_csv(path)
    date_col = next((x for x in ("date", "Date", "Date", "date") if x in c.columns), None)
    event_col = next((x for x in ("event", "Event", "Event", "event", "Type") if x in c.columns), None)
    if date_col is None or event_col is None:
        raise ValueError(f"Calendar missing recognizable date/event columns: {path}")
    c = c.copy(); c["Date"] = pd.to_datetime(c[date_col], errors="coerce").dt.normalize()
    c["Macro_Event"] = c[event_col].astype(str).str.upper().str.strip()
    c = c.dropna(subset=["Date"])
    c = c[c["Date"] <= STUDY_END_DATE.normalize()]
    c = c[c["Macro_Event"].isin(["FOMC", "CPI"])]
    return c


def macro_jump_analysis(bns: pd.DataFrame, calendar: pd.DataFrame):
    macro_by_date = calendar.groupby("Date")["Macro_Event"].agg(lambda x: "+".join(sorted(set(x))))
    x = bns.copy()
    x["Macro_Event"] = macro_by_date.reindex(x.index)
    x["Macro_Day"] = x["Macro_Event"].notna()
    valid = x["p_BNS_one_sided"].notna()
    y = x.loc[valid]
    a = int((y["Macro_Day"] & y["Jump_BNS_FDR"]).sum())
    b = int((y["Macro_Day"] & ~y["Jump_BNS_FDR"]).sum())
    c = int((~y["Macro_Day"] & y["Jump_BNS_FDR"]).sum())
    d = int((~y["Macro_Day"] & ~y["Jump_BNS_FDR"]).sum())
    if min(a+b, c+d) > 0:
        oratio, p_two = fisher_exact([[a,b],[c,d]], alternative="two-sided")
        _, p_greater = fisher_exact([[a,b],[c,d]], alternative="greater")
    else:
        oratio, p_two, p_greater = np.nan, np.nan, np.nan
    table = pd.DataFrame([{
        "Jump_macro":a, "NoJump_macro":b, "Jump_no_macro":c, "NoJump_no_macro":d,
        "P_jump_given_macro": a/(a+b) if a+b else np.nan,
        "P_jump_given_no_macro": c/(c+d) if c+d else np.nan,
        "OddsRatio":oratio, "Fisher_p_two_sided":p_two,
        "Fisher_p_macro_greater_than":p_greater,
    }])
    return x, table


def main(force=False, calendar_path=None):
    with stage_logging(STAGE):
        print("STAGE 5 - temporal and counterfactual test of mechanism Jump -> D_H", flush=True)
        s1 = load_checkpoint("stage1_data")
        s3 = load_checkpoint("stage3_rolling")
        df = s1["df"].copy(); daily = s1["daily"].copy().sort_index()
        rolling = s3["rolling_df"].copy().sort_index()
        input_sha = s1.get("input_sha", "without_sha")
        fp = _fingerprint_stage5(input_sha)
        dates = pd.DatetimeIndex(daily.index).normalize()
        tables = {}

        # ---- Daily formal BNS
        cp_bns = CHECKPOINT_DIR / f"stage5_bns_daily_{fp}.pkl"
        payload_bns = None if force else _load_stage5_cp(cp_bns, fp)
        if payload_bns is not None:
            bns = payload_bns["bns"]
            print(f"[resume] Daily BNS loaded: {len(bns):,} days", flush=True)
        else:
            print("[1/4] Computing formal daily BNS (RV/BV/TQ using actual adjacency)...", flush=True)
            bns = compute_bns_daily(df)
            _save_stage5_cp(cp_bns, {"bns":bns}, fp)
        tables["BNS_Daily_Jump_Test"] = bns.reset_index()

        nvalid = int(bns["p_BNS_one_sided"].notna().sum())
        nfdr = int(bns["Jump_BNS_FDR"].sum())
        nstrict = int(bns["Jump_BNS_p0001"].sum())
        print(f"BNS: valid days={nvalid}; jumps FDR(q={BNS_FDR_Q})={nfdr}; p<0.001={nstrict}")

        cal_path = find_macro_calendar(calendar_path); macro_calendar = None
        if cal_path is not None:
            try:
                macro_calendar = load_macro_calendar(cal_path)
                bns_macro, macro_table = macro_jump_analysis(bns, macro_calendar)
                bns = bns_macro
                tables["Macro_Jump_Contingency"] = macro_table
                tables["Macro_Calendar_Used"] = macro_calendar
                print(f"Macro calendar loaded: {cal_path} ({len(macro_calendar)} FOMC/CPI within the study)")
            except Exception as e:
                print(f"[Macro] calendar found but unusable: {e}")
        else:
            print("[Macro] No calendar found; the event study will still run.")
        tables["BNS_Daily_Jump_Test"] = bns.reset_index()

        # ---- Event studies
        print("[2/4] PRE/POST event study excluding the jump day...", flush=True)
        summaries = []; all_treat=[]; all_placebo=[]; all_pairs=[]; selection_diag_rows=[]; balance_rows=[]
        reset_p_pairs=[]; did_p_pairs=[]
        for L in EVENT_WINDOWS:
            cp_ev = CHECKPOINT_DIR / f"stage5_event_{L}d_{fp}.pkl"
            ev_payload = None if force else _load_stage5_cp(cp_ev, fp)
            if ev_payload is not None:
                treat = ev_payload["treat"]; placebo = ev_payload["placebo"]; pairs = ev_payload["pairs"]
                episodes_df = ev_payload.get("episodes", pd.DataFrame())
                selection_diag = ev_payload.get("selection_diag", {
                    "N_jump_days_FDR": int(bns["Jump_BNS_FDR"].sum()),
                    "N_episodes_descriptive": np.nan,
                    "N_jump_candidates_with_margins": np.nan,
                    "N_selected_nonoverlap": len(treat),
                })
                print(f"  [resume] {L}d: treatment={len(treat)}, pairs={len(pairs)}")
            else:
                selected_pos, jump_positions, episodes_df, selection_diag = select_nonoverlap_jump_positions(bns, dates, L)
                print(
                    f"  {L}d: jump-days={selection_diag['N_jump_days_FDR']}; "
                    f"descriptive episodes={selection_diag['N_episodes_descriptive']}; "
                    f"jump candidates with margins={selection_diag['N_jump_candidates_with_margins']}; "
                    f"maximum-cardinality nonoverlap selection={selection_diag['N_selected_nonoverlap']}"
                )
                rows = Parallel(n_jobs=N_JOBS, backend="loky", verbose=JOBLIB_VERBOSE)(
                    delayed(_event_metrics_one)(p, L, dates, df, daily, bns) for p in selected_pos)
                treat = pd.DataFrame([r for r in rows if r is not None])
                if len(treat):
                    treat = treat[(treat["R2_pre"] >= DFA_EVENT_R2_MIN) & (treat["R2_post"] >= DFA_EVENT_R2_MIN)].copy()
                    if "Macro_Event" in bns.columns:
                        treat["Macro_Event"] = [bns.at[d,"Macro_Event"] if d in bns.index else np.nan for d in treat["Date"]]
                        treat["Macro_Day"] = treat["Macro_Event"].notna()
                placebo_pool = _eligible_placebo_positions(dates, daily, bns, rolling, L, jump_positions)
                matches = match_placebos(treat, placebo_pool, L)
                placebo_rows = Parallel(n_jobs=N_JOBS, backend="loky", verbose=JOBLIB_VERBOSE)(
                    delayed(_event_metrics_one)(int(p), L, dates, df, daily, bns) for p in matches.get("Placebo_Pos", pd.Series(dtype=int)).tolist())
                placebo = pd.DataFrame([r for r in placebo_rows if r is not None])
                if len(placebo):
                    placebo = placebo[(placebo["R2_pre"] >= DFA_EVENT_R2_MIN) & (placebo["R2_post"] >= DFA_EVENT_R2_MIN)].copy()
                if len(matches) and len(placebo):
                    pmap = placebo.set_index("Pos")
                    pair_rows=[]
                    tmap = treat.set_index("Pos") if len(treat) else pd.DataFrame()
                    for _,m in matches.iterrows():
                        tp=int(m["Treatment_Pos"]); pp=int(m["Placebo_Pos"])
                        if tp not in tmap.index or pp not in pmap.index: continue
                        tr=tmap.loc[tp]; pl=pmap.loc[pp]
                        pair_rows.append({**m.to_dict(),
                            "Treatment_Delta_D_H":float(tr["Delta_D_H"]),
                            "Placebo_Delta_D_H":float(pl["Delta_D_H"]),
                            "DiD_Delta_D_H":float(tr["Delta_D_H"]-pl["Delta_D_H"]),
                            "Treatment_Z_BNS":float(tr["Z_BNS"]),
                            "Placebo_H_pre_actual":float(pl["H_pre"]),
                            "AbsDiff_H_pre_actual":abs(float(tr["H_pre"])-float(pl["H_pre"])),
                            "Placebo_D_H_pre_actual":float(pl["D_H_pre"]),
                            "AbsDiff_D_H_pre_actual":abs(float(tr["D_H_pre"])-float(pl["D_H_pre"])),
                            "Same_H_side_actual":bool((float(tr["H_pre"])>=0.5)==(float(pl["H_pre"])>=0.5)),
                        })
                    pairs=pd.DataFrame(pair_rows)
                else:
                    pairs=pd.DataFrame()
                _save_stage5_cp(cp_ev, {"treat":treat,"placebo":placebo,"pairs":pairs,"episodes":episodes_df,"selection_diag":selection_diag}, fp)

            try:
                _, _jump_positions_bal = build_jump_episodes(bns, dates)
                _pool_bal = _eligible_placebo_positions(dates, daily, bns, rolling, L, _jump_positions_bal)
                _bal = matching_balance_table(treat, _pool_bal, pairs, L)
                if len(_bal): balance_rows.append(_bal)
            except Exception as _e:
                print(f"  [WARN] Could not construct matching balance {L}d: {_e}")

            if len(episodes_df):
                tables[f"Jump_Episodes_{L}d"] = episodes_df
            selection_diag_rows.append({"Window_days": L, **selection_diag,
                                        "N_treat_valid_R2": len(treat),
                                        "N_pairs_placebo": len(pairs)})

            if len(treat):
                p_reset = signflip_test(treat["Delta_D_H"], alternative="less", seed=STAGE5_SEED+L)
                lo,hi = bootstrap_mean_ci(treat["Delta_D_H"], seed=STAGE5_SEED+L)
                mean_d=float(treat["Delta_D_H"].mean()); med_d=float(treat["Delta_D_H"].median())
                frac_neg=float((treat["Delta_D_H"]<0).mean())
            else:
                p_reset=lo=hi=mean_d=med_d=frac_neg=np.nan
            if len(pairs):
                p_did_naive = signflip_test(pairs["DiD_Delta_D_H"], alternative="less",
                                            seed=STAGE5_SEED+1000+L)
                p_did, n_did_clusters, _, t_did_wcb = wild_cluster_bootstrap_mean_test(
                    pairs, "DiD_Delta_D_H", "Treatment_Year",
                    alternative="less", seed=STAGE5_SEED+2000+L)
                dlo,dhi = cluster_bootstrap_mean_ci(
                    pairs, "DiD_Delta_D_H", "Treatment_Year",
                    seed=STAGE5_SEED+3000+L)
                did_mean=float(pairs["DiD_Delta_D_H"].mean()); did_med=float(pairs["DiD_Delta_D_H"].median())
            else:
                p_did=p_did_naive=dlo=dhi=did_mean=did_med=np.nan
                n_did_clusters=0
                t_did_wcb=np.nan

            reset_p_pairs.append((f"Reset_DH_{L}d",p_reset))
            did_p_pairs.append((f"DiD_Reset_vs_Placebo_{L}d",p_did))
            summaries.append({
                "Window_pre_post_days":L,
                "N_events_inferential":len(treat), "N_pairs_placebo":len(pairs),
                "Mean_Delta_D_H_jump":mean_d, "Median_Delta_D_H_jump":med_d,
                "Frac_Delta_D_H_negative":frac_neg,
                "Bootstrap95_mean_DH_inf":lo, "Bootstrap95_mean_DH_sup":hi,
                "p_signflip_reset_one_sided":p_reset,
                "Mean_DiD_Delta_D_H":did_mean, "Median_DiD_Delta_D_H":did_med,
                "Bootstrap95_DiD_inf":dlo, "Bootstrap95_DiD_sup":dhi,
                "p_wildcluster_DiD_one_sided":p_did,
                "t_wildcluster_DiD":t_did_wcb,
                "p_eventlevel_signflip_DiD_sensitivity":p_did_naive,
                "N_clusters_year_DiD":n_did_clusters,
            })
            if len(treat):
                tmp=treat.copy(); tmp["Window_pre_post_days"]=L; all_treat.append(tmp)
            if len(placebo):
                tmp=placebo.copy(); tmp["Window_pre_post_days"]=L; all_placebo.append(tmp)
            if len(pairs):
                tmp=pairs.copy(); tmp["Window_pre_post_days"]=L; all_pairs.append(tmp)

        summary = pd.DataFrame(summaries)
        tables["EventSelection_Diagnostics"] = pd.DataFrame(selection_diag_rows)
        fdr_reset = fdr_table_from_pairs(reset_p_pairs,"Stage5_Reset_DH_Events").rename(columns={"p_HAC":"p_raw_signflip"})
        fdr_did = fdr_table_from_pairs(did_p_pairs,"Stage5_DiD_vs_Placebo").rename(columns={"p_HAC":"p_raw_signflip"})
        tables["EventStudy_Summary"] = summary
        tables["EventStudy_FDR"] = pd.concat([fdr_reset,fdr_did],ignore_index=True)
        tables["EventStudy_Jumps_Detail"] = pd.concat(all_treat,ignore_index=True) if all_treat else pd.DataFrame()
        tables["EventStudy_Placebos_Detail"] = pd.concat(all_placebo,ignore_index=True) if all_placebo else pd.DataFrame()
        tables["EventStudy_Pairs_DiD"] = pd.concat(all_pairs,ignore_index=True) if all_pairs else pd.DataFrame()
        tables["Placebo_Balance_PrePost"] = pd.concat(balance_rows,ignore_index=True) if balance_rows else pd.DataFrame()
        if len(tables["EventStudy_Pairs_DiD"]):
            bal = []
            for L, g in tables["EventStudy_Pairs_DiD"].groupby("Window_pre_post_days"):
                bal.append({
                    "Window_days": int(L),
                    "N_pairs": len(g),
                    "Mean_AbsDiff_log_PreRV": float(g["AbsDiff_log_PreRV"].mean()),
                    "Mean_AbsDiff_H_pre_actual": float(g["AbsDiff_H_pre_actual"].mean()),
                    "Median_AbsDiff_H_pre_actual": float(g["AbsDiff_H_pre_actual"].median()),
                    "Mean_AbsDiff_D_H_pre_actual": float(g["AbsDiff_D_H_pre_actual"].mean()),
                    "Frac_same_H_side": float(g["Same_H_side_actual"].mean()),
                    "Frac_side_constraint_relaxed": float(g["H_side_constraint_relaxed"].mean()),
                    "Frac_spacing_placebo_relaxed": float(g["Placebo_spacing_relaxed"].mean()),
                    "Frac_same_year": float((g["Match_Time_Tier"]=="same_year").mean()),
                })
            tables["Placebo_Matching_Diagnostics"] = pd.DataFrame(bal)

        print("\nEvent study summary:")
        print(tabulate(summary, headers="keys", tablefmt="simple", showindex=False, floatfmt=".5f"))
        print("\nFDR for the two confirmatory Stage 5 families:")
        print(tabulate(tables["EventStudy_FDR"], headers="keys", tablefmt="simple", showindex=False, floatfmt=".5f"))

        # ---- Secondary macro stratification
        print("[3/4] Macro classification and secondary comparison...", flush=True)
        macro_strata=[]
        treat_all=tables["EventStudy_Jumps_Detail"]
        if len(treat_all) and "Macro_Day" in treat_all.columns:
            for L in EVENT_WINDOWS:
                sub=treat_all[treat_all["Window_pre_post_days"]==L]
                a=sub[sub["Macro_Day"]==True]["Delta_D_H"].dropna()
                b=sub[sub["Macro_Day"]==False]["Delta_D_H"].dropna()
                p=np.nan
                if len(a)>=3 and len(b)>=3:
                    try: p=float(mannwhitneyu(a,b,alternative="two-sided").pvalue)
                    except Exception: pass
                macro_strata.append({"Window_days":L,"N_macro":len(a),"N_no_macro":len(b),
                                     "Mean_Delta_DH_macro":a.mean() if len(a) else np.nan,
                                     "Mean_Delta_DH_no_macro":b.mean() if len(b) else np.nan,
                                     "MannWhitney_p_two_sided":p})
        tables["EventStudy_Macro_vs_NonMacro"] = pd.DataFrame(macro_strata)

        # ---- Metadata and independent output
        print("[4/4] Exporting Stage 5 evidence package...", flush=True)
        tables["Design_Amendment_v1_to_v3"] = pd.DataFrame([
            {"Item":"Trigger", "Description":"v1 produced an unusually low number of nonoverlapping events (12 at 30d; 2 at 60d), so I reviewed the selection algorithm (not driven by signs or p-values)."},
            {"Item":"Default 1", "Description":"v1 transitively chained overlapping windows. v3 uses exact interval scheduling over ALL eligible BNS-FDR days; episodes remain descriptive only."},
            {"Item":"Default 2", "Description":"v1 did not match baseline D_H_pre. v3 uses robust joint log(PreRV)+D_H_pre distance and audits balance before/after."},
            {"Item":"Default 3", "Description":"v2 averaged DiD by year before sign-flip, changing the weights. v3 uses event-level Rademacher wild-cluster bootstrap-t with annual clusters."},
            {"Item":"Invariants", "Description":"BNS q=0.05, 30/60d windows, event-day exclusion, Delta D_H outcome and DiD<0 contrast remain unchanged."},
            {"Item":"Audit", "Description":"v1 results are neither deleted nor replaced; they must be retained as a record of the mechanical amendment before the v3 run."},
        ])
        meta = pd.DataFrame([
            {"Parameter":"Stage5 version","Value":STAGE5_VERSION},
            {"Parameter":"Pipeline base version","Value":PIPELINE_VERSION},
            {"Parameter":"OHLCV SHA256","Value":input_sha},
            {"Parameter":"BNS FDR q","Value":BNS_FDR_Q},
            {"Parameter":"BNS strict raw p (sensitivity)","Value":BNS_STRICT_RAW_P},
            {"Parameter":"Event windows trading days","Value":str(EVENT_WINDOWS)},
            {"Parameter":"Primary event window","Value":PRIMARY_EVENT_WINDOW},
            {"Parameter":"Jump episode gap days","Value":JUMP_EPISODE_GAP_DAYS},
            {"Parameter":"Event selection","Value":"ALL FDR jump-days; exact max-cardinality non-overlap; tie-break max sum Z_BNS"},
            {"Parameter":"Placebo matching","Value":"same-year/+-1y; robust distance log(PreRV)+D_H_pre; same H-side preferred; no replacement"}, 
            {"Parameter":"Placebo matching pre-RV days","Value":MATCH_PREVOL_DAYS},
            {"Parameter":"DFA R2 minimum","Value":DFA_EVENT_R2_MIN},
            {"Parameter":"Randomization reps","Value":RANDOMIZATION_REPS},
            {"Parameter":"Bootstrap reps","Value":BOOTSTRAP_REPS},
            {"Parameter":"Wild-cluster bootstrap-t reps","Value":WILD_CLUSTER_REPS},
            {"Parameter":"Design amendment","Value":"v1 retained; v3 corrects outcome-independent mechanical selection/matching/inference defects before rerun"},
            {"Parameter":"Calendar macro","Value":str(cal_path) if cal_path else "NOT FOUND"},
            {"Parameter":"Macro attribution","Value":"date level only; no intraday timing"},
            {"Parameter":"Lee-Mykland intraday","Value":"not run: requires documented timezone/session"},
            {"Parameter":"Data vendor","Value":DATA_VENDOR},
            {"Parameter":"Source timezone","Value":SOURCE_TIMEZONE},
        ])
        tables["Stage5_Parameters"] = meta

        used=set()
        with pd.ExcelWriter(STAGE5_EXCEL,engine="openpyxl") as writer:
            for name,t in tables.items():
                if t is None or len(t)==0:
                    t=pd.DataFrame({"Note":["No data / block unavailable"]})
                t.to_excel(writer,sheet_name=safe_sheet_name(name,used),index=False)
        print(f"Stage 5 Excel: {STAGE5_EXCEL}")

        final_payload={"bns":bns,"event_summary":summary,"event_fdr":tables["EventStudy_FDR"],
                       "excel_path":str(STAGE5_EXCEL),"fingerprint":fp}
        save_checkpoint(STAGE,final_payload)
        mark_stage(5,"complete",f"BNS FDR jumps={nfdr}; event windows={EVENT_WINDOWS}")
        print("\nPRE-SPECIFIED INTERPRETATION:")
        print("  Reset hypothesis supported ONLY if Delta_D_H<0 and, above all, DiD vs placebo<0")
        print("  with consistent inference/FDR and adequate placebo balance.")
        print("  This v3 retains v1 for audit purposes and previously corrected confirmatory selection over all jump days, D_H_pre matching, and wild-cluster bootstrap-t.")


if __name__ == "__main__":
    import argparse
    p=argparse.ArgumentParser(description="Stage 5: BNS + event study + placebos + macro-day classification")
    p.add_argument("--force",action="store_true",help="recompute BNS/events ignoring Stage 5 checkpoints")
    p.add_argument("--calendar",default=None,help="optional path to economic_calendar_publication_master.csv")
    a=p.parse_args(); main(force=a.force, calendar_path=a.calendar)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage6_dh_calibration.py
# -*- coding: utf-8 -*-
"""Stage 6: null calibration of D_H."""
from __future__ import annotations

from config import *

import argparse
import json
import math
import re
import os
import platform
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

STAGE6_VERSION = "XAUUSD_STAGE6_NULLCAL_v1_2026-09-11"
STAGE6_DIR = CHECKPOINT_DIR / "stage6"
STAGE6_DIR.mkdir(parents=True, exist_ok=True)
STAGE6_OUTPUT_DIR = OUTPUT_DIR / "stage6"
STAGE6_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = (30, 60)
ALPHA = 0.05
DFA_R2_STAGE6 = DFA_R2_MIN

LIBRARY_N_PRIMARY = int(os.getenv("XAUUSD_STAGE6_LIBRARY_N", "1200"))
IAAFT_PER_REAL_PAIR = int(os.getenv("XAUUSD_STAGE6_IAAFT_PER_PAIR", "1"))
IAAFT_LOCAL_MAX_ITER = IAAFT_MAX_ITER
IAAFT_LOCAL_TOL = IAAFT_CONVERGENCE_TOL

OUTER_MC = int(os.getenv("XAUUSD_STAGE6_OUTER_MC", "10000"))
K_NEAREST = int(os.getenv("XAUUSD_STAGE6_K_NEAREST", "25"))
LIB_BATCH = int(os.getenv("XAUUSD_STAGE6_LIB_BATCH", "40"))

ANCOVA_WILD_B = int(os.getenv("XAUUSD_STAGE6_ANCOVA_WILD_B", "99999"))

NULL_SPECS = {
    "N1_Gaussian_IID": {"kind": "gaussian"},
    "N2_Empirical_IID": {"kind": "empirical"},
    "N3_GARCH11": {"kind": "garch"},
    "N4_fGn_H040": {"kind": "fgn", "H": 0.40},
    "N4_fGn_H050": {"kind": "fgn", "H": 0.50},
    "N4_fGn_H055": {"kind": "fgn", "H": 0.55},
    "N4_fGn_H060": {"kind": "fgn", "H": 0.60},
}
COMPOSITE_PRIMARY = (
    "N2_Empirical_IID",
    "N3_GARCH11",
    "N4_fGn_H050",
    "N5_IAAFT_Local",
)

# N6 positive control: moderate latent H reset of 0.05.
POSITIVE_CONTROL_H_PRE = 0.55
POSITIVE_CONTROL_H_POST = 0.50
POSITIVE_CONTROL_MIN_POWER = 0.80

COUPLING_MIN_NULLS_INSIDE = 4

RNG_BASE = RANDOM_SEED + 600_000
RUN_TAG = "UNSET"
DATA_TAG = "UNSET"

def _stable_int(text, mod=100000):
    h=hashlib.sha256(str(text).encode("utf-8")).hexdigest()
    return int(h[:12],16)%mod


def _safe_corr(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def _robust_scale(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) == 0:
        return 1.0
    med = np.median(x)
    mad = np.median(np.abs(x - med)) * 1.4826
    if not np.isfinite(mad) or mad <= 1e-12:
        sd = np.std(x, ddof=1) if len(x) > 1 else 1.0
        return float(sd if np.isfinite(sd) and sd > 1e-12 else 1.0)
    return float(mad)


def _bh_adjust(df, pcol="p_raw", family_col="Family"):
    out = df.copy()
    out["p_FDR"] = np.nan
    out["Significant_FDR"] = False
    for fam, ii in out.groupby(family_col).groups.items():
        p = out.loc[ii, pcol].astype(float).to_numpy()
        good = np.isfinite(p)
        if good.any():
            rej, adj, _, _ = multipletests(p[good], alpha=ALPHA, method="fdr_bh")
            idx_good = np.asarray(list(ii))[good]
            out.loc[idx_good, "p_FDR"] = adj
            out.loc[idx_good, "Significant_FDR"] = rej
    return out


def _stage5_default_path():
    return OUTPUT_DIR / "stage5_mechanism_v3_results.xlsx"


def _read_stage5(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Stage 5 v3 Excel not found: {path}")
    pairs = pd.read_excel(path, sheet_name="EventStudy_Pairs_DiD")
    jumps = pd.read_excel(path, sheet_name="EventStudy_Jumps_Detail")
    placebos = pd.read_excel(path, sheet_name="EventStudy_Placebos_Detail")
    summary = pd.read_excel(path, sheet_name="EventStudy_Summary")
    return pairs, jumps, placebos, summary


def _normalize_dates(s):
    values = pd.Series(s, copy=True)
    if pd.api.types.is_datetime64_any_dtype(values):
        return pd.to_datetime(values, errors="coerce").dt.normalize()
    numeric = pd.to_numeric(values, errors="coerce")
    numeric_values = values.map(lambda v: isinstance(v, (int, float, np.integer, np.floating))
                                and not isinstance(v, (bool, np.bool_)))
    numeric_strings = values.map(lambda v: isinstance(v, str) and
                                 bool(re.fullmatch(r"\d{4,6}(?:\.\d+)?", v.strip())))
    excel_serial = numeric.notna() & numeric.between(10000, 90000) & (numeric_values | numeric_strings)
    ambiguous_numeric = numeric.notna() & numeric_values & ~excel_serial
    if ambiguous_numeric.any():
        raise ValueError("Date column contains unsupported numeric dates; expected ISO/datetime or Excel serial 10000..90000")
    parsed = pd.to_datetime(values.mask(excel_serial), errors="coerce")
    if excel_serial.any():
        parsed.loc[excel_serial] = pd.to_datetime(
            numeric.loc[excel_serial], unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(parsed).dt.normalize()


def _prepare_pairs(pairs, jumps, placebos):
    p = pairs.copy()
    j = jumps.copy(); c = placebos.copy()
    p["Treatment_Date"] = _normalize_dates(p["Treatment_Date"])
    p["Placebo_Date"] = _normalize_dates(p["Placebo_Date"])
    j["Date"] = _normalize_dates(j["Date"]); c["Date"] = _normalize_dates(c["Date"])
    jcols = j[["Date", "Window_pre_post_days", "Nret_pre", "Nret_post"]].rename(
        columns={"Nret_pre":"Treatment_Nret_pre", "Nret_post":"Treatment_Nret_post"})
    ccols = c[["Date", "Window_pre_post_days", "Nret_pre", "Nret_post"]].rename(
        columns={"Date":"Placebo_Date", "Nret_pre":"Placebo_Nret_pre", "Nret_post":"Placebo_Nret_post"})
    p = p.merge(jcols, left_on=["Treatment_Date","Window_pre_post_days"], right_on=["Date","Window_pre_post_days"], how="left")
    p = p.drop(columns=["Date"])
    p = p.merge(ccols, on=["Placebo_Date","Window_pre_post_days"], how="left")
    req = ["Treatment_Nret_pre","Treatment_Nret_post","Placebo_Nret_pre","Placebo_Nret_post"]
    if p[req].isna().any().any():
        raise RuntimeError("Could not join PRE/POST Nret from Stage 5 detail.")
    return p

def _cluster_ols(y, X, clusters):
    y=np.asarray(y,float); X=np.asarray(X,float); clusters=np.asarray(clusters)
    n,k=X.shape
    bread=np.linalg.pinv(X.T@X)
    beta=bread@(X.T@y); e=y-X@beta
    uniq=np.unique(clusters); G=len(uniq); meat=np.zeros((k,k))
    for g in uniq:
        m=clusters==g; s=X[m].T@e[m]; meat += np.outer(s,s)
    corr=(G/(G-1))*((n-1)/(n-k)) if G>1 and n>k else 1.0
    V=corr*bread@meat@bread; se=np.sqrt(np.maximum(np.diag(V),0))
    return beta,se,V,e,corr


def _wild_cluster_bootstrap_intercept(y, x1, x2, clusters, B=ANCOVA_WILD_B, seed=RNG_BASE+91):
    y=np.asarray(y,float); x1=np.asarray(x1,float); x2=np.asarray(x2,float); clusters=np.asarray(clusters)
    X=np.column_stack([np.ones(len(y)),x1,x2])
    beta,se,_,_,corr=_cluster_ols(y,X,clusters); t_obs=beta[0]/se[0] if se[0]>0 else np.nan
    # H0 tau=0: restricted model without an intercept.
    X0=np.column_stack([x1,x2]); b0=np.linalg.lstsq(X0,y,rcond=None)[0]; fit0=X0@b0; u0=y-fit0
    bread=np.linalg.pinv(X.T@X); A=bread@X.T; a=bread[0,:]; h=X@a
    uniq=np.unique(clusters); G=len(uniq); gmap={g:i for i,g in enumerate(uniq)}; ci=np.array([gmap[g] for g in clusters])
    rng=np.random.default_rng(seed); le=0; done=0; batch=5000
    for start in range(0,B,batch):
        b=min(batch,B-start); W=rng.choice(np.array([-1.0,1.0]),size=(G,b))
        Y=fit0[:,None]+u0[:,None]*W[ci,:]; Bet=A@Y; E=Y-X@Bet
        S=np.zeros((G,b))
        for gi in range(G):
            m=ci==gi; S[gi,:]=(h[m,None]*E[m,:]).sum(axis=0)
        var0=corr*np.sum(S*S,axis=0); T=Bet[0,:]/np.sqrt(np.maximum(var0,1e-30))
        le += int(np.sum(T <= t_obs)); done += b
    p=(le+1)/(done+1)
    return {"tau":float(beta[0]),"se_cluster":float(se[0]),"t_cluster":float(t_obs),"p_wild_one_sided":float(p),"N_clusters":int(G)}


def observed_ancova(pairs):
    rows=[]
    for L,g in pairs.groupby("Window_pre_post_days"):
        if int(L) not in HORIZONS: continue
        tpre=g["Treatment_D_H_pre"].to_numpy(float)
        cpre=g["Placebo_D_H_pre_actual"].to_numpy(float)
        tpost=tpre+g["Treatment_Delta_D_H"].to_numpy(float)
        cpost=cpre+g["Placebo_Delta_D_H"].to_numpy(float)
        y=tpost-cpost
        x1=tpre-cpre
        x2=g["Treatment_log_PreRV"].to_numpy(float)-g["Placebo_log_PreRV"].to_numpy(float)
        cl=g["Treatment_Year"].to_numpy(int)
        z=_wild_cluster_bootstrap_intercept(y,x1,x2,cl)
        z.update({"Window_days":int(L),"N_pairs":len(g),"Mean_post_diff":float(np.mean(y)),"Mean_Dpre_diff":float(np.mean(x1)),"Mean_logPreRV_diff":float(np.mean(x2))})
        rows.append(z)
    return pd.DataFrame(rows).sort_values("Window_days")


def observed_geometry(pairs):
    rows=[]
    for L,g in pairs.groupby("Window_pre_post_days"):
        if int(L) not in HORIZONS: continue
        for role in ("Treatment","Placebo"):
            if role=="Treatment":
                pre=g["Treatment_D_H_pre"].to_numpy(float); delta=g["Treatment_Delta_D_H"].to_numpy(float)
            else:
                pre=g["Placebo_D_H_pre_actual"].to_numpy(float); delta=g["Placebo_Delta_D_H"].to_numpy(float)
            post=pre+delta; mean=(pre+post)/2
            rows.append({
                "Window_days":int(L),"Group":role,"N":len(pre),
                "Corr_Dpre_DeltaD":_safe_corr(pre,delta),
                "Oldham_Corr_Mean_DeltaD":_safe_corr(mean,delta),
                "SD_pre":float(np.std(pre,ddof=1)),"SD_post":float(np.std(post,ddof=1)),
                "SDpost_SDpre":float(np.std(post,ddof=1)/np.std(pre,ddof=1)) if np.std(pre,ddof=1)>0 else np.nan,
                "Mean_DeltaD":float(np.mean(delta)),"Median_DeltaD":float(np.median(delta)),
                "Frac_DeltaD_neg":float(np.mean(delta<0)),
            })
    return pd.DataFrame(rows)

def fit_garch11_normal(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)>250_000:
        ix=np.linspace(0,len(x)-1,250_000).astype(int); x=x[ix]
    scale=float(np.std(x,ddof=1)); y=x/scale
    v=float(np.var(y));
    def unpack(theta):
        # omega>0, alpha in (0,.45), beta in (0, .999-alpha)
        omega=np.exp(theta[0])
        alpha=0.45/(1+np.exp(-theta[1]))
        beta=(0.999-alpha)/(1+np.exp(-theta[2]))
        return omega,alpha,beta
    def nll(theta):
        omega,alpha,beta=unpack(theta); h=np.empty(len(y)); h[0]=v
        for t in range(1,len(y)):
            h[t]=omega+alpha*y[t-1]**2+beta*h[t-1]
            if not np.isfinite(h[t]) or h[t]<=1e-12: return 1e100
        return 0.5*float(np.sum(np.log(h)+y*y/h))
    a0=.06; b0=.92; o0=max(v*(1-a0-b0),1e-4)
    def logit(z): return math.log(z/(1-z))
    t0=np.array([math.log(o0), logit(a0/.45), logit(b0/(.999-a0))])
    res=minimize(nll,t0,method="L-BFGS-B",options={"maxiter":250})
    omega,alpha,beta=unpack(res.x)
    return {"omega_scaled":float(omega),"alpha":float(alpha),"beta":float(beta),"scale":scale,
            "converged":bool(res.success),"objective":float(res.fun),"alpha_plus_beta":float(alpha+beta)}


def sim_garch(n, rng, pars):
    omega=pars["omega_scaled"]; alpha=pars["alpha"]; beta=pars["beta"]; scale=pars["scale"]
    h=np.empty(n); z=rng.standard_normal(n); y=np.empty(n)
    h[0]=omega/max(1-alpha-beta,1e-6); y[0]=math.sqrt(h[0])*z[0]
    for t in range(1,n):
        h[t]=omega+alpha*y[t-1]**2+beta*h[t-1]
        y[t]=math.sqrt(max(h[t],1e-12))*z[t]
    return y*scale


def fgn_davies_harte(n, H, rng):
    """fGn with variance ~1 via Davies-Harte / circulant embedding."""
    n=int(n); H=float(H)
    if n<2: return rng.standard_normal(n)
    k=np.arange(0,n,dtype=float)
    gamma=0.5*(np.abs(k-1)**(2*H)-2*(k**(2*H))+np.abs(k+1)**(2*H))
    c=np.concatenate([gamma,[0.0],gamma[1:][::-1]])
    lam=np.real(np.fft.fft(c)); minlam=float(lam.min())
    if minlam < -1e-8:
        raise RuntimeError(f"Negative Davies-Harte eigenvalue H={H}: {minlam}")
    lam=np.maximum(lam,0.0); N=2*n
    W=np.zeros(N,dtype=complex)
    W[0]=math.sqrt(lam[0]/N)*rng.standard_normal()
    W[n]=math.sqrt(lam[n]/N)*rng.standard_normal()
    a=rng.standard_normal(n-1); b=rng.standard_normal(n-1)
    W[1:n]=np.sqrt(lam[1:n]/(2*N))*(a+1j*b)
    W[n+1:]=np.conj(W[n-1:0:-1])
    return np.fft.fft(W).real[:n]


def _simulate_pair(kind, npre, npost, rng, empirical, sigma, garch_pars=None, H=None, Hpost=None):
    npre=int(npre); npost=int(npost)
    if kind=="gaussian":
        x=rng.normal(0,sigma,npre+npost); return x[:npre],x[npre:]
    if kind=="empirical":
        x=rng.choice(empirical,size=npre+npost,replace=True); return x[:npre],x[npre:]
    if kind=="garch":
        x=sim_garch(npre+npost,rng,garch_pars); return x[:npre],x[npre:]
    if kind=="fgn":
        if Hpost is None or abs(Hpost-H)<1e-15:
            x=fgn_davies_harte(npre+npost,H,rng)*sigma; return x[:npre],x[npre:]
        return fgn_davies_harte(npre,H,rng)*sigma, fgn_davies_harte(npost,Hpost,rng)*sigma
    raise ValueError(kind)


def _pair_features(pre, post, L):
    Hpre,r2pre,_,_=dfa(pre); Hpost,r2post,_,_=dfa(post)
    dpre=abs(Hpre-.5) if np.isfinite(Hpre) else np.nan; dpost=abs(Hpost-.5) if np.isfinite(Hpost) else np.nan
    pre_scale=float(np.sqrt(np.sum(np.asarray(pre,float)**2)/max(float(L),1.0))) if len(pre) else np.nan
    return {"H_pre":Hpre,"H_post":Hpost,"R2_pre":r2pre,"R2_post":r2post,"D_pre":dpre,"D_post":dpost,
            "Delta_D":dpost-dpre if np.isfinite(dpre) and np.isfinite(dpost) else np.nan,
            "log_PreRV":math.log(max(pre_scale,1e-15)),"Nret_pre":len(pre),"Nret_post":len(post)}


def _length_pool(pairs, L):
    g=pairs[pairs["Window_pre_post_days"]==L]
    npre=np.concatenate([g["Treatment_Nret_pre"].to_numpy(int),g["Placebo_Nret_pre"].to_numpy(int)])
    npost=np.concatenate([g["Treatment_Nret_post"].to_numpy(int),g["Placebo_Nret_post"].to_numpy(int)])
    return npre,npost


def _library_checkpoint(name,L):
    return STAGE6_DIR / f"lib_{name}_{L}d_{STAGE6_VERSION}_{RUN_TAG}.pkl"


def generate_library(name, spec, L, pairs, empirical, sigma, garch_pars, target_n=LIBRARY_N_PRIMARY, force=False):
    cp=_library_checkpoint(name,L)
    done=pd.read_pickle(cp) if cp.exists() and not force else pd.DataFrame()
    if len(done)>=target_n:
        return done.iloc[:target_n].reset_index(drop=True)
    npre_pool,npost_pool=_length_pool(pairs,L)
    start=len(done); missing=target_n-start
    print(f"  [{name} {L}d] validated library={start}/{target_n}; missing={missing}",flush=True)

    def one(i):
        rng=np.random.default_rng(RNG_BASE + (_stable_int(name,100_000))*100_000 + L*10_000 + i)
        for attempt in range(40):
            npre=int(rng.choice(npre_pool)); npost=int(rng.choice(npost_pool))
            pre,post=_simulate_pair(spec["kind"],npre,npost,rng,empirical,sigma,garch_pars,H=spec.get("H"))
            z=_pair_features(pre,post,L)
            if np.isfinite(z["R2_pre"]) and np.isfinite(z["R2_post"]) and z["R2_pre"]>=DFA_R2_STAGE6 and z["R2_post"]>=DFA_R2_STAGE6:
                z.update({"Library_ID":i+1,"Null":name,"Window_days":L,"Attempt":attempt+1}); return z
        return {"Library_ID":i+1,"Null":name,"Window_days":L,"Attempt":40,
                "H_pre":np.nan,"H_post":np.nan,"R2_pre":np.nan,"R2_post":np.nan,"D_pre":np.nan,"D_post":np.nan,"Delta_D":np.nan,"log_PreRV":np.nan,"Nret_pre":np.nan,"Nret_post":np.nan}

    ids=list(range(start,target_n))
    for k in range(0,len(ids),LIB_BATCH):
        batch_ids=ids[k:k+LIB_BATCH]
        rr=Parallel(n_jobs=min(N_JOBS,len(batch_ids)),backend="loky",verbose=JOBLIB_VERBOSE)(delayed(one)(i) for i in batch_ids)
        done=pd.concat([done,pd.DataFrame(rr)],ignore_index=True).drop_duplicates("Library_ID",keep="last").sort_values("Library_ID")
        pd.to_pickle(done,cp); print(f"    checkpoint {len(done)}/{target_n}",flush=True)
    valid=done.dropna(subset=["D_pre","D_post","log_PreRV"]).reset_index(drop=True)
    if len(valid)<0.90*target_n:
        warnings.warn(f"{name} {L}d: only {len(valid)}/{target_n} pairs valid after R2.")
    return valid

def _daily_positions(df):
    return pd.DatetimeIndex(sorted(pd.unique(df.index.normalize())))


def _returns_event_pair(df, dates, event_date, L):
    d=pd.Timestamp(event_date).normalize(); pos=dates.get_indexer([d])[0]
    if pos<0 or pos-L<0 or pos+L>=len(dates): return None,None
    pre_start=dates[pos-L]; pre_end=dates[pos-1]+pd.Timedelta(days=1)
    post_start=dates[pos+1]; post_end=dates[pos+L]+pd.Timedelta(days=1)
    pre=df.loc[(df.index>=pre_start)&(df.index<pre_end),"LogRet"].dropna().to_numpy(float)
    post=df.loc[(df.index>=post_start)&(df.index<post_end),"LogRet"].dropna().to_numpy(float)
    return pre,post


def generate_iaaft_local_library(L, pairs, df, force=False):
    name="N5_IAAFT_Local"; cp=_library_checkpoint(name,L)
    done=pd.read_pickle(cp) if cp.exists() and not force else pd.DataFrame()
    dates=_daily_positions(df)
    events=[]
    g=pairs[pairs["Window_pre_post_days"]==L]
    for role,col in [("T","Treatment_Date"),("C","Placebo_Date")]:
        for d in g[col].tolist(): events.append((role,pd.Timestamp(d).normalize()))
    targets=[]
    for rep in range(IAAFT_PER_REAL_PAIR):
        for role,d in events: targets.append((rep,role,d))
    done_keys=set(zip(done.get("Rep",[]),done.get("Role",[]),pd.to_datetime(done.get("Date",pd.Series([],dtype="datetime64[ns]"))).dt.normalize())) if len(done) else set()
    missing=[x for x in targets if x not in done_keys]
    print(f"  [N5_IAAFT_Local {L}d] {len(done)}/{len(targets)}; missing={len(missing)}",flush=True)

    def one(rep,role,d):
        seed=RNG_BASE+500_000+L*1000+rep*100_000+(int(pd.Timestamp(d).toordinal())%100_000)+(0 if role=="T" else 50_000)
        rng=np.random.default_rng(seed); pre,post=_returns_event_pair(df,dates,d,L)
        if pre is None or len(pre)<100 or len(post)<100: return None
        x=np.concatenate([pre,post]); xs,err=iaaft_surrogate(x,rng,max_iter=IAAFT_LOCAL_MAX_ITER,tol=IAAFT_LOCAL_TOL)
        spre=xs[:len(pre)]; spost=xs[len(pre):]; z=_pair_features(spre,spost,L)
        z.update({"Rep":rep,"Role":role,"Date":d,"Null":name,"Window_days":L,"IAAFT_spec_error":err})
        return z

    for k in range(0,len(missing),max(1,min(LIB_BATCH,8))):
        batch=missing[k:k+max(1,min(LIB_BATCH,8))]
        rr=Parallel(n_jobs=min(N_JOBS,len(batch)),backend="loky",verbose=JOBLIB_VERBOSE)(delayed(one)(*x) for x in batch)
        rr=[x for x in rr if x is not None]
        done=pd.concat([done,pd.DataFrame(rr)],ignore_index=True)
        if len(done): done=done.drop_duplicates(["Rep","Role","Date"],keep="last")
        pd.to_pickle(done,cp); print(f"    checkpoint {len(done)}/{len(targets)}",flush=True)
    return done.dropna(subset=["D_pre","D_post","log_PreRV"]).reset_index(drop=True)

def _nearest_lists(lib, target_D, target_RV, k=K_NEAREST):
    D=lib["D_pre"].to_numpy(float); R=lib["log_PreRV"].to_numpy(float)
    ds=_robust_scale(np.r_[D,target_D]); rs=_robust_scale(np.r_[R,target_RV])
    out=[]
    kk=min(k,len(lib))
    for d,r in zip(target_D,target_RV):
        dist=np.abs(D-d)/ds + np.abs(R-r)/rs
        out.append(np.argpartition(dist,kk-1)[:kk])
    return out


def _ancova_tau_pairfe(tpre,tpost,trv,cpre,cpost,crv):
    y=np.asarray(tpost)-np.asarray(cpost)
    x1=np.asarray(tpre)-np.asarray(cpre); x2=np.asarray(trv)-np.asarray(crv)
    X=np.column_stack([np.ones(len(y)),x1,x2])
    b=np.linalg.lstsq(X,y,rcond=None)[0]
    return float(b[0])


def _study_stats(t, c):
    tpre=t["D_pre"].to_numpy(float); tpost=t["D_post"].to_numpy(float); td=tpost-tpre
    cpre=c["D_pre"].to_numpy(float); cpost=c["D_post"].to_numpy(float); cd=cpost-cpre
    did=td-cd
    return {
        "Mean_Delta_T":float(np.mean(td)),
        "Mean_Delta_C":float(np.mean(cd)),
        "Mean_DiD":float(np.mean(did)),
        "ANCOVA_tau":_ancova_tau_pairfe(tpre,tpost,t["log_PreRV"].to_numpy(float),cpre,cpost,c["log_PreRV"].to_numpy(float)),
        "Corr_Dpre_Delta_T":_safe_corr(tpre,td),
        "Corr_Dpre_Delta_C":_safe_corr(cpre,cd),
        "Oldham_T":_safe_corr((tpre+tpost)/2,td),
        "Oldham_C":_safe_corr((cpre+cpost)/2,cd),
        "SDratio_T":float(np.std(tpost,ddof=1)/np.std(tpre,ddof=1)) if np.std(tpre,ddof=1)>0 else np.nan,
        "SDratio_C":float(np.std(cpost,ddof=1)/np.std(cpre,ddof=1)) if np.std(cpre,ddof=1)>0 else np.nan,
    }


def outer_mc(lib, pairs_L, B=OUTER_MC, conditioned=True, seed=RNG_BASE+700_000):
    lib=lib.reset_index(drop=True)
    n=len(pairs_L); rng=np.random.default_rng(seed)
    target_t_D=pairs_L["Treatment_D_H_pre"].to_numpy(float)
    target_t_R=pairs_L["Treatment_log_PreRV"].to_numpy(float)
    target_c_D=pairs_L["Placebo_D_H_pre_actual"].to_numpy(float)
    target_c_R=pairs_L["Placebo_log_PreRV"].to_numpy(float)
    if conditioned:
        nearT=_nearest_lists(lib,target_t_D,target_t_R); nearC=_nearest_lists(lib,target_c_D,target_c_R)
    rows=[]
    for b in range(B):
        if conditioned:
            it=np.array([rng.choice(a) for a in nearT]); ic=np.array([rng.choice(a) for a in nearC])
        else:
            it=rng.integers(0,len(lib),size=n); ic=rng.integers(0,len(lib),size=n)
        t=lib.iloc[it]; c=lib.iloc[ic]; z=_study_stats(t,c); z["MC_ID"]=b+1; rows.append(z)
    return pd.DataFrame(rows)


def _obs_stats_from_pairs(g):
    t=pd.DataFrame({"D_pre":g["Treatment_D_H_pre"],
                    "D_post":g["Treatment_D_H_pre"]+g["Treatment_Delta_D_H"],
                    "log_PreRV":g["Treatment_log_PreRV"]})
    c=pd.DataFrame({"D_pre":g["Placebo_D_H_pre_actual"],
                    "D_post":g["Placebo_D_H_pre_actual"]+g["Placebo_Delta_D_H"],
                    "log_PreRV":g["Placebo_log_PreRV"]})
    return _study_stats(t,c)


def empirical_p_lower(sim, obs):
    x=np.asarray(sim,float); x=x[np.isfinite(x)]
    return float((np.sum(x<=obs)+1)/(len(x)+1)) if len(x) else np.nan


def empirical_p_two_sided_center(sim, obs):
    x=np.asarray(sim,float); x=x[np.isfinite(x)]
    if not len(x): return np.nan
    med=np.median(x); dev=abs(obs-med)
    return float((np.sum(np.abs(x-med)>=dev)+1)/(len(x)+1))


def summarize_null(null_name,L,mc_cond,mc_uncond,obs):
    rows=[]
    directional=("Mean_Delta_T","Mean_DiD","ANCOVA_tau")
    for stat in ["Mean_Delta_T","Mean_DiD","ANCOVA_tau","Corr_Dpre_Delta_T","Oldham_T","SDratio_T"]:
        arr=mc_cond[stat].dropna().to_numpy(float); o=float(obs[stat])
        rows.append({"Null":null_name,"Window_days":L,"Calibration":"baseline_conditioned","Statistic":stat,
                     "Observed":o,"Null_mean":float(np.mean(arr)),"Null_median":float(np.median(arr)),
                     "Null_q025":float(np.quantile(arr,.025)),"Null_q05":float(np.quantile(arr,.05)),
                     "Null_q95":float(np.quantile(arr,.95)),"Null_q975":float(np.quantile(arr,.975)),
                     "p_emp":empirical_p_lower(arr,o) if stat in directional else empirical_p_two_sided_center(arr,o),
                     "Observed_inside_95":bool(np.quantile(arr,.025)<=o<=np.quantile(arr,.975))})
    for stat in ["Corr_Dpre_Delta_T","Oldham_T","SDratio_T"]:
        arr=mc_uncond[stat].dropna().to_numpy(float); o=float(obs[stat])
        rows.append({"Null":null_name,"Window_days":L,"Calibration":"unconditional","Statistic":stat,
                     "Observed":o,"Null_mean":float(np.mean(arr)),"Null_median":float(np.median(arr)),
                     "Null_q025":float(np.quantile(arr,.025)),"Null_q05":float(np.quantile(arr,.05)),
                     "Null_q95":float(np.quantile(arr,.95)),"Null_q975":float(np.quantile(arr,.975)),
                     "p_emp":empirical_p_two_sided_center(arr,o),
                     "Observed_inside_95":bool(np.quantile(arr,.025)<=o<=np.quantile(arr,.975))})
    return pd.DataFrame(rows)

def generate_positive_library(L,pairs,empirical,sigma,target_n=LIBRARY_N_PRIMARY,force=False):
    name="N6_Positive_Reset_H055_to_H050"; cp=_library_checkpoint(name,L)
    done=pd.read_pickle(cp) if cp.exists() and not force else pd.DataFrame()
    if len(done)>=target_n: return done.iloc[:target_n].reset_index(drop=True)
    npre_pool,npost_pool=_length_pool(pairs,L)
    def one(i):
        rng=np.random.default_rng(RNG_BASE+900_000+L*10_000+i)
        for attempt in range(40):
            npre=int(rng.choice(npre_pool)); npost=int(rng.choice(npost_pool))
            pre,post=_simulate_pair("fgn",npre,npost,rng,empirical,sigma,H=POSITIVE_CONTROL_H_PRE,Hpost=POSITIVE_CONTROL_H_POST)
            z=_pair_features(pre,post,L)
            if z["R2_pre"]>=DFA_R2_STAGE6 and z["R2_post"]>=DFA_R2_STAGE6:
                z.update({"Library_ID":i+1,"Null":name,"Window_days":L}); return z
        return None
    ids=list(range(len(done),target_n))
    for k in range(0,len(ids),LIB_BATCH):
        batch=ids[k:k+LIB_BATCH]
        rr=Parallel(n_jobs=min(N_JOBS,len(batch)),backend="loky",verbose=JOBLIB_VERBOSE)(delayed(one)(i) for i in batch)
        rr=[r for r in rr if r is not None]
        done=pd.concat([done,pd.DataFrame(rr)],ignore_index=True).drop_duplicates("Library_ID",keep="last")
        pd.to_pickle(done,cp); print(f"  [N6 {L}d] checkpoint {len(done)}/{target_n}",flush=True)
    return done.dropna(subset=["D_pre","D_post","log_PreRV"]).reset_index(drop=True)


def _matched_study_mc(treat_lib, control_lib, pairs_L, B, seed):
    """Sample treatments and match simulated controls by D_pre + logPreRV."""
    rng=np.random.default_rng(seed); n=len(pairs_L)
    treat_lib=treat_lib.reset_index(drop=True); control_lib=control_lib.reset_index(drop=True)
    rows=[]
    for b in range(B):
        it=rng.integers(0,len(treat_lib),size=n); t=treat_lib.iloc[it].reset_index(drop=True)
        near=_nearest_lists(control_lib,t["D_pre"].to_numpy(float),t["log_PreRV"].to_numpy(float))
        ic=np.array([rng.choice(a) for a in near]); c=control_lib.iloc[ic].reset_index(drop=True)
        rows.append(_study_stats(t,c))
    return pd.DataFrame(rows)


def positive_control_power(null_lib, alt_lib, pairs_L, B=OUTER_MC, seed=RNG_BASE+950_000):
    nullmc=_matched_study_mc(null_lib,null_lib,pairs_L,B,seed)
    alt=_matched_study_mc(alt_lib,null_lib,pairs_L,B,seed+1)
    crit_did=float(np.quantile(nullmc["Mean_DiD"],ALPHA)); crit_anc=float(np.quantile(nullmc["ANCOVA_tau"],ALPHA))
    p_did=float(np.mean(alt["Mean_DiD"]<=crit_did)); p_anc=float(np.mean(alt["ANCOVA_tau"]<=crit_anc))
    return pd.DataFrame([{
        "Window_days":int(pairs_L["Window_pre_post_days"].iloc[0]),
        "H_pre_latent":POSITIVE_CONTROL_H_PRE,"H_post_latent_tratamiento":POSITIVE_CONTROL_H_POST,
        "H_post_latent_control":POSITIVE_CONTROL_H_PRE,"N_MC":B,
        "Critical_DiD_null5pct":crit_did,"Power_DiD":p_did,
        "Critical_ANCOVA_null5pct":crit_anc,"Power_ANCOVA":p_anc,
        "Power_target":POSITIVE_CONTROL_MIN_POWER,
        "Adequate_power":bool(max(p_did,p_anc)>=POSITIVE_CONTROL_MIN_POWER),
    }]), alt, nullmc

def decision_tables(summary_nulls, positive_power):
    rows=[]
    for L in HORIZONS:
        s=summary_nulls[(summary_nulls["Window_days"]==L)&(summary_nulls["Calibration"]=="baseline_conditioned")]
        c=s[s["Statistic"]=="Corr_Dpre_Delta_T"]
        inside={r.Null:bool(r.Observed_inside_95) for r in c.itertuples()}
        n_inside=sum(inside.values()); req=inside.get("N2_Empirical_IID",False) and inside.get("N3_GARCH11",False) and n_inside>=COUPLING_MIN_NULLS_INSIDE
        rows.append({"Window_days":L,"Decision":"Baseline_change_coupling",
                     "Criterion":f"inside95 in >= {COUPLING_MIN_NULLS_INSIDE} nulls and includes N2+N3",
                     "Result":f"{n_inside} nulls inside95; N2={inside.get('N2_Empirical_IID')}; N3={inside.get('N3_GARCH11')}",
                     "Cumple":bool(req),"Interpretation":"compatible with geometry/estimation" if req else "not fully explained by tested nulls"})
    comp=[]
    for L in HORIZONS:
        for stat in ("Mean_DiD","ANCOVA_tau"):
            s=summary_nulls[(summary_nulls["Window_days"]==L)&(summary_nulls["Calibration"]=="baseline_conditioned")&(summary_nulls["Statistic"]==stat)]
            ps={r.Null:float(r.p_emp) for r in s.itertuples() if r.Null in COMPOSITE_PRIMARY}
            pcomp=max(ps.values()) if all(k in ps for k in COMPOSITE_PRIMARY) else np.nan
            obs=float(s.iloc[0]["Observed"]) if len(s) else np.nan
            comp.append({"Family":"Residual_vs_composite_null","Window_days":L,"Statistic":stat,"Observed":obs,"p_composite_worstcase":pcomp,"Reset_direction":bool(obs<0)})
    compdf=pd.DataFrame(comp)
    compdf["p_FDR_horizons"] = np.nan; compdf["Reject_composite_after_FDR"] = False
    for stat,ii in compdf.groupby("Statistic").groups.items():
        p=compdf.loc[ii,"p_composite_worstcase"].to_numpy(float); good=np.isfinite(p)
        if good.any():
            rej,adj,_,_=multipletests(p[good],alpha=ALPHA,method="fdr_bh"); ids=np.asarray(list(ii))[good]
            compdf.loc[ids,"p_FDR_horizons"]=adj; compdf.loc[ids,"Reject_composite_after_FDR"]=rej & compdf.loc[ids,"Reset_direction"].to_numpy(bool)
    for L in HORIZONS:
        sub=compdf[compdf["Window_days"]==L]
        both=bool(len(sub)==2 and sub["Reject_composite_after_FDR"].all())
        rows.append({"Window_days":L,"Decision":"Residual_reset_beyond_tested_nulls",
                     "Criterion":"negative DiD AND ANCOVA and worst-case FDR p<.05 under the composite null",
                     "Result":"; ".join(f"{r.Statistic}: pFDR={r.p_FDR_horizons:.4g}, obs={r.Observed:+.5f}" for r in sub.itertuples()),
                     "Cumple":both,"Interpretation":"unexplained dynamic residual" if both else "explanation by nulls/artifacts not rejected"})
    # Positive power.
    for r in positive_power.itertuples():
        rows.append({"Window_days":int(r.Window_days),"Decision":"Positive_control_power",
                     "Criterion":f"max(power DiD, ANCOVA)>={POSITIVE_CONTROL_MIN_POWER:.2f} for latent 0.05 reset",
                     "Result":f"DiD={r.Power_DiD:.3f}; ANCOVA={r.Power_ANCOVA:.3f}",
                     "Cumple":bool(r.Adequate_power),"Interpretation":"design has adequate power for a moderate effect" if r.Adequate_power else "real null result does not strongly rule out a moderate effect"})
    return pd.DataFrame(rows), compdf

def main(stage5_excel: Path, force=False, smoke=False):
    print("="*100)
    print(f"STAGE 6A - null calibration of D_H | {STAGE6_VERSION}")
    print("Pre-registered rules: PREREG_STAGE6.md")
    

    s1=load_checkpoint("stage1_data")
    df=s1["df"].copy(); empirical=df["LogRet"].dropna().to_numpy(float)
    sigma=float(np.std(empirical,ddof=1))
    pairs,jumps,placebos,stage5_summary=_read_stage5(stage5_excel)
    pairs=_prepare_pairs(pairs,jumps,placebos)
    global RUN_TAG, DATA_TAG
    DATA_TAG=str(s1.get("input_sha","NO_SHA"))[:12]
    fpd={"input_sha":s1.get("input_sha"),"stage5_sha":file_sha256(stage5_excel),"version":STAGE6_VERSION,
         "horizons":HORIZONS,"dfa_r2":DFA_R2_STAGE6,"null_specs":NULL_SPECS,
         "iaaft_iter":IAAFT_LOCAL_MAX_ITER,"iaaft_tol":IAAFT_LOCAL_TOL}
    RUN_TAG=hashlib.sha256(json.dumps(fpd,sort_keys=True,default=str).encode("utf-8")).hexdigest()[:12]
    print(f"Stage6 fingerprint={RUN_TAG} | data_tag={DATA_TAG}")

    global LIBRARY_N_PRIMARY, OUTER_MC, IAAFT_PER_REAL_PAIR, ANCOVA_WILD_B
    if smoke:
        LIBRARY_N_PRIMARY=min(LIBRARY_N_PRIMARY,30); OUTER_MC=min(OUTER_MC,100); IAAFT_PER_REAL_PAIR=0; ANCOVA_WILD_B=min(ANCOVA_WILD_B,999)
        print("smoke mode: reduced parameters")

    # Real-data diagnostics before any simulation.
    anc=observed_ancova(pairs); geo=observed_geometry(pairs)
    print("\nReal paired ANCOVA (year-level wild cluster):")
    print(anc.to_string(index=False))

    # GARCH calibrated only once.
    garch_cp=STAGE6_DIR/f"garch_fit_{DATA_TAG}.json"
    if garch_cp.exists() and not force:
        garch_pars=json.loads(garch_cp.read_text(encoding="utf-8"))
    else:
        print("\nFitting normal GARCH(1,1) to intraday returns for N3...",flush=True)
        garch_pars=fit_garch11_normal(empirical); garch_cp.write_text(json.dumps(garch_pars,indent=2),encoding="utf-8")
    print("GARCH:",garch_pars)

    libraries={}; mc_cache={}; null_summaries=[]; mc_export=[]
    for L in HORIZONS:
        pg=pairs[pairs["Window_pre_post_days"]==L].reset_index(drop=True)
        print(f"\n--- Horizon {L}d | real pairs={len(pg)} ---")
        for name,spec in NULL_SPECS.items():
            lib=generate_library(name,spec,L,pairs,empirical,sigma,garch_pars,force=force)
            libraries[(name,L)]=lib
            cond=outer_mc(lib,pg,B=OUTER_MC,conditioned=True,seed=RNG_BASE+L*100+_stable_int(name,10000))
            unc=outer_mc(lib,pg,B=OUTER_MC,conditioned=False,seed=RNG_BASE+10_000+L*100+_stable_int(name,10000))
            obs=_obs_stats_from_pairs(pg)
            null_summaries.append(summarize_null(name,L,cond,unc,obs))
            mc_cache[(name,L)]={"library":lib,"mc_cond":cond,"mc_uncond":unc}
            tmp=cond.copy(); tmp["Null"]=name; tmp["Window_days"]=L; tmp["Calibration"]="baseline_conditioned"; mc_export.append(tmp)

        if not smoke and IAAFT_PER_REAL_PAIR>0:
            name="N5_IAAFT_Local"; lib=generate_iaaft_local_library(L,pairs,df,force=force)
            libraries[(name,L)]=lib
            cond=outer_mc(lib,pg,B=OUTER_MC,conditioned=True,seed=RNG_BASE+50_000+L)
            unc=outer_mc(lib,pg,B=OUTER_MC,conditioned=False,seed=RNG_BASE+60_000+L)
            obs=_obs_stats_from_pairs(pg)
            null_summaries.append(summarize_null(name,L,cond,unc,obs))
            mc_cache[(name,L)]={"library":lib,"mc_cond":cond,"mc_uncond":unc}
            tmp=cond.copy(); tmp["Null"]=name; tmp["Window_days"]=L; tmp["Calibration"]="baseline_conditioned"; mc_export.append(tmp)

    summary_nulls=pd.concat(null_summaries,ignore_index=True)

    # N6 positive control vs H=.55 null.
    powers=[]; alt_mc_rows=[]
    for L in HORIZONS:
        pg=pairs[pairs["Window_pre_post_days"]==L].reset_index(drop=True)
        altlib=generate_positive_library(L,pairs,empirical,sigma,force=force)
        nulllib=mc_cache[("N4_fGn_H055",L)]["library"]
        pwr,alt,nullpower=positive_control_power(nulllib,altlib,pg,B=OUTER_MC,seed=RNG_BASE+80_000+L)
        powers.append(pwr); alt["Window_days"]=L; alt_mc_rows.append(alt)
    power_df=pd.concat(powers,ignore_index=True)
    decisions,composite=decision_tables(summary_nulls,power_df)

    meta=pd.DataFrame([
        {"Parameter":"Stage6_version","Value":STAGE6_VERSION},
        {"Parameter":"Stage6_fingerprint","Value":RUN_TAG},
        {"Parameter":"Input_data_tag","Value":DATA_TAG},
        {"Parameter":"Pipeline_version","Value":PIPELINE_VERSION},
        {"Parameter":"Stage5_excel","Value":str(stage5_excel)},
        {"Parameter":"LIBRARY_N_PRIMARY","Value":LIBRARY_N_PRIMARY},
        {"Parameter":"OUTER_MC","Value":OUTER_MC},
        {"Parameter":"K_NEAREST","Value":K_NEAREST},
        {"Parameter":"IAAFT_PER_REAL_PAIR","Value":IAAFT_PER_REAL_PAIR},
        {"Parameter":"ANCOVA_WILD_B","Value":ANCOVA_WILD_B},
        {"Parameter":"HORIZONS","Value":str(HORIZONS)},
        {"Parameter":"ALPHA","Value":ALPHA},
        {"Parameter":"DFA_R2_MIN","Value":DFA_R2_STAGE6},
        {"Parameter":"Composite_primary_nulls","Value":str(COMPOSITE_PRIMARY)},
        {"Parameter":"Positive_control","Value":f"fGn H {POSITIVE_CONTROL_H_PRE}-> {POSITIVE_CONTROL_H_POST}; control {POSITIVE_CONTROL_H_PRE}->{POSITIVE_CONTROL_H_PRE}"},
        {"Parameter":"Positive_control_min_power","Value":POSITIVE_CONTROL_MIN_POWER},
        {"Parameter":"Floor_effect_note","Value":"D_H=|H-.5| >=0; -1/sqrt(2) is NOT an exact null, only an algebraic benchmark"},
        {"Parameter":"Python","Value":platform.python_version()},
        {"Parameter":"N_JOBS","Value":N_JOBS},
    ])

    out=STAGE6_OUTPUT_DIR/"stage6_dh_calibration_results.xlsx"
    with pd.ExcelWriter(out,engine="openpyxl") as xw:
        anc.to_excel(xw,sheet_name="ANCOVA_Real_v3",index=False)
        geo.to_excel(xw,sheet_name="Observed_Geometry_v3",index=False)
        summary_nulls.to_excel(xw,sheet_name="Null_Calibration_Summary",index=False)
        composite.to_excel(xw,sheet_name="Composite_Null_Tests",index=False)
        power_df.to_excel(xw,sheet_name="Positive_Control_Power",index=False)
        decisions.to_excel(xw,sheet_name="Decision_Rules_Results",index=False)
        pd.DataFrame([garch_pars]).to_excel(xw,sheet_name="GARCH_Fit",index=False)
        meta.to_excel(xw,sheet_name="Stage6_Parameters",index=False)
        # Summarized libraries for inspection.
        libsum=[]
        for (name,L),lib in libraries.items():
            libsum.append({"Null":name,"Window_days":L,"N_valid":len(lib),"Dpre_mean":lib["D_pre"].mean(),"Dpre_sd":lib["D_pre"].std(),"Delta_mean":lib["Delta_D"].mean(),"Oldham_library":_safe_corr((lib["D_pre"]+lib["D_post"])/2,lib["Delta_D"]),"Coupling_library":_safe_corr(lib["D_pre"],lib["Delta_D"]),"R2pre_min":lib["R2_pre"].min(),"R2post_min":lib["R2_post"].min()})
        pd.DataFrame(libsum).to_excel(xw,sheet_name="Library_Diagnostics",index=False)
        if mc_export:
            sample=pd.concat([z.iloc[:min(500,len(z))] for z in mc_export],ignore_index=True)
            sample.to_excel(xw,sheet_name="MC_Sample_Audit",index=False)

    # Main CSVs for reproducibility without Excel limits.
    summary_nulls.to_csv(STAGE6_OUTPUT_DIR/"null_calibration_summary.csv",index=False)
    composite.to_csv(STAGE6_OUTPUT_DIR/"composite_null_tests.csv",index=False)
    power_df.to_csv(STAGE6_OUTPUT_DIR/"positive_control_power.csv",index=False)
    decisions.to_csv(STAGE6_OUTPUT_DIR/"decision_rules_results.csv",index=False)
    anc.to_csv(STAGE6_OUTPUT_DIR/"ancova_real_v3.csv",index=False)
    for (name,L),z in mc_cache.items():
        z["mc_cond"].to_pickle(STAGE6_DIR/f"outermc_cond_{name}_{L}d_{STAGE6_VERSION}_{RUN_TAG}.pkl")
        z["mc_uncond"].to_pickle(STAGE6_DIR/f"outermc_uncond_{name}_{L}d_{STAGE6_VERSION}_{RUN_TAG}.pkl")

    print("\n"+"="*100)
    print("DECISIONS")
    print(decisions.to_string(index=False))
    print(f"\nExcel: {out}")
    print(f"Checkpoints: {STAGE6_DIR}")
    

if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--stage5-excel",type=Path,default=_stage5_default_path())
    ap.add_argument("--force",action="store_true",help="Recompute libraries; use only for a bug or deliberate configuration change.")
    ap.add_argument("--smoke",action="store_true",help="Reduced NON-REPORTABLE technical test.")
    args=ap.parse_args()
    main(args.stage5_excel,force=args.force,smoke=args.smoke)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage6b_rolling_overlap.py
# -*- coding: utf-8 -*-
"""Stage 6B: effect of overlapping rolling windows."""
from __future__ import annotations
from config import *

import json, os, math, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

STAGE6_VERSION="XAUUSD_STAGE6_NULLCAL_v1_2026-09-11"
OUTDIR=OUTPUT_DIR/"stage6"; OUTDIR.mkdir(parents=True,exist_ok=True)
CPDIR=CHECKPOINT_DIR/"stage6"; CPDIR.mkdir(parents=True,exist_ok=True)

ROLL_REPS=int(os.getenv("XAUUSD_STAGE6B_REPS","30"))
ROLL_DAYS=int(os.getenv("XAUUSD_STAGE6B_DAYS","500"))
HORIZONS=(30,60)
MODELS=("N1_Gaussian_IID","N2_Empirical_IID","N3_GARCH11","N4_fGn_H050")
SEED=RANDOM_SEED+1_600_000


def ac1(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<4 or np.std(x[:-1])==0 or np.std(x[1:])==0: return np.nan
    return float(np.corrcoef(x[:-1],x[1:])[0,1])


def max_run(mask):
    best=cur=0
    for z in np.asarray(mask,bool):
        if z: cur+=1; best=max(best,cur)
        else: cur=0
    return int(best)


def fgn_davies_harte(n,H,rng):
    k=np.arange(0,n,dtype=float); gamma=.5*(np.abs(k-1)**(2*H)-2*(k**(2*H))+np.abs(k+1)**(2*H))
    c=np.concatenate([gamma,[0.0],gamma[1:][::-1]]); lam=np.maximum(np.real(np.fft.fft(c)),0); N=2*n
    W=np.zeros(N,complex); W[0]=math.sqrt(lam[0]/N)*rng.standard_normal(); W[n]=math.sqrt(lam[n]/N)*rng.standard_normal()
    a=rng.standard_normal(n-1); b=rng.standard_normal(n-1); W[1:n]=np.sqrt(lam[1:n]/(2*N))*(a+1j*b); W[n+1:]=np.conj(W[n-1:0:-1])
    return np.fft.fft(W).real[:n]


def sim_garch(n,rng,p):
    h=np.empty(n); z=rng.standard_normal(n); y=np.empty(n); a=p['alpha']; b=p['beta']; o=p['omega_scaled']; scale=p['scale']
    h[0]=o/max(1-a-b,1e-6); y[0]=math.sqrt(h[0])*z[0]
    for t in range(1,n): h[t]=o+a*y[t-1]**2+b*h[t-1]; y[t]=math.sqrt(max(h[t],1e-12))*z[t]
    return y*scale


def generate_path(model,counts,empirical,sigma,garch,rng):
    n=int(np.sum(counts))
    if model=="N1_Gaussian_IID": x=rng.normal(0,sigma,n)
    elif model=="N2_Empirical_IID": x=rng.choice(empirical,size=n,replace=True)
    elif model=="N3_GARCH11": x=sim_garch(n,rng,garch)
    elif model=="N4_fGn_H050": x=fgn_davies_harte(n,.5,rng)*sigma
    else: raise ValueError(model)
    offsets=np.concatenate([[0],np.cumsum(counts)])
    return x,offsets


def rolling_H(x,offsets,L):
    nd=len(offsets)-1; ids=list(range(L-1,nd))
    def one(i):
        a=offsets[i-L+1]; b=offsets[i+1]; h,r2,_,_=dfa(x[a:b]); return h,r2
    rr=Parallel(n_jobs=N_JOBS,backend="loky",verbose=JOBLIB_VERBOSE)(delayed(one)(i) for i in ids)
    H=np.array([z[0] for z in rr],float); R=np.array([z[1] for z in rr],float); H[R<DFA_R2_MIN]=np.nan
    return H


def one_rep(model,rep,count_pool,empirical,sigma,garch):
    rng=np.random.default_rng(SEED+rep*100+int(hashlib.sha256(model.encode()).hexdigest()[:6],16)%100)
    counts=rng.choice(count_pool,size=ROLL_DAYS,replace=True).astype(int); x,off=generate_path(model,counts,empirical,sigma,garch,rng)
    out=[]
    for L in HORIZONS:
        H=rolling_H(x,off,L); D=np.abs(H-.5); valid=np.isfinite(D); H=H[valid]; D=D[valid]
        Hn=H[::L]; Dn=D[::L]
        threshold=float(np.nanquantile(Dn,.95)) if len(Dn)>=5 else np.nan
        rho=ac1(D); neff=len(D)*(1-rho)/(1+rho) if np.isfinite(rho) and rho<.999 else np.nan
        out.append({"Model":model,"Rep":rep+1,"Window_days":L,"N_step1":len(D),"N_nonoverlap":len(Dn),
                    "AC1_H_step1":ac1(H),"AC1_DH_step1":rho,"AC1_H_stepL":ac1(Hn),"AC1_DH_stepL":ac1(Dn),
                    "Approx_Neff_AR1_DH":neff,"MeanAbsDeltaDH_step1":float(np.nanmean(np.abs(np.diff(D)))) if len(D)>1 else np.nan,
                    "MeanAbsDeltaDH_stepL":float(np.nanmean(np.abs(np.diff(Dn)))) if len(Dn)>1 else np.nan,
                    "Threshold_DH_p95_nonoverlap":threshold,"MaxRun_step1_above_nonoverlap_p95":max_run(D>threshold) if np.isfinite(threshold) else np.nan})
    return out


def main(force=False):
    s1=load_checkpoint("stage1_data"); df=s1['df']; empirical=df['LogRet'].dropna().to_numpy(float); sigma=float(np.std(empirical,ddof=1))
    counts=df['LogRet'].groupby(df.index.normalize()).count().to_numpy(int); counts=counts[counts>=20]
    data_tag=str(s1.get('input_sha','NO_SHA'))[:12]
    gpath=CPDIR/f"garch_fit_{data_tag}.json"
    if not gpath.exists(): raise FileNotFoundError("Run stage6_dh_calibration.py first to calibrate N3 GARCH.")
    garch=json.loads(gpath.read_text(encoding='utf-8'))
    cp=CPDIR/f"rolling_overlap_{STAGE6_VERSION}_{data_tag}.pkl"; done=pd.read_pickle(cp) if cp.exists() and not force else pd.DataFrame()
    existing=set(zip(done.get('Model',[]),done.get('Rep',[]))) if len(done) else set()
    tasks=[(m,r) for m in MODELS for r in range(ROLL_REPS) if (m,r+1) not in existing]
    print(f"Stage6B: completed={len(existing)}/{len(MODELS)*ROLL_REPS}; missing={len(tasks)}")
    for m,r in tasks:
        rr=one_rep(m,r,counts,empirical,sigma,garch); done=pd.concat([done,pd.DataFrame(rr)],ignore_index=True); pd.to_pickle(done,cp)
        print(f"  {m} rep {r+1}/{ROLL_REPS} saved",flush=True)
    summary=done.groupby(['Model','Window_days']).agg(
        Reps=('Rep','count'),AC1_DH_step1_mean=('AC1_DH_step1','mean'),AC1_DH_step1_q025=('AC1_DH_step1',lambda x:x.quantile(.025)),AC1_DH_step1_q975=('AC1_DH_step1',lambda x:x.quantile(.975)),
        AC1_DH_nonoverlap_mean=('AC1_DH_stepL','mean'),Approx_Neff_mean=('Approx_Neff_AR1_DH','mean'),MaxRun_mean=('MaxRun_step1_above_nonoverlap_p95','mean'),MaxRun_p95=('MaxRun_step1_above_nonoverlap_p95',lambda x:x.quantile(.95)),
        MeanAbsDelta_step1=('MeanAbsDeltaDH_step1','mean'),MeanAbsDelta_nonoverlap=('MeanAbsDeltaDH_stepL','mean')).reset_index()
    out=OUTDIR/'stage6b_rolling_overlap_results.xlsx'
    with pd.ExcelWriter(out,engine='openpyxl') as xw:
        summary.to_excel(xw,sheet_name='Summary',index=False); done.to_excel(xw,sheet_name='Replications',index=False)
        pd.DataFrame([{'Parameter':'ROLL_REPS','Value':ROLL_REPS},{'Parameter':'ROLL_DAYS','Value':ROLL_DAYS},{'Parameter':'HORIZONS','Value':str(HORIZONS)},{'Parameter':'MODELS','Value':str(MODELS)},{'Parameter':'DFA_R2_MIN','Value':DFA_R2_MIN}]).to_excel(xw,sheet_name='Parameters',index=False)
    summary.to_csv(OUTDIR/'rolling_overlap_summary.csv',index=False)
    print(summary.to_string(index=False)); print(f"{out}")

if __name__=='__main__':
    import argparse
    ap=argparse.ArgumentParser(); ap.add_argument('--force',action='store_true'); a=ap.parse_args(); main(a.force)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage7_structural_discrimination.py
# -*- coding: utf-8 -*-
"""Stage 7: seasonal GARCH models for the 30d anomaly."""
from __future__ import annotations

from config import *

import argparse
import hashlib
import json
import math
import re
import os
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.signal import lfilter
from statsmodels.stats.multitest import multipletests

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        def deco(fn):
            return fn
        return deco

STAGE7_VERSION = "XAUUSD_STAGE7_v2"
STAGE7_DIR = CHECKPOINT_DIR / "stage7"
STAGE7_DIR.mkdir(parents=True, exist_ok=True)
STAGE7_OUTPUT_DIR = OUTPUT_DIR / "stage7"
STAGE7_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_HORIZON = 30
SECONDARY_HORIZON = 60
HORIZONS = (30, 60)
FREQUENCIES = (15, 30)
MATCH_PREVOL_DAYS = 20
DFA_R2_STAGE7 = DFA_R2_MIN
ALPHA = 0.05

LIBRARY_N = int(os.getenv("XAUUSD_STAGE7_LIBRARY_N", "800"))
OUTER_MC = int(os.getenv("XAUUSD_STAGE7_OUTER_MC", "10000"))
BLOCK_MC = int(os.getenv("XAUUSD_STAGE7_BLOCK_MC", "3000"))
K_NEAREST = int(os.getenv("XAUUSD_STAGE7_K_NEAREST", "25"))
LIB_BATCH = int(os.getenv("XAUUSD_STAGE7_LIB_BATCH", "24"))
FIGARCH_TRUNC = int(os.getenv("XAUUSD_STAGE7_FIGARCH_TRUNC", "250"))
FIT_MAX_N = int(os.getenv("XAUUSD_STAGE7_FIT_MAX_N", "0"))  # 0=complete temporally ordered series
SEASON_MIN_COUNT = int(os.getenv("XAUUSD_STAGE7_SEASON_MIN_COUNT", "100"))

# Joint endpoints fixed BEFORE the final run.
G_COMPONENTS = ("Mean_DeltaD", "Corr_Dpre_DeltaD", "Oldham", "Log_SDratio")

MODEL_SPECS = {
    "M0_GARCH_N_raw": {
        "kind": "garch_n", "seasonal": False, "role": "primary",
        "mechanism": "normal GARCH benchmark, complete temporal order"
    },
    "M1_Seasonal_GARCH_N": {
        "kind": "garch_n", "seasonal": True, "role": "primary",
        "mechanism": "intraday periodicity + GARCH clustering"
    },
    "M2_Seasonal_GARCH_t": {
        "kind": "garch_t", "seasonal": True, "role": "primary",
        "mechanism": "M1 + Student-t tails"
    },
    "M3_Seasonal_EGARCH_t": {
        "kind": "egarch_t", "seasonal": True, "role": "primary",
        "mechanism": "M2 + asymmetric/leverage log-volatility response"
    },
    "M4_Seasonal_FIGARCH_t": {
        "kind": "figarch_t", "seasonal": True, "role": "primary",
        "mechanism": "M2 + fractional long memory in volatility"
    },
    "M5_Seasonal_GJR_t": {
        "kind": "gjr_t", "seasonal": True, "role": "sensitivity",
        "mechanism": "sensitivity: alternative threshold asymmetry to EGARCH"
    },
}
PRIMARY_MODELS = tuple(k for k,v in MODEL_SPECS.items() if v["role"]=="primary")

RNG_BASE = RANDOM_SEED + 700_000
RUN_TAG = "UNSET"

def _stable_int(text, mod=1_000_000):
    h = hashlib.sha256(str(text).encode("utf-8")).hexdigest()
    return int(h[:14], 16) % mod


def _safe_corr(x, y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y); x=x[m]; y=y[m]
    if len(x)<3 or np.std(x)==0 or np.std(y)==0: return np.nan
    return float(np.corrcoef(x,y)[0,1])


def _robust_scale(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)==0: return 1.0
    med=np.median(x); mad=1.4826*np.median(np.abs(x-med))
    if not np.isfinite(mad) or mad<=1e-12:
        sd=np.std(x,ddof=1) if len(x)>1 else 1.0
        return float(sd if np.isfinite(sd) and sd>1e-12 else 1.0)
    return float(mad)


def _logit(p):
    p=float(np.clip(p,1e-8,1-1e-8)); return math.log(p/(1-p))


def _sigmoid(x):
    if x>=0:
        z=math.exp(-x); return 1/(1+z)
    z=math.exp(x); return z/(1+z)


def _softmax3(a,b):
    z=np.array([a,b,0.0],float); z-=z.max(); e=np.exp(z); return e/e.sum()


def _two_sided_empirical(sim, obs):
    x=np.asarray(sim,float); x=x[np.isfinite(x)]
    if len(x)==0: return np.nan
    center=np.median(x); d=abs(obs-center)
    return float((np.sum(np.abs(x-center)>=d)+1)/(len(x)+1))


def _upper_empirical(sim, obs):
    x=np.asarray(sim,float); x=x[np.isfinite(x)]
    return float((np.sum(x>=obs)+1)/(len(x)+1)) if len(x) else np.nan


def _normalize_dates(s):
    values = pd.Series(s, copy=True)
    if pd.api.types.is_datetime64_any_dtype(values):
        return pd.to_datetime(values, errors="coerce").dt.normalize()
    numeric = pd.to_numeric(values, errors="coerce")
    numeric_values = values.map(lambda v: isinstance(v, (int, float, np.integer, np.floating))
                                and not isinstance(v, (bool, np.bool_)))
    numeric_strings = values.map(lambda v: isinstance(v, str) and
                                 bool(re.fullmatch(r"\d{4,6}(?:\.\d+)?", v.strip())))
    excel_serial = numeric.notna() & numeric.between(10000, 90000) & (numeric_values | numeric_strings)
    ambiguous_numeric = numeric.notna() & numeric_values & ~excel_serial
    if ambiguous_numeric.any():
        raise ValueError("Date column contains unsupported numeric dates; expected ISO/datetime or Excel serial 10000..90000")
    parsed = pd.to_datetime(values.mask(excel_serial), errors="coerce")
    if excel_serial.any():
        parsed.loc[excel_serial] = pd.to_datetime(
            numeric.loc[excel_serial], unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(parsed).dt.normalize()


def _file_sig(path: Path):
    if not path.exists(): return "MISSING"
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()


def _checkpoint_path(kind, *parts):
    name="__".join([kind]+[str(x) for x in parts]+[RUN_TAG])
    return STAGE7_DIR/f"{name}.pkl"


def _json_checkpoint(kind,*parts):
    name="__".join([kind]+[str(x) for x in parts]+[RUN_TAG])
    return STAGE7_DIR/f"{name}.json"

def _make_frequency_data(stage1, minutes):
    df15=stage1["df"].copy()
    if minutes==15:
        d=df15.copy()
        rv=by_day(d.dropna(subset=["LogRet"])["LogRet"]).apply(lambda x:np.sqrt(np.sum(np.asarray(x,float)**2)))
        daily=pd.DataFrame({"RV":rv})
        return d,daily
    rule=f"{minutes}min"
    ohlc=resample_ohlc(df15,rule)
    d,daily=realized_daily_from_frequency(ohlc,expected_minutes=minutes)
    return d,daily[["RV"]].copy()


def _slot_minutes(index):
    idx=pd.DatetimeIndex(index)
    return (idx.hour*60+idx.minute).to_numpy(int)


def estimate_seasonality(df, min_count=SEASON_MIN_COUNT):
    r=pd.to_numeric(df["LogRet"],errors="coerce")
    m=r.notna(); rr=r[m].to_numpy(float); slots=_slot_minutes(df.index[m])
    global_med=float(np.median(np.abs(rr)))
    rows=[]; factors={}
    for s in sorted(np.unique(slots)):
        x=np.abs(rr[slots==s]); n=len(x)
        med=float(np.median(x)) if n else np.nan
        f=med/global_med if global_med>0 and n>=min_count else 1.0
        f=float(np.clip(f,0.20,5.0))
        factors[int(s)]=f; rows.append({"Slot_minute":int(s),"N":n,"MedianAbsRet":med,"RawFactor":f})
    weights={int(s):int(np.sum(slots==s)) for s in np.unique(slots)}
    rms=math.sqrt(sum(weights[s]*factors[s]**2 for s in factors)/sum(weights.values()))
    for s in factors: factors[s]/=rms
    out=pd.DataFrame(rows); out["Factor"]=[factors[int(s)] for s in out["Slot_minute"]]
    out["HHMM"]=out["Slot_minute"].map(lambda x:f"{x//60:02d}:{x%60:02d}")
    return factors,out


def deseasonalized_returns(df, factors, use_seasonality=True):
    r=pd.to_numeric(df["LogRet"],errors="coerce")
    m=r.notna(); x=r[m].to_numpy(float)
    if use_seasonality:
        slots=_slot_minutes(df.index[m]); sf=np.array([factors.get(int(s),1.0) for s in slots])
        x=x/sf
    x=x[np.isfinite(x)]; x=x-np.mean(x)
    return x

@njit(cache=True)
def _garch_h(y, omega, alpha, beta):
    n=len(y); h=np.empty(n); v=np.mean(y*y); h[0]=max(v,1e-8)
    for t in range(1,n):
        h[t]=omega+alpha*y[t-1]*y[t-1]+beta*h[t-1]
        if h[t]<1e-10 or not np.isfinite(h[t]): h[t]=1e-10
    return h

@njit(cache=True)
def _gjr_h(y, omega, alpha, gamma, beta):
    n=len(y); h=np.empty(n); v=np.mean(y*y); h[0]=max(v,1e-8)
    for t in range(1,n):
        e=y[t-1]; a=alpha+(gamma if e<0 else 0.0)
        h[t]=omega+a*e*e+beta*h[t-1]
        if h[t]<1e-10 or not np.isfinite(h[t]): h[t]=1e-10
    return h

@njit(cache=True)
def _egarch_h(y, omega, alpha, gamma, beta, eabs):
    n=len(y); h=np.empty(n); v=np.mean(y*y); h[0]=max(v,1e-8)
    lh=math.log(h[0])
    for t in range(1,n):
        z=y[t-1]/math.sqrt(max(h[t-1],1e-12))
        lh=omega+beta*lh+alpha*(abs(z)-eabs)+gamma*z
        if lh>20: lh=20
        if lh<-20: lh=-20
        h[t]=math.exp(lh)
    return h

@njit(cache=True)
def _sim_garch_core(n,burn,omega,alpha,beta,z):
    N=n+burn; h=np.empty(N); y=np.empty(N)
    h[0]=max(omega/max(1-alpha-beta,1e-4),1e-8); y[0]=math.sqrt(h[0])*z[0]
    for t in range(1,N):
        h[t]=omega+alpha*y[t-1]*y[t-1]+beta*h[t-1]
        if h[t]<1e-10: h[t]=1e-10
        y[t]=math.sqrt(h[t])*z[t]
    return y[burn:]

@njit(cache=True)
def _sim_gjr_core(n,burn,omega,alpha,gamma,beta,z):
    N=n+burn; h=np.empty(N); y=np.empty(N)
    pers=alpha+0.5*gamma+beta
    h[0]=max(omega/max(1-pers,1e-4),1e-8); y[0]=math.sqrt(h[0])*z[0]
    for t in range(1,N):
        e=y[t-1]; a=alpha+(gamma if e<0 else 0.0)
        h[t]=omega+a*e*e+beta*h[t-1]
        if h[t]<1e-10: h[t]=1e-10
        y[t]=math.sqrt(h[t])*z[t]
    return y[burn:]

@njit(cache=True)
def _sim_egarch_core(n,burn,omega,alpha,gamma,beta,eabs,z):
    N=n+burn; h=np.empty(N); y=np.empty(N); lh=omega/max(1-beta,1e-4)
    h[0]=math.exp(min(max(lh,-20),20)); y[0]=math.sqrt(h[0])*z[0]
    for t in range(1,N):
        zp=y[t-1]/math.sqrt(max(h[t-1],1e-12))
        lh=omega+beta*math.log(max(h[t-1],1e-12))+alpha*(abs(zp)-eabs)+gamma*zp
        if lh>20: lh=20
        if lh<-20: lh=-20
        h[t]=math.exp(lh); y[t]=math.sqrt(h[t])*z[t]
    return y[burn:]

@njit(cache=True)
def _sim_figarch_core(n,burn,omega_tilde,lam,z):
    m=len(lam); N=n+burn+m; y=np.zeros(N); h=np.empty(N)
    v=max(omega_tilde/max(1.0-np.sum(lam),1e-4),1e-8)
    for t in range(m):
        h[t]=v; y[t]=math.sqrt(v)*z[t]
    for t in range(m,N):
        s=0.0
        for j in range(m): s += lam[j]*y[t-1-j]*y[t-1-j]
        h[t]=omega_tilde+s
        if h[t]<1e-10: h[t]=1e-10
        y[t]=math.sqrt(h[t])*z[t]
    start=m+burn
    return y[start:start+n]


def _std_t_eabs(nu):
    # E|Z| for standardized t with Var=1.
    logv=math.log(2.0)+0.5*math.log(nu-2.0)+gammaln((nu+1)/2)-math.log(nu-1)-0.5*math.log(math.pi)-gammaln(nu/2)
    return float(math.exp(logv))


def _nll_normal(y,h):
    if not np.all(np.isfinite(h)) or np.any(h<=0): return 1e100
    return 0.5*float(np.sum(np.log(2*math.pi)+np.log(h)+y*y/h))


def _nll_std_t(y,h,nu):
    if nu<=2.02 or not np.all(np.isfinite(h)) or np.any(h<=0): return 1e100
    z=y/np.sqrt(h)
    c=gammaln((nu+1)/2)-gammaln(nu/2)-0.5*math.log(math.pi*(nu-2))
    ll=c-0.5*np.log(h)-((nu+1)/2)*np.log1p(z*z/(nu-2))
    val=-float(np.sum(ll))
    return val if np.isfinite(val) else 1e100


def _figarch_weights(phi,d,beta,trunc=FIGARCH_TRUNC):
    delta=np.empty(trunc,float); lam=np.empty(trunc,float)
    delta[0]=d; lam[0]=d-beta+phi
    for j in range(2,trunc+1):
        delta[j-1]=((j-1-d)/j)*delta[j-2]
        lam[j-1]=beta*lam[j-2]+delta[j-1]-phi*delta[j-2]
    return lam


def _fit_prepare(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    x=x-np.mean(x); scale=float(np.std(x,ddof=1)); y=x/scale
    if FIT_MAX_N>0 and len(y)>FIT_MAX_N:
        y=y[-FIT_MAX_N:]
    return y,scale


def fit_model(x, kind):
    y,scale=_fit_prepare(x); n=len(y)
    if n<2000: raise RuntimeError(f"Too few returns for {kind}: {n}")
    var=float(np.var(y))

    if kind in ("garch_n","garch_t"):
        def unpack(th):
            omega=math.exp(th[0]); rho=.999*_sigmoid(th[1]); share=_sigmoid(th[2]); alpha=rho*share; beta=rho*(1-share)
            nu=2.05+97.95*_sigmoid(th[3]) if kind=="garch_t" else np.nan
            return omega,alpha,beta,nu
        def obj(th):
            omega,a,b,nu=unpack(th); h=_garch_h(y,omega,a,b)
            return _nll_std_t(y,h,nu) if kind=="garch_t" else _nll_normal(y,h)
        t0=[math.log(max(.02*var,1e-4)),_logit(.97/.999),_logit(.08),_logit((8-2.05)/97.95)] if kind=="garch_t" else [math.log(max(.02*var,1e-4)),_logit(.97/.999),_logit(.08)]
        res=minimize(obj,np.array(t0,float),method="L-BFGS-B",options={"maxiter":350,"ftol":1e-9})
        omega,a,b,nu=unpack(res.x)
        pars={"omega":omega,"alpha":a,"beta":b,"nu":nu,"persistence":a+b}
        k=4 if kind=="garch_t" else 3

    elif kind=="gjr_t":
        def unpack(th):
            omega=math.exp(th[0]); rho=.999*_sigmoid(th[1]); shares=_softmax3(th[2],th[3]); a=rho*shares[0]; gamma=2*rho*shares[1]; b=rho*shares[2]; nu=2.05+97.95*_sigmoid(th[4]); return omega,a,gamma,b,nu
        def obj(th):
            omega,a,g,b,nu=unpack(th); h=_gjr_h(y,omega,a,g,b); return _nll_std_t(y,h,nu)
        t0=np.array([math.log(.02),_logit(.97/.999),-1.5,-2.2,_logit((8-2.05)/97.95)])
        res=minimize(obj,t0,method="L-BFGS-B",options={"maxiter":450,"ftol":1e-9})
        omega,a,g,b,nu=unpack(res.x); pars={"omega":omega,"alpha":a,"gamma":g,"beta":b,"nu":nu,"persistence":a+0.5*g+b}; k=5

    elif kind=="egarch_t":
        def unpack(th):
            omega=th[0]; alpha=math.exp(np.clip(th[1],-8,2)); gamma=1.5*math.tanh(th[2]); beta=.999*_sigmoid(th[3]); nu=2.05+97.95*_sigmoid(th[4]); return omega,alpha,gamma,beta,nu
        def obj(th):
            omega,a,g,b,nu=unpack(th); h=_egarch_h(y,omega,a,g,b,_std_t_eabs(nu)); return _nll_std_t(y,h,nu)
        t0=np.array([(1-.97)*math.log(var),math.log(.08),-0.05,_logit(.97/.999),_logit((8-2.05)/97.95)])
        res=minimize(obj,t0,method="L-BFGS-B",options={"maxiter":450,"ftol":1e-9})
        omega,a,g,b,nu=unpack(res.x); pars={"omega":omega,"alpha":a,"gamma":g,"beta":b,"nu":nu,"persistence":b}; k=5

    elif kind=="figarch_t":
        def unpack(th):
            d=.98*_sigmoid(th[0]); phi=((1-d)/2)*_sigmoid(th[1]); beta=(d+phi)*_sigmoid(th[2]); nu=2.05+97.95*_sigmoid(th[3]); return phi,d,beta,nu
        def obj(th):
            phi,d,b,nu=unpack(th); lam=_figarch_weights(phi,d,b)
            if np.any(~np.isfinite(lam)) or np.any(lam< -1e-8) or np.sum(lam)>=.9995: return 1e100
            omega=(1-b)*max(1-np.sum(lam),1e-6)
            pad=np.full(FIGARCH_TRUNC,var); xshift=np.concatenate([pad,y*y])
            conv=lfilter(lam,[1.0],xshift)
            h=omega/max(1-b,1e-6)+conv[FIGARCH_TRUNC-1:FIGARCH_TRUNC-1+n]
            return _nll_std_t(y,h,nu)
        t0=np.array([_logit(.35/.98),_logit(.5),_logit(.5),_logit((8-2.05)/97.95)])
        res=minimize(obj,t0,method="L-BFGS-B",options={"maxiter":350,"ftol":1e-8})
        phi,d,b,nu=unpack(res.x); lam=_figarch_weights(phi,d,b); omega=(1-b)*max(1-np.sum(lam),1e-6)
        pars={"omega":omega,"phi":phi,"d":d,"beta":b,"nu":nu,"lambda_sum":float(np.sum(lam)),"truncation":FIGARCH_TRUNC}; k=4
    else:
        raise ValueError(kind)

    nll=float(res.fun); return {"kind":kind,"scale":scale,"n_fit":int(n),"converged":bool(res.success),"message":str(res.message),"nll":nll,"AIC":2*k+2*nll,"BIC":k*math.log(n)+2*nll,"params":pars}


def _draw_z(rng,n,dist,nu=None):
    if dist=="normal": return rng.standard_normal(n)
    z=rng.standard_t(float(nu),size=n)*math.sqrt((float(nu)-2)/float(nu)); return z


def simulate_model(fit,n,rng):
    kind=fit["kind"]; p=fit["params"]; burn=800
    dist="normal" if kind=="garch_n" else "t"
    nu=p.get("nu",np.nan); z=_draw_z(rng,n+burn+FIGARCH_TRUNC+10,dist,nu)
    if kind in ("garch_n","garch_t"):
        y=_sim_garch_core(n,burn,p["omega"],p["alpha"],p["beta"],z[:n+burn])
    elif kind=="gjr_t":
        y=_sim_gjr_core(n,burn,p["omega"],p["alpha"],p["gamma"],p["beta"],z[:n+burn])
    elif kind=="egarch_t":
        y=_sim_egarch_core(n,burn,p["omega"],p["alpha"],p["gamma"],p["beta"],_std_t_eabs(p["nu"]),z[:n+burn])
    elif kind=="figarch_t":
        lam=_figarch_weights(p["phi"],p["d"],p["beta"]); omega_t=p["omega"]/max(1-p["beta"],1e-6)
        need=n+burn+len(lam); zz=z[:need]
        y=_sim_figarch_core(n,burn,omega_t,lam,zz)
    else: raise ValueError(kind)
    return np.asarray(y,float)*float(fit["scale"])

def _returns_for_dates(df, dates):
    if len(dates)==0: return np.array([],float), pd.DatetimeIndex([])
    date_set=set(pd.DatetimeIndex(dates).normalize())
    idx=pd.DatetimeIndex(df.index); m=np.array([d in date_set for d in idx.normalize()]) & pd.to_numeric(df["LogRet"],errors="coerce").notna().to_numpy()
    return pd.to_numeric(df.loc[m,"LogRet"],errors="coerce").to_numpy(float), pd.DatetimeIndex(df.index[m])


def event_feature(df,daily,event_date,L):
    valid=df.dropna(subset=["LogRet"])
    dates=pd.DatetimeIndex(sorted(pd.unique(valid.index.normalize())))
    d=pd.Timestamp(event_date).normalize()
    loc=np.where(dates==d)[0]
    if len(loc)!=1: return None
    p=int(loc[0])
    if p<L or p+L>=len(dates): return None
    pre_dates=dates[p-L:p]; post_dates=dates[p+1:p+1+L]; event_dates=dates[p:p+1]
    rpre,tpre=_returns_for_dates(df,pre_dates); rpost,tpost=_returns_for_dates(df,post_dates); rev,tev=_returns_for_dates(df,event_dates)
    Hpre,R2pre,_,_=dfa(rpre); Hpost,R2post,_,_=dfa(rpost)
    if not all(np.isfinite([Hpre,R2pre,Hpost,R2post])): return None
    prv=daily["RV"].reindex(pre_dates).dropna().tail(MATCH_PREVOL_DAYS)
    pre_rv=float(prv.mean()) if len(prv) else np.nan
    return {
        "Date":d,"H_pre":Hpre,"R2_pre":R2pre,"D_pre":abs(Hpre-.5),"Npre":len(rpre),
        "H_post":Hpost,"R2_post":R2post,"D_post":abs(Hpost-.5),"Npost":len(rpost),
        "Delta_D":abs(Hpost-.5)-abs(Hpre-.5),"log_PreRV":math.log(max(pre_rv,1e-15)),
        "pre_slots":_slot_minutes(tpre),"event_slots":_slot_minutes(tev),"post_slots":_slot_minutes(tpost),
        "pre_returns":rpre,"post_returns":rpost,
    }


def load_frozen_pairs(stage5_path):
    p=pd.read_excel(stage5_path,sheet_name="EventStudy_Pairs_DiD")
    p["Treatment_Date"]=_normalize_dates(p["Treatment_Date"]); p["Placebo_Date"]=_normalize_dates(p["Placebo_Date"])
    return p[p["Window_pre_post_days"].isin(HORIZONS)].copy()


def observed_pairs_frequency(pairs,df,daily,minutes):
    rows=[]
    cache={}
    for _,r in pairs.iterrows():
        L=int(r["Window_pre_post_days"]); keyT=(pd.Timestamp(r["Treatment_Date"]),L); keyC=(pd.Timestamp(r["Placebo_Date"]),L)
        if keyT not in cache: cache[keyT]=event_feature(df,daily,*keyT)
        if keyC not in cache: cache[keyC]=event_feature(df,daily,*keyC)
        t=cache[keyT]; c=cache[keyC]
        if t is None or c is None: continue
        if min(t["R2_pre"],t["R2_post"],c["R2_pre"],c["R2_post"])<DFA_R2_STAGE7: continue
        rows.append({
            "Frequency_min":minutes,"Window_days":L,"Treatment_Date":keyT[0],"Placebo_Date":keyC[0],
            "T_D_pre":t["D_pre"],"T_D_post":t["D_post"],"T_Delta":t["Delta_D"],"T_log_PreRV":t["log_PreRV"],
            "C_D_pre":c["D_pre"],"C_D_post":c["D_post"],"C_Delta":c["Delta_D"],"C_log_PreRV":c["log_PreRV"],
            "T_R2_pre":t["R2_pre"],"T_R2_post":t["R2_post"],"C_R2_pre":c["R2_pre"],"C_R2_post":c["R2_post"],
        })
    return pd.DataFrame(rows),cache


def _geometry_from_arrays(pre,post):
    pre=np.asarray(pre,float); post=np.asarray(post,float); d=post-pre; mean=(pre+post)/2
    sdpre=float(np.std(pre,ddof=1)); sdpost=float(np.std(post,ddof=1)); ratio=sdpost/sdpre if sdpre>0 else np.nan
    return {"Mean_DeltaD":float(np.mean(d)),"Corr_Dpre_DeltaD":_safe_corr(pre,d),"Oldham":_safe_corr(mean,d),"Log_SDratio":float(np.log(ratio)) if ratio>0 else np.nan}


def observed_geometry_table(obs):
    rows=[]
    for (freq,L),g in obs.groupby(["Frequency_min","Window_days"]):
        for role in ("Treatment","Placebo"):
            pre=g["T_D_pre" if role=="Treatment" else "C_D_pre"].to_numpy(float)
            post=g["T_D_post" if role=="Treatment" else "C_D_post"].to_numpy(float)
            z=_geometry_from_arrays(pre,post); z.update({"Frequency_min":freq,"Window_days":L,"Group":role,"N":len(pre)}); rows.append(z)
    return pd.DataFrame(rows)

def _template_pool(cache,pairs,L,factors,seasonal):
    out=[]
    for _,r in pairs[pairs["Window_pre_post_days"]==L].iterrows():
        for role,col in (("T","Treatment_Date"),("C","Placebo_Date")):
            key=(pd.Timestamp(r[col]).normalize(),L); z=cache.get(key)
            if z is None: continue
            sf_pre=np.array([factors.get(int(s),1.0) for s in z["pre_slots"]],float) if seasonal else np.ones(z["Npre"])
            sf_ev=np.array([factors.get(int(s),1.0) for s in z["event_slots"]],float) if seasonal else np.ones(len(z["event_slots"]))
            sf_post=np.array([factors.get(int(s),1.0) for s in z["post_slots"]],float) if seasonal else np.ones(z["Npost"])
            out.append({"role":role,"date":key[0],"sf_pre":sf_pre,"sf_event":sf_ev,"sf_post":sf_post})
    return out


def _pair_features(pre,post):
    Hp,Rp,_,_=dfa(pre); Hq,Rq,_,_=dfa(post)
    if not all(np.isfinite([Hp,Rp,Hq,Rq])): return None
    dpre=abs(Hp-.5); dpost=abs(Hq-.5); prv=float(np.sqrt(np.sum(np.asarray(pre,float)**2)))
    return {"H_pre":Hp,"H_post":Hq,"R2_pre":Rp,"R2_post":Rq,"D_pre":dpre,"D_post":dpost,"Delta_D":dpost-dpre,"log_PreRV":math.log(max(prv,1e-15))}


def generate_library(model_name,fit,L,templates,target_n=LIBRARY_N,force=False):
    cp=_checkpoint_path("lib",model_name,fit["frequency"],L)
    ledger=cp.with_name(cp.name+".attempted.json")
    if force:
        cp.unlink(missing_ok=True)
        ledger.unlink(missing_ok=True)
    if cp.exists() and not ledger.exists():
        raise RuntimeError(
            f"Legacy Stage 7 library checkpoint without attempted-ID ledger: {cp}. "
            "Its failed candidate IDs cannot be inferred safely. Start in a NEW output directory "
            "instead of silently changing Monte Carlo samples."
        )
    if not ledger.exists():
        initial=ledger.with_name(ledger.name+".partial")
        initial.write_text(json.dumps({"format":1,"attempted_ids":[]}),encoding="utf-8")
        os.replace(initial,ledger)
    previous=json.loads(ledger.read_text(encoding="utf-8"))
    if previous.get("format") != 1:
        raise RuntimeError(f"Unsupported Stage 7 resume ledger: {ledger}")
    done=pd.read_pickle(cp) if cp.exists() else pd.DataFrame()
    attempted=set(int(i) for i in previous["attempted_ids"])
    if len(done):
        if "Library_ID" not in done.columns:
            raise RuntimeError(f"Malformed Stage 7 library checkpoint: {cp}")
        done=done.drop_duplicates("Library_ID",keep="last").sort_values("Library_ID")
        accepted_ids=set(int(i) for i in done["Library_ID"])
        if not all(1 <= i <= target_n for i in accepted_ids):
            raise RuntimeError(f"Stage 7 library contains candidate IDs outside 1..{target_n}: {cp}")
        # draws are always known to have been attempted.
        attempted.update(accepted_ids)
    if not all(1 <= i <= target_n for i in attempted):
        raise RuntimeError(f"Stage 7 attempted-ID ledger does not match target_n={target_n}: {ledger}")
    ids=[i for i in range(target_n) if i+1 not in attempted]
    def one(i):
        rng=np.random.default_rng(RNG_BASE+_stable_int(model_name)*1000+fit["frequency"]*100+L*10+i)
        for attempt in range(20):
            tpl=templates[int(rng.integers(0,len(templates)))]
            npre=len(tpl["sf_pre"]); nev=len(tpl["sf_event"]); npost=len(tpl["sf_post"]); n=npre+nev+npost
            base=simulate_model(fit,n,rng); a=0; pre=base[a:a+npre]*tpl["sf_pre"]; a+=npre; a+=nev; post=base[a:a+npost]*tpl["sf_post"]
            z=_pair_features(pre,post)
            if z is not None and z["R2_pre"]>=DFA_R2_STAGE7 and z["R2_post"]>=DFA_R2_STAGE7:
                z.update({"Library_ID":i+1,"Model":model_name,"Frequency_min":fit["frequency"],"Window_days":L}); return z
        return None
    for k in range(0,len(ids),LIB_BATCH):
        batch=ids[k:k+LIB_BATCH]
        rr=Parallel(n_jobs=min(N_JOBS,len(batch)),backend="loky",verbose=JOBLIB_VERBOSE)(delayed(one)(i) for i in batch)
        rr=[x for x in rr if x is not None]
        if rr:
            done=pd.concat([done,pd.DataFrame(rr)],ignore_index=True)
            done=done.drop_duplicates("Library_ID",keep="last").sort_values("Library_ID")
        partial_cp=cp.with_name(cp.name+".partial")
        pd.to_pickle(done,partial_cp)
        os.replace(partial_cp,cp)
        attempted.update(i+1 for i in batch)
        partial_ledger=ledger.with_name(ledger.name+".partial")
        partial_ledger.write_text(json.dumps({"format":1,"attempted_ids":sorted(attempted)}),encoding="utf-8")
        os.replace(partial_ledger,ledger)
        print(f"    [{model_name} {fit['frequency']}m {L}d] accepted {len(done)}/{target_n}; attempted {len(attempted)}/{target_n}",flush=True)
    if len(attempted)!=target_n:
        raise RuntimeError(f"Stage 7 incomplete library after resume: {len(attempted)}/{target_n}")
    if done.empty:
        raise RuntimeError(f"Stage 7 accepted no valid simulations in {target_n} deterministic draws")
    return done.dropna(subset=["D_pre","D_post","log_PreRV"]).reset_index(drop=True)


def _nearest_lists(lib,target_D,target_R,k=K_NEAREST):
    D=lib["D_pre"].to_numpy(float); R=lib["log_PreRV"].to_numpy(float)
    ds=_robust_scale(np.r_[D,target_D]); rs=_robust_scale(np.r_[R,target_R]); kk=min(k,len(lib)); out=[]
    for d,r in zip(target_D,target_R):
        dist=np.abs(D-d)/ds+np.abs(R-r)/rs; out.append(np.argpartition(dist,kk-1)[:kk])
    return out


def geometry_mc(lib,pre_targets,rv_targets,B,seed):
    lib=lib.reset_index(drop=True); rng=np.random.default_rng(seed); near=_nearest_lists(lib,np.asarray(pre_targets,float),np.asarray(rv_targets,float)); rows=[]
    for b in range(B):
        idx=np.array([rng.choice(a) for a in near]); s=lib.iloc[idx]; z=_geometry_from_arrays(s["D_pre"],s["D_post"]); z["MC_ID"]=b+1; rows.append(z)
    return pd.DataFrame(rows)


def joint_discrepancy(mc,obs):
    X=mc[list(G_COMPONENTS)].dropna().to_numpy(float); o=np.array([obs[k] for k in G_COMPONENTS],float)
    if len(X) < 20 or not np.all(np.isfinite(o)):
        base={"T_Mahalanobis":np.nan,"p_Mahalanobis":np.nan,"T_MaxZ":np.nan,"p_MaxZ":np.nan,"N_components_inside95":0,"Adequate":False}
        return base,pd.DataFrame([{"Component":k,"Observed":obs.get(k,np.nan),"Null_mean":np.nan,"Null_q025":np.nan,"Null_q975":np.nan,"p_two_sided":np.nan,"Inside95":False,"Z_abs":np.nan} for k in G_COMPONENTS])
    mu=X.mean(axis=0); cov=np.cov(X,rowvar=False); ridge=max(np.trace(cov)/len(G_COMPONENTS)*1e-6,1e-12); cov=cov+np.eye(len(G_COMPONENTS))*ridge; inv=np.linalg.pinv(cov)
    dif=X-mu; To=float((o-mu)@inv@(o-mu)); Ts=np.einsum("ij,jk,ik->i",dif,inv,dif); pmah=_upper_empirical(Ts,To)
    sd=X.std(axis=0,ddof=1); sd=np.where(sd>1e-12,sd,1.0); zobs=np.abs((o-mu)/sd); zsim=np.max(np.abs((X-mu)/sd),axis=1); tmax=float(np.max(zobs)); pmax=_upper_empirical(zsim,tmax)
    inside=[]; rows=[]
    for j,k in enumerate(G_COMPONENTS):
        q025,q975=np.quantile(X[:,j],[.025,.975]); pin=_two_sided_empirical(X[:,j],o[j]); ok=bool(q025<=o[j]<=q975); inside.append(ok)
        rows.append({"Component":k,"Observed":o[j],"Null_mean":mu[j],"Null_q025":q025,"Null_q975":q975,"p_two_sided":pin,"Inside95":ok,"Z_abs":zobs[j]})
    adequate=bool(pmah>=ALPHA and pmax>=ALPHA and sum(inside)>=3)
    return {"T_Mahalanobis":To,"p_Mahalanobis":pmah,"T_MaxZ":tmax,"p_MaxZ":pmax,"N_components_inside95":int(sum(inside)),"Adequate":adequate},pd.DataFrame(rows)


def run_model_tests(model_name,fit,lib,obs_freq_L,B=OUTER_MC,block=False):
    rows=[]; comps=[]
    for role in ("Treatment","Placebo"):
        precol="T_D_pre" if role=="Treatment" else "C_D_pre"; postcol="T_D_post" if role=="Treatment" else "C_D_post"; rvcol="T_log_PreRV" if role=="Treatment" else "C_log_PreRV"
        pre=obs_freq_L[precol].to_numpy(float); post=obs_freq_L[postcol].to_numpy(float); obs=_geometry_from_arrays(pre,post)
        mc=geometry_mc(lib,pre,obs_freq_L[rvcol].to_numpy(float),B,RNG_BASE+_stable_int(model_name+role)+fit["frequency"]*1000+int(obs_freq_L["Window_days"].iloc[0]))
        jd,cd=joint_discrepancy(mc,obs); base={"Model":model_name,"Frequency_min":fit["frequency"],"Window_days":int(obs_freq_L["Window_days"].iloc[0]),"Group":role,"N_real":len(pre),**obs,**jd}; rows.append(base)
        cd["Model"]=model_name; cd["Frequency_min"]=fit["frequency"]; cd["Window_days"]=base["Window_days"]; cd["Group"]=role; comps.append(cd)
    return pd.DataFrame(rows),pd.concat(comps,ignore_index=True)


def temporal_blocks(obs30):
    x=obs30.sort_values("Treatment_Date").reset_index(drop=True); ids=np.array_split(np.arange(len(x)),4); out=[]
    for b,ii in enumerate(ids,1):
        g=x.iloc[ii].copy(); g["Temporal_Block"]=b; out.append(g)
    return pd.concat(out,ignore_index=True)

def adjudicate(test_summary):
    rows=[]
    # Full-sample primary: 15m, 30d, both groups.
    s=test_summary[(test_summary.Frequency_min==15)&(test_summary.Window_days==30)]
    passmap={m: bool(len(s[s.Model==m])==2 and s[s.Model==m]["Adequate"].all()) for m in PRIMARY_MODELS}
    first=next((m for m in PRIMARY_MODELS if passmap[m]),None)
    if first=="M0_GARCH_N_raw": interp="prior anomaly attributable to GARCH calibration/fit; no additional structural evidence"
    elif first=="M1_Seasonal_GARCH_N": interp="intraday periodicity is the minimal sufficient ingredient"
    elif first=="M2_Seasonal_GARCH_t": interp="additional heavy tails are necessary/sufficient within the model ladder"
    elif first=="M3_Seasonal_EGARCH_t": interp="EGARCH asymmetry is the first adequate mechanism within the model ladder"
    elif first=="M4_Seasonal_FIGARCH_t": interp="FIGARCH long-memory volatility is the first adequate mechanism within the model ladder"
    else: interp="no primary model jointly reproduces the 15m-30d geometry"
    rows.append({"Decision":"Minimal_ingredient_15m_30d","Result":first if first else "NONE","Interpretation":interp})

    s30=test_summary[(test_summary.Frequency_min==30)&(test_summary.Window_days==30)]
    any30=any(bool(len(s30[s30.Model==m])==2 and s30[s30.Model==m]["Adequate"].all()) for m in PRIMARY_MODELS)
    rows.append({"Decision":"Frequency_robust_residual","Result":bool(first is None and not any30),"Interpretation":"residual persists at 15m and 30m" if (first is None and not any30) else "does not meet frequency-robust residual criterion"})
    return pd.DataFrame(rows)

def main(stage5_excel=None,stage6_excel=None,force=False,smoke=False):
    global LIBRARY_N,OUTER_MC,BLOCK_MC,RUN_TAG
    if smoke:
        LIBRARY_N=min(LIBRARY_N,24); OUTER_MC=min(OUTER_MC,100); BLOCK_MC=min(BLOCK_MC,80)
    s1=load_checkpoint("stage1_data", strict=not smoke)
    stage5_excel=Path(stage5_excel) if stage5_excel else OUTPUT_DIR/"stage5_mechanism_v3_results.xlsx"
    if not stage5_excel.exists(): raise FileNotFoundError(stage5_excel)
    stage6_excel=Path(stage6_excel) if stage6_excel else OUTPUT_DIR/"stage6"/"stage6_dh_calibration_results.xlsx"
    sig={"input_sha":s1.get("input_sha","NA"),"stage5":_file_sig(stage5_excel),"stage6":_file_sig(stage6_excel),"version":STAGE7_VERSION,"lib":LIBRARY_N,"mc":OUTER_MC,"figtrunc":FIGARCH_TRUNC,"fit_max_n":FIT_MAX_N,"season_min_count":SEASON_MIN_COUNT,"models":MODEL_SPECS}
    RUN_TAG=hashlib.sha256(json.dumps(sig,sort_keys=True).encode()).hexdigest()[:12]
    print("="*100); print(f"STAGE 7 - structural discrimination | {STAGE7_VERSION} | RUN_TAG={RUN_TAG}"); print(f"Numba={NUMBA_AVAILABLE}; N_JOBS={N_JOBS}; LIBRARY_N={LIBRARY_N}; OUTER_MC={OUTER_MC}")
    print("M0 uses the full temporal order (no thinning).",flush=True)

    pairs=load_frozen_pairs(stage5_excel)
    all_obs=[]; season_tables=[]; fit_rows=[]; test_rows=[]; comp_rows=[]; block_rows=[]; libs_diag=[]

    freqdata={}
    for freq in FREQUENCIES:
        df,daily=_make_frequency_data(s1,freq); factors,stab=estimate_seasonality(df); stab["Frequency_min"]=freq; season_tables.append(stab)
        obs,cache=observed_pairs_frequency(pairs,df,daily,freq); all_obs.append(obs); freqdata[freq]=(df,daily,factors,cache,obs)
        print(f"[Data {freq}m] valid pairs: "+", ".join(f"{L}d={len(obs[obs.Window_days==L])}" for L in HORIZONS),flush=True)

    # Fits by frequency/model.
    fits={}
    for freq in FREQUENCIES:
        df,daily,factors,cache,obs=freqdata[freq]
        for name,spec in MODEL_SPECS.items():
            cp=_json_checkpoint("fit",name,freq)
            if cp.exists() and not force:
                fit=json.loads(cp.read_text(encoding="utf-8"))
            else:
                x=deseasonalized_returns(df,factors,spec["seasonal"])
                print(f"[FIT] {name} @ {freq}m n={len(x):,}",flush=True)
                fit=fit_model(x,spec["kind"]); fit.update({"model":name,"frequency":freq,"seasonal":spec["seasonal"],"mechanism":spec["mechanism"]})
                cp.write_text(json.dumps(fit,indent=2),encoding="utf-8")
            fits[(name,freq)]=fit
            rr={k:v for k,v in fit.items() if k!="params"}; rr.update({f"param_{k}":v for k,v in fit["params"].items()}); fit_rows.append(rr)
            print(f"    converged={fit['converged']} AIC={fit['AIC']:.1f} params={fit['params']}",flush=True)

    # Full-sample libraries and tests.
    for freq in FREQUENCIES:
        df,daily,factors,cache,obs=freqdata[freq]
        for L in HORIZONS:
            op=obs[obs.Window_days==L].copy()
            if len(op)<15:
                warnings.warn(f"Only {len(op)} pairs reales en {freq}m/{L}d; fragile secondary inference.")
            for name,spec in MODEL_SPECS.items():
                fit=fits[(name,freq)]; templates=_template_pool(cache,pairs,L,factors,spec["seasonal"])
                if not templates: raise RuntimeError(f"No templates for {freq}m {L}d")
                lib=generate_library(name,fit,L,templates,LIBRARY_N,force=force)
                libs_diag.append({"Model":name,"Frequency_min":freq,"Window_days":L,"N_library":len(lib),"Dpre_mean":lib.D_pre.mean(),"Dpre_sd":lib.D_pre.std(),"Delta_mean":lib.Delta_D.mean()})
                ts,cs=run_model_tests(name,fit,lib,op,OUTER_MC); test_rows.append(ts); comp_rows.append(cs)

    tests=pd.concat(test_rows,ignore_index=True); comps=pd.concat(comp_rows,ignore_index=True)

    obs15=freqdata[15][4]; bdf=temporal_blocks(obs15[obs15.Window_days==PRIMARY_HORIZON])
    for b,g in bdf.groupby("Temporal_Block"):
        if len(g) < 8:
            print(f"[Blocks] block {b} skipped: n={len(g)} < 8", flush=True)
            continue
        for name,spec in MODEL_SPECS.items():
            fit=fits[(name,15)]; lib=pd.read_pickle(_checkpoint_path("lib",name,15,PRIMARY_HORIZON))
            ts,_=run_model_tests(name,fit,lib,g,BLOCK_MC); ts["Temporal_Block"]=int(b); block_rows.append(ts)
    blocks=pd.concat(block_rows,ignore_index=True) if block_rows else pd.DataFrame()

    decisions=adjudicate(tests)
    obs_all=pd.concat(all_obs,ignore_index=True); geom=observed_geometry_table(obs_all)
    season=pd.concat(season_tables,ignore_index=True); fitsdf=pd.DataFrame(fit_rows); libdiag=pd.DataFrame(libs_diag)

    audit=[]
    p15=obs_all[obs_all.Frequency_min==15]
    for L,g in p15.groupby("Window_days"):
        orig=pairs[pairs.Window_pre_post_days==L]
        mg=g.merge(orig[["Treatment_Date","Placebo_Date","Treatment_D_H_pre","Treatment_Delta_D_H","Placebo_D_H_pre_actual","Placebo_Delta_D_H"]],on=["Treatment_Date","Placebo_Date"],how="left")
        audit.append({"Window_days":L,"N":len(mg),"max_abs_diff_T_Dpre":float(np.nanmax(np.abs(mg.T_D_pre-mg.Treatment_D_H_pre))),"max_abs_diff_T_Delta":float(np.nanmax(np.abs(mg.T_Delta-mg.Treatment_Delta_D_H))),"max_abs_diff_C_Dpre":float(np.nanmax(np.abs(mg.C_D_pre-mg.Placebo_D_H_pre_actual))),"max_abs_diff_C_Delta":float(np.nanmax(np.abs(mg.C_Delta-mg.Placebo_Delta_D_H)))})
    audit=pd.DataFrame(audit)

    garch_amend=[]
    if stage6_excel.exists():
        try:
            old=pd.read_excel(stage6_excel,sheet_name="GARCH_Fit").iloc[0].to_dict()
            new=fits[("M0_GARCH_N_raw",15)]["params"]
            garch_amend=[
                {"Item":"Stage6_fit_method","Old":"if n>250000: uniform subsampling treated as consecutive","New":"complete temporal order (or a contiguous block only if FIT_MAX_N is explicitly set)"},
                {"Item":"alpha","Old":old.get("alpha",np.nan),"New":new.get("alpha",np.nan)},
                {"Item":"beta","Old":old.get("beta",np.nan),"New":new.get("beta",np.nan)},
                {"Item":"persistence","Old":old.get("alpha_plus_beta",np.nan),"New":new.get("persistence",np.nan)},
            ]
        except Exception as e:
            garch_amend=[{"Item":"Stage6_compare_error","Old":str(e),"New":""}]
    garch_amend=pd.DataFrame(garch_amend)

    params=pd.DataFrame([
        {"Parameter":"STAGE7_VERSION","Value":STAGE7_VERSION},{"Parameter":"RUN_TAG","Value":RUN_TAG},{"Parameter":"LIBRARY_N","Value":LIBRARY_N},{"Parameter":"OUTER_MC","Value":OUTER_MC},{"Parameter":"BLOCK_MC","Value":BLOCK_MC},{"Parameter":"K_NEAREST","Value":K_NEAREST},{"Parameter":"FIGARCH_TRUNC","Value":FIGARCH_TRUNC},{"Parameter":"FIT_MAX_N","Value":FIT_MAX_N},{"Parameter":"NUMBA_AVAILABLE","Value":NUMBA_AVAILABLE},{"Parameter":"Python","Value":platform.python_version()},{"Parameter":"Stage5_SHA256","Value":_file_sig(stage5_excel)},{"Parameter":"Stage6_SHA256","Value":_file_sig(stage6_excel)},
    ])

    out=STAGE7_OUTPUT_DIR/"stage7_structural_discrimination_results.xlsx"
    with pd.ExcelWriter(out,engine="openpyxl") as w:
        geom.to_excel(w,sheet_name="Observed_Geometry",index=False)
        obs_all.to_excel(w,sheet_name="Observed_Pairs_15_30m",index=False)
        audit.to_excel(w,sheet_name="Audit_15m_vs_Stage5",index=False)
        season.to_excel(w,sheet_name="Intraday_Seasonality",index=False)
        fitsdf.to_excel(w,sheet_name="Model_Fits",index=False)
        libdiag.to_excel(w,sheet_name="Library_Diagnostics",index=False)
        tests.to_excel(w,sheet_name="Joint_Model_Tests",index=False)
        comps.to_excel(w,sheet_name="Component_Tests",index=False)
        blocks.to_excel(w,sheet_name="Temporal_Block_Tests",index=False)
        decisions.to_excel(w,sheet_name="Preregistered_Decisions",index=False)
        garch_amend.to_excel(w,sheet_name="Stage6_GARCH_Amendment",index=False)
        params.to_excel(w,sheet_name="Stage7_Parameters",index=False)
    print("\nDECISIONS:"); print(decisions.to_string(index=False)); print(f"\n{out}")
    return out


if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--stage5-excel",default=None)
    ap.add_argument("--stage6-excel",default=None)
    ap.add_argument("--force",action="store_true")
    ap.add_argument("--smoke",action="store_true")
    args=ap.parse_args()
    main(args.stage5_excel,args.stage6_excel,args.force,args.smoke)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage7b_jump_regime.py
# -*- coding: utf-8 -*-
"""Stage 7B: local recalibration and jump model."""
from __future__ import annotations

from config import *
import stage7_structural_discrimination as e7
from stage7_structural_discrimination import (
    load_checkpoint, _make_frequency_data, estimate_seasonality, deseasonalized_returns,
    load_frozen_pairs, observed_pairs_frequency, _template_pool, _pair_features,
    _geometry_from_arrays, joint_discrepancy, geometry_mc, _robust_scale,
    _stable_int, _sigmoid, _logit, _garch_h, _nll_normal, _sim_garch_core,
    dfa, _normalize_dates, _file_sig, _slot_minutes, RANDOM_SEED, N_JOBS,
    DFA_R2_MIN, OUTPUT_DIR, CHECKPOINT_DIR, MATCH_PREVOL_DAYS
)

import argparse, hashlib, json, math, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests

STAGE7B_VERSION = "XAUUSD_STAGE7B_JUMPREGIME_v1_2026-09-12"
OUTDIR = OUTPUT_DIR / "stage7b"
CPDIR = CHECKPOINT_DIR / "stage7b"
OUTDIR.mkdir(parents=True, exist_ok=True); CPDIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
HORIZONS = (30, 60)
FREQUENCIES = (15, 30)
PRIMARY_HORIZON = 30
LIBRARY_N = int(os.getenv("XAUUSD_STAGE7B_LIBRARY_N", "900"))
OUTER_MC = int(os.getenv("XAUUSD_STAGE7B_OUTER_MC", "10000"))
BLOCK_LIBRARY_N = int(os.getenv("XAUUSD_STAGE7B_BLOCK_LIBRARY_N", "600"))
BLOCK_MC = int(os.getenv("XAUUSD_STAGE7B_BLOCK_MC", "5000"))
K_NEAREST = int(os.getenv("XAUUSD_STAGE7B_K_NEAREST", "25"))
MAX_REJECTION_ATTEMPTS = int(os.getenv("XAUUSD_STAGE7B_MAX_REJECTION", "120"))
RNG_BASE = RANDOM_SEED + 770_000
G_COMPONENTS = ("Mean_DeltaD","Corr_Dpre_DeltaD","Oldham","Log_SDratio")
RUN_TAG = "UNSET"

MU1 = math.sqrt(2.0/math.pi)
OMEGA_BNS = MU1**(-4) + 2*MU1**(-2) - 5


def bns_p_one_sided(returns, slots, minutes):
    r=np.asarray(returns,float); s=np.asarray(slots,int)
    m=np.isfinite(r); r=r[m]; s=s[m]
    M=len(r)
    if M < 40: return np.nan, np.nan
    rv=float(np.sum(r*r))
    if rv<=0: return np.nan, np.nan
    if M<3: return np.nan, np.nan
    adj=np.diff(s)==minutes
    pair_idx=np.where(adj)[0]
    trip_idx=np.where(adj[:-1] & adj[1:])[0]
    P=len(pair_idx); T=len(trip_idx)
    if P<30 or T<20: return np.nan, np.nan
    bv=(math.pi/2.0)*(M/P)*float(np.sum(np.abs(r[pair_idx])*np.abs(r[pair_idx+1])))
    mu43 = (2.0**(2.0/3.0))*math.gamma(7.0/6.0)/math.sqrt(math.pi)
    tq=(M*(M/T))*(mu43**-3)*float(np.sum(np.abs(r[trip_idx])**(4/3)*np.abs(r[trip_idx+1])**(4/3)*np.abs(r[trip_idx+2])**(4/3)))
    denom=math.sqrt(max(OMEGA_BNS*max(1.0, tq/max(bv*bv,1e-30)),1e-30))
    z=math.sqrt(M)*(1.0-bv/rv)/denom
    # norm.sf without scipy.stats for speed
    p=0.5*math.erfc(z/math.sqrt(2.0))
    return float(z), float(p)


def _fit_prepare_jump(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]; x=x-np.mean(x)
    scale=float(np.std(x,ddof=1)); y=x/scale
    return y,scale


def _jump_unpack(th, p_low, p_high):
    omega=math.exp(float(th[0]))
    rho=.999*_sigmoid(float(th[1])); share=_sigmoid(float(th[2]))
    alpha=rho*share; beta=rho*(1-share)
    p=float(p_low) + (float(p_high)-float(p_low))*_sigmoid(float(th[3]))
    muJ=6.0*math.tanh(float(th[4]))
    sigJ=0.05+9.95*_sigmoid(float(th[5]))
    return omega,alpha,beta,p,muJ,sigJ


def _jump_mix_nll(y,th,p_low,p_high):
    omega,a,b,p,muJ,sigJ=_jump_unpack(th,p_low,p_high)
    h=_garch_h(y,omega,a,b)
    if np.any(~np.isfinite(h)) or np.any(h<=0): return 1e100
    # no-jump N(0,h), jump N(muJ,h+sigJ^2)
    l0=math.log(max(1-p,1e-12)) - .5*(np.log(2*math.pi*h)+y*y/h)
    hj=h+sigJ*sigJ
    l1=math.log(max(p,1e-12)) - .5*(np.log(2*math.pi*hj)+(y-muJ)**2/hj)
    ll=logsumexp(np.vstack([l0,l1]),axis=0)
    val=-float(np.sum(ll))
    return val if np.isfinite(val) else 1e100


def fit_garch_jump(x, p_center, garch_start=None):
    y,scale=_fit_prepare_jump(x); n=len(y)
    if n<2000: raise RuntimeError(f"Too few returns for GARCH-Jump: {n}")
    if garch_start:
        p0=garch_start["params"]
        rho=min(float(p0["alpha"]+p0["beta"]),.998)
        share=float(p0["alpha"])/max(rho,1e-8)
        omega=float(p0["omega"])
    else:
        omega=.01; rho=.97; share=.08
    p_center=float(max(p_center,1e-6)); p_low=max(1e-6,0.50*p_center); p_high=min(0.02,2.0*p_center)
    pinit=p_center
    t0=np.array([math.log(max(omega,1e-8)),_logit(rho/.999),_logit(np.clip(share,1e-4,.9999)),
                 _logit(np.clip((pinit-p_low)/max(p_high-p_low,1e-12),1e-6,1-1e-6)),0.0,_logit((3.0-.05)/9.95)],float)
    res=minimize(lambda th:_jump_mix_nll(y,th,p_low,p_high),t0,method="L-BFGS-B",options={"maxiter":450,"ftol":1e-9,"maxls":30})
    omega,a,b,p,muJ,sigJ=_jump_unpack(res.x,p_low,p_high)
    nll=float(res.fun); k=6
    return {"kind":"garch_jump_n","scale":scale,"n_fit":int(n),"converged":bool(res.success),
            "message":str(res.message),"nll":nll,"AIC":2*k+2*nll,"BIC":k*math.log(n)+2*nll,
            "params":{"omega":omega,"alpha":a,"beta":b,"p_jump_bar":p,
                      "lambda_bar":-math.log(max(1-p,1e-12)),"mu_jump":muJ,"sigma_jump":sigJ,
                      "p_jump_center_BNS":p_center,"p_jump_lower":p_low,"p_jump_upper":p_high,
                      "p_jump_at_bound":bool((p-p_low)<0.02*(p_high-p_low) or (p_high-p)<0.02*(p_high-p_low)),
                      "persistence":a+b}}


def simulate_garch_jump(fit,n,rng,burn=800):
    p=fit["params"]; N=n+burn
    h=np.empty(N,float); y=np.empty(N,float)
    pers=p["alpha"]+p["beta"]
    h[0]=max(p["omega"]/max(1-pers,1e-4),1e-8)
    z=rng.standard_normal(N)
    hit=rng.random(N)<p["p_jump_bar"]
    J=np.zeros(N,float); nh=int(hit.sum())
    if nh: J[hit]=rng.normal(p["mu_jump"],p["sigma_jump"],size=nh)
    y[0]=math.sqrt(h[0])*z[0]+J[0]
    for t in range(1,N):
        h[t]=p["omega"]+p["alpha"]*y[t-1]*y[t-1]+p["beta"]*h[t-1]
        if not np.isfinite(h[t]) or h[t]<1e-10: h[t]=1e-10
        y[t]=math.sqrt(h[t])*z[t]+J[t]
    return y[burn:]*float(fit["scale"]), hit[burn:]


def _template_pool_jump(cache,pairs,L,factors):
    out=[]
    for _,r in pairs[pairs["Window_pre_post_days"]==L].iterrows():
        for role,col in (("Treatment","Treatment_Date"),("Placebo","Placebo_Date")):
            key=(pd.Timestamp(r[col]).normalize(),L); z=cache.get(key)
            if z is None: continue
            out.append({"source_role":role,"date":key[0],
                "sf_pre":np.array([factors.get(int(s),1.0) for s in z["pre_slots"]],float),
                "sf_event":np.array([factors.get(int(s),1.0) for s in z["event_slots"]],float),
                "sf_post":np.array([factors.get(int(s),1.0) for s in z["post_slots"]],float),
                "event_slots":np.asarray(z["event_slots"],int)})
    return out


def _cp(name,*parts):
    return CPDIR/("_".join([name,RUN_TAG]+[str(x) for x in parts])+".pkl")


def build_jump_conditioned_library(model_name,fit,L,templates,minutes,pstar,target_n=LIBRARY_N,force=False):
    cp=_cp("jlib",model_name,minutes,L)
    if cp.exists() and not force:
        d=pd.read_pickle(cp)
        if len(d)>=2*target_n: return d
    rows=[] if force or not cp.exists() else pd.read_pickle(cp).to_dict("records")
    # Separate IDs by role; permits resumption.
    have={(r.get("Role"),int(r.get("Library_ID",-1))) for r in rows}
    tasks=[]
    for role in ("Treatment","Placebo"):
        for i in range(target_n):
            if (role,i+1) not in have: tasks.append((role,i))

    def one(role,i):
        rng=np.random.default_rng(RNG_BASE+_stable_int(model_name+role)*1000+minutes*100+L*10+i)
        for attempt in range(1,MAX_REJECTION_ATTEMPTS+1):
            tpl=templates[int(rng.integers(0,len(templates)))]
            npre=len(tpl["sf_pre"]); nev=len(tpl["sf_event"]); npost=len(tpl["sf_post"]); n=npre+nev+npost
            base,hits=simulate_garch_jump(fit,n,rng)
            a=0; pre=base[a:a+npre]*tpl["sf_pre"]; a+=npre
            ev=base[a:a+nev]*tpl["sf_event"]; a+=nev
            post=base[a:a+npost]*tpl["sf_post"]
            z_bns,p_bns=bns_p_one_sided(ev,tpl["event_slots"],minutes)
            if not np.isfinite(p_bns): continue
            qualifies = (p_bns <= pstar)
            latent_event=int(np.sum(hits[npre:npre+nev]))
            if role=="Treatment" and (not qualifies or latent_event<1): continue
            if role=="Placebo" and qualifies: continue
            f=_pair_features(pre,post)
            if f is None or f["R2_pre"]<DFA_R2_MIN or f["R2_post"]<DFA_R2_MIN: continue
            f.update({"Role":role,"Library_ID":i+1,"Model":model_name,"Frequency_min":minutes,
                      "Window_days":L,"BNS_Z_event":z_bns,"BNS_p_event":p_bns,
                      "Latent_jump_count_event":latent_event,"Attempts":attempt})
            return f
        return None

    batch=24
    for k in range(0,len(tasks),batch):
        rr=Parallel(n_jobs=min(N_JOBS,len(tasks[k:k+batch])),backend="loky")(delayed(one)(*x) for x in tasks[k:k+batch])
        rr=[x for x in rr if x is not None]; rows.extend(rr)
        pd.DataFrame(rows).to_pickle(cp)
        print(f"    [{model_name} {minutes}m {L}d] {len(rows)}/{2*target_n} accepted",flush=True)
    return pd.DataFrame(rows)


def nearest_mc_role(lib,role,pre_targets,rv_targets,B,seed):
    x=lib[lib.Role==role].reset_index(drop=True)
    return geometry_mc(x,pre_targets,rv_targets,B,seed)


def add_fdr(df,pcol,family_cols,outcol="p_FDR"):
    out=df.copy(); out[outcol]=np.nan
    if len(out)==0: return out
    for _,idx in out.groupby(family_cols,dropna=False).groups.items():
        ids=list(idx); p=pd.to_numeric(out.loc[ids,pcol],errors="coerce").to_numpy(float); m=np.isfinite(p)
        if m.any():
            q=np.full(len(ids),np.nan); q[m]=multipletests(p[m],alpha=ALPHA,method="fdr_bh")[1]; out.loc[ids,outcol]=q
    return out


def main(stage5_excel=None,stage7_excel=None,force=False,smoke=False):
    global LIBRARY_N,OUTER_MC,BLOCK_LIBRARY_N,BLOCK_MC,RUN_TAG
    if smoke:
        LIBRARY_N=min(LIBRARY_N,30); OUTER_MC=min(OUTER_MC,120); BLOCK_LIBRARY_N=min(BLOCK_LIBRARY_N,24); BLOCK_MC=min(BLOCK_MC,100)
    s1=load_checkpoint("stage1_data", strict=not smoke)
    stage5_excel=Path(stage5_excel) if stage5_excel else OUTPUT_DIR/"stage5_mechanism_v3_results.xlsx"
    stage7_excel=Path(stage7_excel) if stage7_excel else OUTPUT_DIR/"stage7"/"stage7_structural_discrimination_results.xlsx"
    if not stage5_excel.exists(): raise FileNotFoundError(stage5_excel)
    if not stage7_excel.exists(): raise FileNotFoundError(stage7_excel)
    sig={"input_sha":s1.get("input_sha","NA"),"stage5":_file_sig(stage5_excel),"stage7":_file_sig(stage7_excel),
         "version":STAGE7B_VERSION,"lib":LIBRARY_N,"mc":OUTER_MC,"blocklib":BLOCK_LIBRARY_N,"blockmc":BLOCK_MC}
    RUN_TAG=hashlib.sha256(json.dumps(sig,sort_keys=True).encode()).hexdigest()[:12]
    e7.RUN_TAG=RUN_TAG
    print("="*100); print(f"STAGE 7B | {STAGE7B_VERSION} | RUN_TAG={RUN_TAG}")

    pairs=load_frozen_pairs(stage5_excel)
    bns=pd.read_excel(stage5_excel,sheet_name="BNS_Daily_Jump_Test")
    bns["Date"]=_normalize_dates(bns["Date"])
    realjump=bns[bns["Jump_BNS_FDR"].astype(bool) & pd.to_numeric(bns["p_BNS_one_sided"],errors="coerce").notna()]
    if len(realjump)==0: raise RuntimeError("Could not extract real BNS threshold")
    pstar=float(realjump["p_BNS_one_sided"].max())
    print(f"[BNS] empirical threshold equivalent to actual FDR rejection: p <= {pstar:.8g}; real jumps={len(realjump)}")

    # Observed data/frequencies and pairs.
    freqdata={}; observed=[]
    for freq in FREQUENCIES:
        df,daily=_make_frequency_data(s1,freq); factors,stab=estimate_seasonality(df)
        obs,cache=observed_pairs_frequency(pairs,df,daily,freq)
        freqdata[freq]=(df,daily,factors,cache,obs); observed.append(obs)

    local_fit_rows=[]; local_test_rows=[]; local_comp_rows=[]
    df15,daily15,fg15,cache15,obs15=freqdata[15]
    op30=obs15[obs15.Window_days==30].sort_values("Treatment_Date").reset_index(drop=True)
    split=np.array_split(np.arange(len(op30)),4)
    for b,ii in enumerate(split,1):
        g=op30.iloc[ii].copy(); d0=pd.Timestamp(g.Treatment_Date.min()).normalize(); d1=pd.Timestamp(g.Treatment_Date.max()).normalize()
        m=(df15.index.normalize()>=d0)&(df15.index.normalize()<=d1); dblock=df15.loc[m].copy()
        fac,_=estimate_seasonality(dblock)
        x=deseasonalized_returns(dblock,fac,True)
        from stage7_structural_discrimination import fit_model, generate_library, run_model_tests
        fit=fit_model(x,"garch_n"); fit.update({"model":f"B{b}_Local_M1","frequency":15,"seasonal":True,"mechanism":"local seasonal GARCH-N"})
        rr={"Temporal_Block":b,"Start":d0,"End":d1,**{k:v for k,v in fit.items() if k!="params"},**{f"param_{k}":v for k,v in fit["params"].items()}}
        local_fit_rows.append(rr)
        templates=_template_pool(cache15,pairs[pairs.Treatment_Date.isin(g.Treatment_Date)],30,fac,True)
        if not templates:
            templates=_template_pool(cache15,pairs,30,fac,True)
        lib=generate_library(f"B{b}_Local_M1_7B",fit,30,templates,BLOCK_LIBRARY_N,force=force)
        ts,cs=run_model_tests(f"B{b}_Local_M1_7B",fit,lib,g,BLOCK_MC)
        ts["Temporal_Block"]=b; ts["Block_Start"]=d0; ts["Block_End"]=d1; local_test_rows.append(ts)
        cs["Temporal_Block"]=b; local_comp_rows.append(cs)
    localtests=pd.concat(local_test_rows,ignore_index=True); localcomps=pd.concat(local_comp_rows,ignore_index=True)
    localtests=add_fdr(localtests,"p_Mahalanobis",[],"q_Mahalanobis_BH") if False else localtests
    for pc,qc in [("p_Mahalanobis","q_Mahalanobis_BH"),("p_MaxZ","q_MaxZ_BH")]:
        p=localtests[pc].to_numpy(float); localtests[qc]=multipletests(p,method="fdr_bh")[1]
    localtests["Adequate_after_BH"]=((localtests.q_Mahalanobis_BH>=ALPHA)&(localtests.q_MaxZ_BH>=ALPHA)&(localtests.N_components_inside95>=3))

    jump_fit_rows=[]; jump_tests=[]; jump_comps=[]; jump_libdiag=[]
    for freq in FREQUENCIES:
        df,daily,factors,cache,obs=freqdata[freq]
        x=deseasonalized_returns(df,factors,True)
        # GARCH-N starting point to stabilize MLE.
        from stage7_structural_discrimination import fit_model
        g0=fit_model(x,"garch_n")
        fitcp=CPDIR/f"jumpfit_{RUN_TAG}_{freq}m.json"
        if fitcp.exists() and not force:
            jfit=json.loads(fitcp.read_text(encoding="utf-8"))
        else:
            print(f"[JUMP FIT] {freq}m n={len(x):,} (full temporal order)",flush=True)
            valid_bns=bns[pd.to_numeric(bns["p_BNS_one_sided"],errors="coerce").notna()]
            q_day=len(realjump)/max(len(valid_bns),1)
            med_bars=float(df.dropna(subset=["LogRet"]).groupby(df.dropna(subset=["LogRet"]).index.normalize()).size().median())
            p_center=1.0-(1.0-q_day)**(1.0/max(med_bars,1.0))
            jfit=fit_garch_jump(x,p_center,g0); jfit.update({"model":"J1_Seasonal_GARCH_Jump_N","frequency":freq,"seasonal":True,
                "real_BNS_day_rate":q_day,"median_bars_per_day":med_bars})
            fitcp.write_text(json.dumps(jfit,indent=2),encoding="utf-8")
        row={"frequency":freq,**{k:v for k,v in jfit.items() if k!="params"},**{f"param_{k}":v for k,v in jfit["params"].items()},
             "GARCH_N_AIC":g0["AIC"],"GARCH_N_BIC":g0["BIC"],"Delta_AIC_jump_minus_GARCH":jfit["AIC"]-g0["AIC"],"Delta_BIC_jump_minus_GARCH":jfit["BIC"]-g0["BIC"]}
        jump_fit_rows.append(row)
        for L in HORIZONS:
            op=obs[obs.Window_days==L].copy(); templates=_template_pool_jump(cache,pairs,L,factors)
            lib=build_jump_conditioned_library("J1_Seasonal_GARCH_Jump_N",jfit,L,templates,freq,pstar,LIBRARY_N,force)
            for role in ("Treatment","Placebo"):
                lr=lib[lib.Role==role]
                jump_libdiag.append({"Frequency_min":freq,"Window_days":L,"Role":role,"N":len(lr),
                                     "Acceptance_rate_proxy":float(len(lr)/max(lr.Attempts.sum(),1)) if len(lr) else np.nan,
                                     "Mean_attempts":float(lr.Attempts.mean()) if len(lr) else np.nan,
                                     "Mean_BNS_p":float(lr.BNS_p_event.mean()) if len(lr) else np.nan,
                                     "Mean_latent_jumps_event":float(lr.Latent_jump_count_event.mean()) if len(lr) else np.nan})
                precol="T_D_pre" if role=="Treatment" else "C_D_pre"; postcol="T_D_post" if role=="Treatment" else "C_D_post"; rvcol="T_log_PreRV" if role=="Treatment" else "C_log_PreRV"
                pre=op[precol].to_numpy(float); post=op[postcol].to_numpy(float); rv=op[rvcol].to_numpy(float)
                obsgeom=_geometry_from_arrays(pre,post)
                mc=nearest_mc_role(lib,role,pre,rv,OUTER_MC,RNG_BASE+_stable_int(role)+freq*1000+L)
                jd,cd=joint_discrepancy(mc,obsgeom)
                base={"Model":"J1_Seasonal_GARCH_Jump_N","Frequency_min":freq,"Window_days":L,"Group":role,"N_real":len(pre),**obsgeom,**jd}
                jump_tests.append(base)
                cd["Model"]="J1_Seasonal_GARCH_Jump_N"; cd["Frequency_min"]=freq; cd["Window_days"]=L; cd["Group"]=role; jump_comps.append(cd)
    jtests=pd.DataFrame(jump_tests); jcomps=pd.concat(jump_comps,ignore_index=True)

    for pc,qc in [("p_Mahalanobis","q_Mahalanobis_BH"),("p_MaxZ","q_MaxZ_BH")]:
        p=jtests[pc].to_numpy(float); jtests[qc]=multipletests(p,method="fdr_bh")[1]
    jtests["Adequate_after_BH"]=((jtests.q_Mahalanobis_BH>=ALPHA)&(jtests.q_MaxZ_BH>=ALPHA)&(jtests.N_components_inside95>=3))

    jcomps["q_component_BH"]=np.nan; jcomps["FDR_Family"]="exploratory_unadjusted"
    mask=(jcomps.Group=="Treatment")&(jcomps.Window_days==30)&(jcomps.Component=="Mean_DeltaD")
    if mask.sum():
        jcomps.loc[mask,"q_component_BH"]=multipletests(jcomps.loc[mask,"p_two_sided"],method="fdr_bh")[1]; jcomps.loc[mask,"FDR_Family"]="Primary_MeanDeltaD_Treatment_30d_15m30m"
    mask2=(jcomps.Group=="Treatment")&(jcomps.Window_days==30)&(jcomps.Frequency_min==30)&(jcomps.Component.isin(["Oldham","Log_SDratio"]))
    if mask2.sum():
        jcomps.loc[mask2,"q_component_BH"]=multipletests(jcomps.loc[mask2,"p_two_sided"],method="fdr_bh")[1]; jcomps.loc[mask2,"FDR_Family"]="Secondary_30m_Oldham_SDratio"

    # Transparent decisions.
    dec=[]
    loc_pass=localtests.groupby("Temporal_Block")["Adequate_after_BH"].all()
    dec.append({"Decision":"H_B_local_recalibration_all4","Result":bool(loc_pass.all()),"Detail":f"blocks passing both groups after BH: {int(loc_pass.sum())}/4"})
    jt30=jtests[(jtests.Window_days==30)&(jtests.Group=="Treatment")]
    dec.append({"Decision":"H_A_jump_DGP_explains_Treatment_30d_both_freq","Result":bool(len(jt30)==2 and jt30.Adequate_after_BH.all()),"Detail":"J1 must be adequate at 15m and 30m after BH"})
    foc=jcomps[(jcomps.FDR_Family=="Primary_MeanDeltaD_Treatment_30d_15m30m")]
    dec.append({"Decision":"MeanDeltaD_residual_after_jump_DGP","Result":bool((foc.q_component_BH<ALPHA).any()),"Detail":"True means focal Mean_DeltaD remains rejected at >=1 frequency after BH"})

    outfile=OUTDIR/"stage7b_jump_regime_results.xlsx"
    with pd.ExcelWriter(outfile,engine="openpyxl") as w:
        pd.DataFrame(local_fit_rows).to_excel(w,sheet_name="Local_Block_Fits",index=False)
        localtests.to_excel(w,sheet_name="Local_Block_Joint_Tests",index=False)
        localcomps.to_excel(w,sheet_name="Local_Block_Components",index=False)
        pd.DataFrame(jump_fit_rows).to_excel(w,sheet_name="Jump_Model_Fits",index=False)
        pd.DataFrame(jump_libdiag).to_excel(w,sheet_name="Jump_Library_Diagnostics",index=False)
        jtests.to_excel(w,sheet_name="Jump_Joint_Tests_FDR",index=False)
        jcomps.to_excel(w,sheet_name="Jump_Component_Tests_FDR",index=False)
        pd.DataFrame(dec).to_excel(w,sheet_name="Preregistered_Decisions",index=False)
        pd.DataFrame([
            {"Parameter":"STAGE7B_VERSION","Value":STAGE7B_VERSION},{"Parameter":"RUN_TAG","Value":RUN_TAG},
            {"Parameter":"BNS_real_pstar","Value":pstar},{"Parameter":"LIBRARY_N_per_role","Value":LIBRARY_N},
            {"Parameter":"OUTER_MC","Value":OUTER_MC},{"Parameter":"BLOCK_LIBRARY_N","Value":BLOCK_LIBRARY_N},
            {"Parameter":"BLOCK_MC","Value":BLOCK_MC},{"Parameter":"ALPHA","Value":ALPHA},
            {"Parameter":"Primary_endpoint","Value":"Mean_DeltaD Treatment 30d across 15m/30m"},
            {"Parameter":"Jump_DGP","Value":"Seasonal GARCH + Bernoulli(Poisson-limit) Normal jumps, ML mixture fit; BNS-conditioned treatment/placebo"},
        ]).to_excel(w,sheet_name="Stage7B_Parameters",index=False)
    print(f"{outfile}")
    print(pd.DataFrame(dec).to_string(index=False))
    return outfile

if __name__=="__main__":
    ap=argparse.ArgumentParser(); ap.add_argument("--stage5-excel"); ap.add_argument("--stage7-excel"); ap.add_argument("--force",action="store_true"); ap.add_argument("--smoke",action="store_true")
    a=ap.parse_args(); main(a.stage5_excel,a.stage7_excel,a.force,a.smoke)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage7c_garcht_jump.py
# -*- coding: utf-8 -*-
"""Stage 7C: seasonal GARCH-t with BNS-anchored jumps."""
from __future__ import annotations

from config import *
import stage7_structural_discrimination as e7
import stage7b_jump_regime as e7b

from stage7_structural_discrimination import (
    load_checkpoint, _make_frequency_data, estimate_seasonality,
    deseasonalized_returns, load_frozen_pairs, observed_pairs_frequency,
    _pair_features, _geometry_from_arrays, joint_discrepancy, geometry_mc,
    _robust_scale, _stable_int, _file_sig, _normalize_dates, _slot_minutes,
    _garch_h, fit_model, _draw_z, RANDOM_SEED, N_JOBS, DFA_R2_STAGE7,
    OUTPUT_DIR, CHECKPOINT_DIR, MATCH_PREVOL_DAYS
)
from stage7b_jump_regime import bns_p_one_sided, _template_pool_jump

import argparse, hashlib, json, math, os
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests

STAGE7C_VERSION = "XAUUSD_STAGE7C_GARCHT_JUMP_v1_2026-09-12"
OUTDIR = OUTPUT_DIR / "stage7c"
CPDIR = CHECKPOINT_DIR / "stage7c"
OUTDIR.mkdir(parents=True, exist_ok=True)
CPDIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
FREQUENCIES = (15, 30)
HORIZONS = (30, 60)
PRIMARY_HORIZON = 30
LIBRARY_N = int(os.getenv("XAUUSD_STAGE7C_LIBRARY_N", "1000"))
OUTER_MC = int(os.getenv("XAUUSD_STAGE7C_OUTER_MC", "12000"))
K_NEAREST = int(os.getenv("XAUUSD_STAGE7C_K_NEAREST", "25"))
MAX_REJECTION_ATTEMPTS = int(os.getenv("XAUUSD_STAGE7C_MAX_REJECTION", "160"))
LIB_BATCH = int(os.getenv("XAUUSD_STAGE7C_LIB_BATCH", "20"))
RNG_BASE = RANDOM_SEED + 780_000
RUN_TAG = "UNSET"
G_COMPONENTS = ("Mean_DeltaD", "Corr_Dpre_DeltaD", "Oldham", "Log_SDratio")


def _cp(name, *parts):
    return CPDIR / ("_".join([name, RUN_TAG] + [str(x) for x in parts]) + ".pkl")


def _full_deseasonalized_series(df, factors):
    r = pd.to_numeric(df["LogRet"], errors="coerce")
    m = r.notna()
    idx = pd.DatetimeIndex(df.index[m])
    x = r[m].to_numpy(float)
    slots = _slot_minutes(idx)
    sf = np.array([factors.get(int(s), 1.0) for s in slots], float)
    x = x / sf
    x = x - np.mean(x)
    return pd.Series(x, index=idx, name="Deseasonalized_LogRet")


def calibrate_bns_anchored_jump_marks(df, factors, fit_t, bns, realjump):
    s = _full_deseasonalized_series(df, factors)
    y = s.to_numpy(float) / float(fit_t["scale"])
    p = fit_t["params"]
    h = _garch_h(y, float(p["omega"]), float(p["alpha"]), float(p["beta"]))
    z = y / np.sqrt(np.maximum(h, 1e-12))
    tab = pd.DataFrame({"DateTime": s.index, "y": y, "h": h, "z": z})
    tab["Date"] = pd.DatetimeIndex(tab.DateTime).normalize()
    jump_dates = set(pd.DatetimeIndex(realjump["Date"]).normalize())
    rows = []
    for d, g in tab[tab.Date.isin(jump_dates)].groupby("Date", sort=True):
        if len(g) == 0:
            continue
        j = int(np.nanargmax(np.abs(g["z"].to_numpy(float))))
        rr = g.iloc[j]
        rows.append({
            "Date": pd.Timestamp(d), "DateTime_proxy": pd.Timestamp(rr.DateTime),
            "JumpMark_y": float(rr.y), "Abs_z_proxy": float(abs(rr.z)),
            "h_proxy": float(rr.h), "sign": int(np.sign(rr.y))
        })
    marks = pd.DataFrame(rows)
    if len(marks) < 30:
        raise RuntimeError(f"Too few BNS jump marks for calibration: {len(marks)}")

    valid_bns = bns[pd.to_numeric(bns["p_BNS_one_sided"], errors="coerce").notna()]
    q_day = len(realjump) / max(len(valid_bns), 1)
    rr = df.dropna(subset=["LogRet"])
    bars_day = rr.groupby(rr.index.normalize()).size()
    med_bars = float(bars_day.median())
    p_bar = 1.0 - (1.0 - q_day) ** (1.0 / max(med_bars, 1.0))
    diag = {
        "N_jump_marks": int(len(marks)), "BNS_day_rate": float(q_day),
        "median_bars_per_day": med_bars, "p_jump_bar_fixed": float(p_bar),
        "mark_mean": float(marks.JumpMark_y.mean()),
        "mark_sd": float(marks.JumpMark_y.std(ddof=1)),
        "mark_q01": float(marks.JumpMark_y.quantile(.01)),
        "mark_q05": float(marks.JumpMark_y.quantile(.05)),
        "mark_median": float(marks.JumpMark_y.median()),
        "mark_q95": float(marks.JumpMark_y.quantile(.95)),
        "mark_q99": float(marks.JumpMark_y.quantile(.99)),
        "abs_z_proxy_median": float(marks.Abs_z_proxy.median()),
        "abs_z_proxy_q95": float(marks.Abs_z_proxy.quantile(.95)),
    }
    return marks, diag


def simulate_garch_t_jump(fit_t, n, rng, jump_marks, p_jump_bar, burn=900):
    p = fit_t["params"]
    N = n + burn
    z = _draw_z(rng, N, "t", float(p["nu"]))
    hit = rng.random(N) < float(p_jump_bar)
    marks = np.zeros(N, float)
    nh = int(hit.sum())
    pool = np.asarray(jump_marks, float)
    if nh:
        marks[hit] = rng.choice(pool, size=nh, replace=True)
    h = np.empty(N, float)
    y = np.empty(N, float)
    pers = float(p["alpha"] + p["beta"])
    h[0] = max(float(p["omega"]) / max(1.0 - pers, 1e-4), 1e-8)
    y[0] = math.sqrt(h[0]) * z[0] + marks[0]
    for t in range(1, N):
        h[t] = float(p["omega"]) + float(p["alpha"]) * y[t-1] * y[t-1] + float(p["beta"]) * h[t-1]
        if not np.isfinite(h[t]) or h[t] < 1e-10:
            h[t] = 1e-10
        y[t] = math.sqrt(h[t]) * z[t] + marks[t]
    return y[burn:] * float(fit_t["scale"]), hit[burn:]


def build_library(fit_t, marks, p_jump_bar, L, templates, minutes, pstar,
                  target_n=LIBRARY_N, force=False):
    model_name = "J2_Seasonal_GARCH_t_BNSanchored_Jump"
    cp = _cp("j2lib", minutes, L)
    if cp.exists() and not force:
        old = pd.read_pickle(cp)
        if len(old) >= 2 * target_n:
            return old
        rows = old.to_dict("records")
    else:
        rows = []
    have = {(r.get("Role"), int(r.get("Library_ID", -1))) for r in rows}
    tasks = [(role, i) for role in ("Treatment", "Placebo") for i in range(target_n)
             if (role, i+1) not in have]
    mark_pool = marks["JumpMark_y"].to_numpy(float)

    def one(role, i):
        rng = np.random.default_rng(RNG_BASE + _stable_int(model_name + role) * 1000 + minutes * 100 + L * 10 + i)
        for attempt in range(1, MAX_REJECTION_ATTEMPTS + 1):
            tpl = templates[int(rng.integers(0, len(templates)))]
            npre = len(tpl["sf_pre"]); nev = len(tpl["sf_event"]); npost = len(tpl["sf_post"])
            base, hits = simulate_garch_t_jump(fit_t, npre + nev + npost, rng, mark_pool, p_jump_bar)
            a = 0
            pre = base[a:a+npre] * tpl["sf_pre"]; a += npre
            ev = base[a:a+nev] * tpl["sf_event"]; a += nev
            post = base[a:a+npost] * tpl["sf_post"]
            zb, pb = bns_p_one_sided(ev, tpl["event_slots"], minutes)
            if not np.isfinite(pb):
                continue
            latent = int(np.sum(hits[npre:npre+nev]))
            qualifies = pb <= pstar
            if role == "Treatment" and (not qualifies or latent < 1):
                continue
            if role == "Placebo" and qualifies:
                continue
            f = _pair_features(pre, post)
            if f is None or f["R2_pre"] < DFA_R2_STAGE7 or f["R2_post"] < DFA_R2_STAGE7:
                continue
            f.update({
                "Role": role, "Library_ID": i+1, "Model": model_name,
                "Frequency_min": minutes, "Window_days": L,
                "BNS_Z_event": zb, "BNS_p_event": pb,
                "Latent_jump_count_event": latent, "Attempts": attempt,
            })
            return f
        return None

    for k in range(0, len(tasks), LIB_BATCH):
        batch = tasks[k:k+LIB_BATCH]
        rr = Parallel(n_jobs=min(N_JOBS, len(batch)), backend="loky")(
            delayed(one)(role, i) for role, i in batch
        )
        rows.extend([x for x in rr if x is not None])
        pd.DataFrame(rows).to_pickle(cp)
        print(f"    [J2 {minutes}m {L}d] {len(rows)}/{2*target_n} accepted", flush=True)
    return pd.DataFrame(rows)


def _nearest_lists(lib, target_D, target_R, k=K_NEAREST):
    lib = lib.reset_index(drop=True)
    D = lib["D_pre"].to_numpy(float); R = lib["log_PreRV"].to_numpy(float)
    target_D = np.asarray(target_D, float); target_R = np.asarray(target_R, float)
    ds = _robust_scale(np.r_[D, target_D]); rs = _robust_scale(np.r_[R, target_R])
    kk = min(k, len(lib)); out = []
    for d, r in zip(target_D, target_R):
        dist = np.abs(D-d)/ds + np.abs(R-r)/rs
        out.append(np.argpartition(dist, kk-1)[:kk])
    return out


def paired_tminusp_mc(lib, op, B, seed):
    lt = lib[lib.Role == "Treatment"].reset_index(drop=True)
    lp = lib[lib.Role == "Placebo"].reset_index(drop=True)
    tnear = _nearest_lists(lt, op.T_D_pre.to_numpy(float), op.T_log_PreRV.to_numpy(float))
    pnear = _nearest_lists(lp, op.C_D_pre.to_numpy(float), op.C_log_PreRV.to_numpy(float))
    rng = np.random.default_rng(seed)
    out = np.empty(B, float)
    for b in range(B):
        ti = np.array([rng.choice(ix) for ix in tnear], int)
        pi = np.array([rng.choice(ix) for ix in pnear], int)
        dt = lt.iloc[ti]["Delta_D"].to_numpy(float)
        dp = lp.iloc[pi]["Delta_D"].to_numpy(float)
        out[b] = float(np.mean(dt - dp))
    return out


def _two_sided_emp(sim, obs):
    x = np.asarray(sim, float); x = x[np.isfinite(x)]
    if not len(x): return np.nan
    c = float(np.median(x)); d = abs(float(obs) - c)
    return float((np.sum(np.abs(x-c) >= d) + 1) / (len(x) + 1))


def _summarize_fit(fit_t, markdiag, freq):
    row = {"Frequency_min": freq, "Model": "J2_Seasonal_GARCH_t_BNSanchored_Jump"}
    for k, v in fit_t.items():
        if k != "params": row[f"Continuous_{k}"] = v
    for k, v in fit_t["params"].items(): row[f"param_{k}"] = v
    for k, v in markdiag.items(): row[k] = v
    row["Combined_BIC_available"] = False
    row["Combined_BIC_note"] = "No: jump marks/intensity are BNS-anchored, not jointly ML-estimated; do not compare combined BIC to Stage7 models."
    return row


def main(stage5_excel=None, stage7_excel=None, stage7b_excel=None, force=False, smoke=False):
    global LIBRARY_N, OUTER_MC, RUN_TAG
    if smoke:
        LIBRARY_N = min(LIBRARY_N, 28)
        OUTER_MC = min(OUTER_MC, 140)

    s1 = load_checkpoint("stage1_data", strict=not smoke)
    stage5_excel = Path(stage5_excel) if stage5_excel else OUTPUT_DIR / "stage5_mechanism_v3_results.xlsx"
    stage7_excel = Path(stage7_excel) if stage7_excel else OUTPUT_DIR / "stage7" / "stage7_structural_discrimination_results.xlsx"
    stage7b_excel = Path(stage7b_excel) if stage7b_excel else OUTPUT_DIR / "stage7b" / "stage7b_jump_regime_results.xlsx"
    if not stage5_excel.exists(): raise FileNotFoundError(stage5_excel)

    sig = {
        "input_sha": s1.get("input_sha", "NA"), "stage5": _file_sig(stage5_excel),
        "stage7": _file_sig(stage7_excel) if stage7_excel.exists() else "MISSING",
        "stage7b": _file_sig(stage7b_excel) if stage7b_excel.exists() else "MISSING",
        "version": STAGE7C_VERSION, "lib": LIBRARY_N, "mc": OUTER_MC,
        "k": K_NEAREST, "maxreject": MAX_REJECTION_ATTEMPTS,
    }
    RUN_TAG = hashlib.sha256(json.dumps(sig, sort_keys=True).encode()).hexdigest()[:12]
    print("="*100)
    print(f"STAGE 7C | {STAGE7C_VERSION} | RUN_TAG={RUN_TAG}")
    print("Student-t background + BNS-anchored jumps; Mean_DeltaD and T-P 30d are primary families.")

    pairs = load_frozen_pairs(stage5_excel)
    bns = pd.read_excel(stage5_excel, sheet_name="BNS_Daily_Jump_Test")
    bns["Date"] = _normalize_dates(bns["Date"])
    realjump = bns[bns["Jump_BNS_FDR"].astype(bool) & pd.to_numeric(bns["p_BNS_one_sided"], errors="coerce").notna()].copy()
    if len(realjump) == 0: raise RuntimeError("No BNS-FDR jumps in Stage 5 v3")
    pstar = float(realjump["p_BNS_one_sided"].max())
    print(f"[BNS] real jumps={len(realjump)}; p*={pstar:.8g}")

    fit_rows = []; mark_rows = []; libdiag = []; joint_rows = []; comp_rows = []; pair_rows = []; obs_rows = []

    for freq in FREQUENCIES:
        df, daily = _make_frequency_data(s1, freq)
        factors, _ = estimate_seasonality(df)
        obs, cache = observed_pairs_frequency(pairs, df, daily, freq)
        x = deseasonalized_returns(df, factors, True)

        fitcp = CPDIR / f"garchtfit_{RUN_TAG}_{freq}m.json"
        if fitcp.exists() and not force:
            fit_t = json.loads(fitcp.read_text(encoding="utf-8"))
        else:
            print(f"[FIT GARCH-t] {freq}m n={len(x):,}", flush=True)
            fit_t = fit_model(x, "garch_t")
            fit_t.update({"model": "Continuous_Seasonal_GARCH_t", "frequency": freq, "seasonal": True})
            fitcp.write_text(json.dumps(fit_t, indent=2), encoding="utf-8")

        marks, markdiag = calibrate_bns_anchored_jump_marks(df, factors, fit_t, bns, realjump)
        marks["Frequency_min"] = freq
        mark_rows.append(marks)
        fit_rows.append(_summarize_fit(fit_t, markdiag, freq))
        print(f"[JUMP CAL] {freq}m marks={len(marks)} p_bar={markdiag['p_jump_bar_fixed']:.7g} nu={fit_t['params']['nu']:.3f}")

        for L in HORIZONS:
            op = obs[obs.Window_days == L].copy().reset_index(drop=True)
            if len(op) < 4:
                print(f"[WARN] {freq}m {L}d: too few real pairs ({len(op)}); skipped")
                continue
            templates = _template_pool_jump(cache, pairs, L, factors)
            lib = build_library(fit_t, marks, markdiag["p_jump_bar_fixed"], L, templates, freq, pstar, LIBRARY_N, force)
            for role in ("Treatment", "Placebo"):
                lr = lib[lib.Role == role]
                libdiag.append({
                    "Frequency_min": freq, "Window_days": L, "Role": role, "N": len(lr),
                    "Mean_attempts": float(lr.Attempts.mean()) if len(lr) else np.nan,
                    "Acceptance_rate_proxy": float(len(lr)/max(lr.Attempts.sum(), 1)) if len(lr) else np.nan,
                    "Mean_BNS_p": float(lr.BNS_p_event.mean()) if len(lr) else np.nan,
                    "Mean_latent_jumps_event": float(lr.Latent_jump_count_event.mean()) if len(lr) else np.nan,
                })

                precol = "T_D_pre" if role == "Treatment" else "C_D_pre"
                postcol = "T_D_post" if role == "Treatment" else "C_D_post"
                rvcol = "T_log_PreRV" if role == "Treatment" else "C_log_PreRV"
                pre = op[precol].to_numpy(float); post = op[postcol].to_numpy(float); rv = op[rvcol].to_numpy(float)
                obsgeom = _geometry_from_arrays(pre, post)
                sublib = lib[lib.Role == role].reset_index(drop=True)
                mc = geometry_mc(sublib, pre, rv, OUTER_MC, RNG_BASE + _stable_int(role) + freq*1000 + L)
                jd, cd = joint_discrepancy(mc, obsgeom)
                joint_rows.append({"Model":"J2_Seasonal_GARCH_t_BNSanchored_Jump", "Frequency_min":freq,
                                   "Window_days":L, "Group":role, "N_real":len(pre), **obsgeom, **jd})
                cd["Model"] = "J2_Seasonal_GARCH_t_BNSanchored_Jump"; cd["Frequency_min"] = freq
                cd["Window_days"] = L; cd["Group"] = role; comp_rows.append(cd)

            obs_tp = float(np.mean(op.T_Delta.to_numpy(float) - op.C_Delta.to_numpy(float)))
            sim_tp = paired_tminusp_mc(lib, op, OUTER_MC, RNG_BASE + 90_000 + freq*1000 + L)
            pair_rows.append({
                "Frequency_min": freq, "Window_days": L, "N_pairs": len(op),
                "Observed_Mean_TminusP_DeltaD": obs_tp,
                "Null_mean": float(np.mean(sim_tp)), "Null_median": float(np.median(sim_tp)),
                "Null_q025": float(np.quantile(sim_tp, .025)), "Null_q975": float(np.quantile(sim_tp, .975)),
                "p_two_sided": _two_sided_emp(sim_tp, obs_tp),
                "Observed_negative": bool(obs_tp < 0),
            })
            obs_rows.append({"Frequency_min":freq,"Window_days":L,"N_pairs":len(op),
                             "Mean_Delta_T":float(op.T_Delta.mean()),"Mean_Delta_P":float(op.C_Delta.mean()),
                             "Mean_TminusP":obs_tp})

    jtests = pd.DataFrame(joint_rows)
    jcomps = pd.concat(comp_rows, ignore_index=True) if comp_rows else pd.DataFrame()
    paired = pd.DataFrame(pair_rows)

    # Joint FDR: 8 cells per statistic (2f*2h*2groups).
    for pc, qc in [("p_Mahalanobis", "q_Mahalanobis_BH"), ("p_MaxZ", "q_MaxZ_BH")]:
        if len(jtests): jtests[qc] = multipletests(jtests[pc].to_numpy(float), method="fdr_bh")[1]
    if len(jtests):
        jtests["Adequate_after_BH"] = ((jtests.q_Mahalanobis_BH >= ALPHA) &
                                         (jtests.q_MaxZ_BH >= ALPHA) &
                                         (jtests.N_components_inside95 >= 3))

    # Components: frozen families.
    if len(jcomps):
        jcomps["q_component_BH"] = np.nan; jcomps["FDR_Family"] = "exploratory_unadjusted"
        m1 = (jcomps.Group=="Treatment") & (jcomps.Window_days==30) & (jcomps.Component=="Mean_DeltaD")
        if m1.sum():
            jcomps.loc[m1, "q_component_BH"] = multipletests(jcomps.loc[m1,"p_two_sided"], method="fdr_bh")[1]
            jcomps.loc[m1, "FDR_Family"] = "Primary_Treatment_MeanDeltaD_30d_15m30m"
        m2 = (jcomps.Group=="Placebo") & (jcomps.Window_days==30) & (jcomps.Component=="Mean_DeltaD")
        if m2.sum():
            jcomps.loc[m2, "q_component_BH"] = multipletests(jcomps.loc[m2,"p_two_sided"], method="fdr_bh")[1]
            jcomps.loc[m2, "FDR_Family"] = "Diagnostic_Placebo_MeanDeltaD_30d_15m30m"
        m3 = (jcomps.Group=="Treatment") & (jcomps.Window_days==30) & (jcomps.Frequency_min==30) & jcomps.Component.isin(["Oldham","Log_SDratio"])
        if m3.sum():
            jcomps.loc[m3, "q_component_BH"] = multipletests(jcomps.loc[m3,"p_two_sided"], method="fdr_bh")[1]
            jcomps.loc[m3, "FDR_Family"] = "Secondary_30m_Oldham_SDratio"

    # Co-primary T-P: 2 frequencies, 30d.
    paired["q_BH"] = np.nan; paired["FDR_Family"] = "Secondary_60d"
    mp = paired.Window_days == 30
    if mp.sum():
        paired.loc[mp, "q_BH"] = multipletests(paired.loc[mp,"p_two_sided"], method="fdr_bh")[1]
        paired.loc[mp, "FDR_Family"] = "Primary_TminusP_MeanDeltaD_30d_15m30m"

    dec = []
    tj = jtests[(jtests.Window_days==30)&(jtests.Group=="Treatment")]
    dec.append({"Decision":"J2_explains_Treatment_joint_30d_both_freq",
                "Result":bool(len(tj)==2 and tj.Adequate_after_BH.all()),
                "Detail":"True only if the joint Treatment 30d geometry is adequate after BH at 15m and 30m."})
    ft = jcomps[(jcomps.FDR_Family=="Primary_Treatment_MeanDeltaD_30d_15m30m")].copy()
    freq_t = bool(len(ft)==2 and (ft.Observed < ft.Null_mean).all() and (ft.q_component_BH < ALPHA).all())
    dec.append({"Decision":"Treatment_MeanDeltaD_negative_residual_both_freq","Result":freq_t,
                "Detail":"Observed<Null_mean and q<0.05 at 15m and 30m."})
    fp = paired[paired.Window_days==30].copy()
    freq_tp = bool(len(fp)==2 and fp.Observed_negative.all() and (fp.q_BH < ALPHA).all())
    dec.append({"Decision":"TminusP_specific_negative_residual_both_freq","Result":freq_tp,
                "Detail":"Co-primary: Mean(Delta_T-Delta_P)<0 (Observed_negative means obs_tp<0, not obs_tp<Null_mean) and q<0.05 at 15m and 30m."})
    dec.append({"Decision":"Strong_postshock_candidate","Result":bool(freq_t and freq_tp),
                "Detail":"Requires both a Treatment residual and a T-P residual, each robust across frequencies."})
    p30 = jcomps[(jcomps.FDR_Family=="Diagnostic_Placebo_MeanDeltaD_30d_15m30m") & (jcomps.Frequency_min==30)]
    dec.append({"Decision":"Placebo_MeanDeltaD_30m_not_rejected","Result":bool(len(p30)==1 and float(p30.p_two_sided.iloc[0])>=ALPHA),
                "Detail":"Diagnostic; non-rejection does not prove equivalence."})

    outfile = OUTDIR / "stage7c_garcht_jump_results.xlsx"
    with pd.ExcelWriter(outfile, engine="openpyxl") as w:
        pd.DataFrame(fit_rows).to_excel(w, sheet_name="Continuous_GARCHt_Fits", index=False)
        pd.concat(mark_rows, ignore_index=True).to_excel(w, sheet_name="BNS_Anchored_Jump_Marks", index=False)
        pd.DataFrame(libdiag).to_excel(w, sheet_name="Library_Diagnostics", index=False)
        pd.DataFrame(obs_rows).to_excel(w, sheet_name="Observed_TP_30_60", index=False)
        jtests.to_excel(w, sheet_name="Joint_Tests_FDR", index=False)
        jcomps.to_excel(w, sheet_name="Component_Tests_FDR", index=False)
        paired.to_excel(w, sheet_name="Paired_TminusP_FDR", index=False)
        pd.DataFrame(dec).to_excel(w, sheet_name="Preregistered_Decisions", index=False)
        pd.DataFrame([
            {"Parameter":"STAGE7C_VERSION","Value":STAGE7C_VERSION},
            {"Parameter":"RUN_TAG","Value":RUN_TAG},
            {"Parameter":"LIBRARY_N_per_role","Value":LIBRARY_N},
            {"Parameter":"OUTER_MC","Value":OUTER_MC},
            {"Parameter":"BNS_pstar_real","Value":pstar},
            {"Parameter":"Primary_endpoint_1","Value":"Treatment Mean_DeltaD 30d across 15m/30m"},
            {"Parameter":"Primary_endpoint_2","Value":"Mean(Delta_T-Delta_P) 30d across 15m/30m"},
            {"Parameter":"Continuous_DGP","Value":"Seasonal GARCH(1,1)-t ML"},
            {"Parameter":"Jump_intensity","Value":"fixed to BNS-FDR daily rate transformed to per-bar Bernoulli intensity"},
            {"Parameter":"Jump_marks","Value":"empirical signed deseasonalized/scaled return at max |GARCH-t standardized residual| bar on each BNS-FDR day; event-day only"},
            {"Parameter":"Combined_BIC","Value":"Not defined: jump mechanism is BNS-anchored rather than joint ML"},
            {"Parameter":"Strong_claim_rule","Value":"Treatment MeanDeltaD residual AND paired T-P residual, both negative/significant after BH at 15m and 30m"},
        ]).to_excel(w, sheet_name="Stage7C_Parameters", index=False)
        if stage7b_excel.exists():
            try: pd.read_excel(stage7b_excel, sheet_name="Preregistered_Decisions").to_excel(w, sheet_name="Prior_7B_Decisions", index=False)
            except Exception: pass
        if stage7_excel.exists():
            try: pd.read_excel(stage7_excel, sheet_name="Preregistered_Decisions").to_excel(w, sheet_name="Prior_7_Decisions", index=False)
            except Exception: pass

    print(f"{outfile}")
    print(pd.DataFrame(dec).to_string(index=False))
    return outfile


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage5-excel")
    ap.add_argument("--stage7-excel")
    ap.add_argument("--stage7b-excel")
    ap.add_argument("--force", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    main(a.stage5_excel, a.stage7_excel, a.stage7b_excel, a.force, a.smoke)


In [ ]:
%%writefile /content/xauusd_scientific_pipeline/stage8_temporal_ablation.py
# -*- coding: utf-8 -*-
"""Stage 8: ablations of temporal organization."""
from __future__ import annotations

from config import *
import stage7_structural_discrimination as e7
from stage7_structural_discrimination import (
    _make_frequency_data, estimate_seasonality, deseasonalized_returns,
    fit_model, simulate_model, _slot_minutes, _garch_h, _draw_z,
    _geometry_from_arrays, _file_sig
)

import argparse
import hashlib
import json
import math
import re
import os
import platform
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests

STAGE8_VERSION = "XAUUSD_STAGE8_TEMPORAL_ABLATION_v1_2026-09-12"
OUTDIR = OUTPUT_DIR / "stage8"
CPDIR = CHECKPOINT_DIR / "stage8"
OUTDIR.mkdir(parents=True, exist_ok=True)
CPDIR.mkdir(parents=True, exist_ok=True)

FREQUENCIES = (15, 30)
PRIMARY_HORIZON = 30
ALPHA = 0.05
REPS = int(os.getenv("XAUUSD_STAGE8_REPS", "199"))
BATCH = int(os.getenv("XAUUSD_STAGE8_BATCH", "4"))
MIN_PAIR_FRACTION = float(os.getenv("XAUUSD_STAGE8_MIN_PAIR_FRACTION", "0.80"))
RNG_BASE = RANDOM_SEED + 880_000
RUN_TAG = "UNSET"

MECHANISMS = (
    "S1_Shuffle_All",
    "S2_IAAFT",
    "S3_SignShuffle_Slot",
    "S4_MagnitudeShuffle_Slot",
    "S5_DayBlock_Shuffle",
    "P1_Seasonal_GARCH_t",
    "P2_Seasonal_GARCH_t_Jump",
    "P3_Seasonal_FIGARCH_t",
)
PRIMARY_ENDPOINTS = ("MFDFA_h_range", "Pooled_MeanDeltaD_30d")


def _stable_int(text: str) -> int:
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:12], 16) % 1_000_000_007


def _cp(mech, freq):
    safe = mech.replace("/", "_")
    return CPDIR / f"stage8_{RUN_TAG}_{safe}_{freq}m.pkl"


def _normalize_dates(s):
    values = pd.Series(s, copy=True)
    if pd.api.types.is_datetime64_any_dtype(values):
        return pd.to_datetime(values, errors="coerce").dt.normalize()
    numeric = pd.to_numeric(values, errors="coerce")
    numeric_values = values.map(lambda v: isinstance(v, (int, float, np.integer, np.floating))
                                and not isinstance(v, (bool, np.bool_)))
    numeric_strings = values.map(lambda v: isinstance(v, str) and
                                 bool(re.fullmatch(r"\d{4,6}(?:\.\d+)?", v.strip())))
    excel_serial = numeric.notna() & numeric.between(10000, 90000) & (numeric_values | numeric_strings)
    ambiguous_numeric = numeric.notna() & numeric_values & ~excel_serial
    if ambiguous_numeric.any():
        raise ValueError("Date column contains unsupported numeric dates; expected ISO/datetime or Excel serial 10000..90000")
    parsed = pd.to_datetime(values.mask(excel_serial), errors="coerce")
    if excel_serial.any():
        parsed.loc[excel_serial] = pd.to_datetime(
            numeric.loc[excel_serial], unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(parsed).dt.normalize()


def load_pairs(path):
    p = pd.read_excel(path, sheet_name="EventStudy_Pairs_DiD")
    need = {"Window_pre_post_days", "Treatment_Date", "Placebo_Date"}
    miss = need - set(p.columns)
    if miss:
        raise RuntimeError(f"Missing columns in EventStudy_Pairs_DiD: {sorted(miss)}")
    p["Treatment_Date"] = _normalize_dates(p["Treatment_Date"])
    p["Placebo_Date"] = _normalize_dates(p["Placebo_Date"])
    return p[p["Window_pre_post_days"] == PRIMARY_HORIZON].dropna(subset=["Treatment_Date", "Placebo_Date"]).copy()


def build_window_specs(valid_index, pairs, L=PRIMARY_HORIZON):
    idx = pd.DatetimeIndex(valid_index)
    days = idx.normalize()
    unique_days = pd.DatetimeIndex(sorted(pd.unique(days)))
    starts = {}; ends = {}
    for d in unique_days:
        loc = np.flatnonzero(days == d)
        starts[pd.Timestamp(d)] = int(loc[0]); ends[pd.Timestamp(d)] = int(loc[-1] + 1)
    pos = {pd.Timestamp(d): i for i, d in enumerate(unique_days)}

    def one(d):
        d = pd.Timestamp(d).normalize()
        if d not in pos: return None
        j = pos[d]
        if j < L or j + L >= len(unique_days): return None
        pre_days = unique_days[j-L:j]
        post_days = unique_days[j+1:j+1+L]
        return (starts[pd.Timestamp(pre_days[0])], ends[pd.Timestamp(pre_days[-1])],
                starts[pd.Timestamp(post_days[0])], ends[pd.Timestamp(post_days[-1])])

    specs=[]
    for _, r in pairs.iterrows():
        t=one(r["Treatment_Date"]); c=one(r["Placebo_Date"])
        if t is not None and c is not None:
            specs.append({"Treatment_Date":pd.Timestamp(r["Treatment_Date"]),
                          "Placebo_Date":pd.Timestamp(r["Placebo_Date"]),
                          "T":t, "C":c})
    return specs


def _event_D(x, spec):
    a,b,c,d = spec
    hp,rp,_,_ = dfa(x[a:b]); hq,rq,_,_ = dfa(x[c:d])
    if not all(np.isfinite([hp,rp,hq,rq])): return None
    if rp < DFA_R2_MIN or rq < DFA_R2_MIN: return None
    dp=abs(float(hp)-.5); dq=abs(float(hq)-.5)
    return dp,dq,dq-dp,float(rp),float(rq)


def event_statistics(x, specs, min_pairs):
    Tpre=[]; Tpost=[]; Tdel=[]; Cpre=[]; Cpost=[]; Cdel=[]
    for s in specs:
        t=_event_D(x,s["T"]); c=_event_D(x,s["C"])
        if t is None or c is None: continue
        Tpre.append(t[0]); Tpost.append(t[1]); Tdel.append(t[2])
        Cpre.append(c[0]); Cpost.append(c[1]); Cdel.append(c[2])
    n=len(Tdel)
    if n < min_pairs:
        return None
    Tpre=np.asarray(Tpre); Tpost=np.asarray(Tpost); Tdel=np.asarray(Tdel)
    Cpre=np.asarray(Cpre); Cpost=np.asarray(Cpost); Cdel=np.asarray(Cdel)
    pre=np.r_[Tpre,Cpre]; post=np.r_[Tpost,Cpost]
    geom=_geometry_from_arrays(pre,post)
    return {
        "N_pairs":n,
        "MeanDelta_Treatment":float(np.mean(Tdel)),
        "MeanDelta_Placebo":float(np.mean(Cdel)),
        "Mean_TminusP":float(np.mean(Tdel-Cdel)),
        "Pooled_MeanDeltaD_30d":float(np.mean(np.r_[Tdel,Cdel])),
        "Pooled_Oldham_30d":float(geom["Oldham"]),
        "Pooled_Log_SDratio_30d":float(geom["Log_SDratio"]),
        "Pooled_Corr_Dpre_DeltaD_30d":float(geom["Corr_Dpre_DeltaD"]),
    }


def mf_statistics(x):
    mf=mf_dfa(x)
    if mf is None: return None
    hr=mfdfa_range(mf)
    aa=mf["alpha"].dropna().to_numpy(float) if "alpha" in mf else np.array([])
    da=float(np.max(aa)-np.min(aa)) if len(aa)>=2 else np.nan
    return {"MFDFA_h_range":float(hr), "MFDFA_Delta_alpha":da}, mf


def all_statistics(x, specs, min_pairs):
    ev=event_statistics(x,specs,min_pairs)
    if ev is None: return None
    mf,_=mf_statistics(x)
    if mf is None: return None
    out={**mf,**ev}
    # Simple magnitude/sign dependence diagnostics.
    if len(x)>2:
        out["AbsRet_ACF1"] = float(np.corrcoef(np.abs(x[:-1]),np.abs(x[1:]))[0,1])
        out["Ret_ACF1"] = float(np.corrcoef(x[:-1],x[1:])[0,1])
    else:
        out["AbsRet_ACF1"]=np.nan; out["Ret_ACF1"]=np.nan
    return out


def preprocess_structure(df):
    r=pd.to_numeric(df["LogRet"],errors="coerce")
    m=r.notna().to_numpy()
    idx=pd.DatetimeIndex(df.index[m]); x=r[m].to_numpy(float)
    slots=_slot_minutes(idx); days=idx.normalize()

    slot_groups={int(s):np.flatnonzero(slots==s) for s in np.unique(slots)}
    day_positions={}
    day_slots={}
    for d in pd.unique(days):
        p=np.flatnonzero(days==d)
        day_positions[pd.Timestamp(d)] = p
        day_slots[pd.Timestamp(d)] = tuple(slots[p].tolist())
    sig_groups={}
    for d,sig in day_slots.items(): sig_groups.setdefault(sig,[]).append(d)
    movable=sum(len(day_positions[d]) for ds in sig_groups.values() if len(ds)>=2 for d in ds)
    return {"x":x,"idx":idx,"slots":slots,"slot_groups":slot_groups,
            "day_positions":day_positions,"sig_groups":sig_groups,
            "dayblock_movable_fraction":movable/max(len(x),1)}


def s_shuffle_all(ctx,rng):
    return rng.permutation(ctx["x"])


def s_iaaft(ctx,rng):
    y,err=iaaft_surrogate(ctx["x"],rng)
    return y,{"IAAFT_error_spectral_rel_L2":float(err)}


def s_signshuffle_slot(ctx,rng):
    x=ctx["x"]; out=np.empty_like(x); mag=np.abs(x); sg=np.sign(x)
    for p in ctx["slot_groups"].values():
        out[p]=mag[p]*rng.permutation(sg[p])
    return out


def s_magnitude_shuffle_slot(ctx,rng):
    x=ctx["x"]; out=np.empty_like(x); mag=np.abs(x); sg=np.sign(x)
    for p in ctx["slot_groups"].values():
        out[p]=sg[p]*rng.permutation(mag[p])
    return out


def s_dayblock(ctx,rng):
    x=ctx["x"]; out=x.copy()
    for ds in ctx["sig_groups"].values():
        if len(ds)<2: continue
        src=list(ds); rng.shuffle(src)
        for target,source in zip(ds,src):
            tp=ctx["day_positions"][target]; sp=ctx["day_positions"][source]
            out[tp]=x[sp]
    return out


def _full_deseasonalized_series(df,factors):
    r=pd.to_numeric(df["LogRet"],errors="coerce"); m=r.notna()
    idx=pd.DatetimeIndex(df.index[m]); x=r[m].to_numpy(float)
    sf=np.array([factors.get(int(s),1.0) for s in _slot_minutes(idx)],float)
    x=x/sf; x=x-np.mean(x)
    return pd.Series(x,index=idx)


def calibrate_jump_marks(df,factors,fit_t,bns,realjump,smoke=False):
    s=_full_deseasonalized_series(df,factors)
    y=s.to_numpy(float)/float(fit_t["scale"])
    p=fit_t["params"]
    h=_garch_h(y,float(p["omega"]),float(p["alpha"]),float(p["beta"]))
    z=y/np.sqrt(np.maximum(h,1e-12))
    tab=pd.DataFrame({"DateTime":s.index,"y":y,"z":z}); tab["Date"]=tab.DateTime.dt.normalize()
    dates=set(pd.DatetimeIndex(realjump["Date"]).normalize())
    marks=[]
    for d,g in tab[tab.Date.isin(dates)].groupby("Date"):
        if len(g):
            j=int(np.nanargmax(np.abs(g.z.to_numpy(float))))
            marks.append(float(g.iloc[j].y))
    min_marks=3 if smoke else 30
    if len(marks)<min_marks: raise RuntimeError(f"Only {len(marks)} BNS marks; minimum={min_marks}")
    valid=bns[pd.to_numeric(bns["p_BNS_one_sided"],errors="coerce").notna()]
    qday=len(realjump)/max(len(valid),1)
    bars=pd.Series(1,index=s.index).groupby(s.index.normalize()).sum()
    med=float(bars.median())
    pbar=1-(1-qday)**(1/max(med,1.0))
    return np.asarray(marks,float),float(pbar),{"N_marks":len(marks),"BNS_day_rate":qday,"median_bars_day":med,"p_jump_bar":pbar}


def simulate_garch_t_jump(fit,n,rng,marks,pbar,burn=900):
    p=fit["params"]; N=n+burn
    z=_draw_z(rng,N,"t",float(p["nu"]))
    hit=rng.random(N)<pbar; jm=np.zeros(N,float)
    if hit.any(): jm[hit]=rng.choice(marks,size=int(hit.sum()),replace=True)
    h=np.empty(N,float); y=np.empty(N,float)
    pers=float(p["alpha"]+p["beta"])
    h[0]=max(float(p["omega"])/max(1-pers,1e-4),1e-8)
    y[0]=math.sqrt(h[0])*z[0]+jm[0]
    for t in range(1,N):
        h[t]=float(p["omega"])+float(p["alpha"])*y[t-1]**2+float(p["beta"])*h[t-1]
        h[t]=max(h[t],1e-10) if np.isfinite(h[t]) else 1e-10
        y[t]=math.sqrt(h[t])*z[t]+jm[t]
    return y[burn:]*float(fit["scale"])


def _two_sided_emp(vals,obs):
    a=np.asarray(vals,float); a=a[np.isfinite(a)]
    if not len(a): return np.nan
    plo=(np.sum(a<=obs)+1)/(len(a)+1); phi=(np.sum(a>=obs)+1)/(len(a)+1)
    return float(min(1.0,2*min(plo,phi)))


def mechanism_one(mech,rep,ctx,specs,min_pairs,param,rngseed):
    rng=np.random.default_rng(rngseed)
    diag={}
    if mech=="S1_Shuffle_All": y=s_shuffle_all(ctx,rng)
    elif mech=="S2_IAAFT": y,diag=s_iaaft(ctx,rng)
    elif mech=="S3_SignShuffle_Slot": y=s_signshuffle_slot(ctx,rng)
    elif mech=="S4_MagnitudeShuffle_Slot": y=s_magnitude_shuffle_slot(ctx,rng)
    elif mech=="S5_DayBlock_Shuffle": y=s_dayblock(ctx,rng)
    elif mech=="P1_Seasonal_GARCH_t":
        y=simulate_model(param["garch_t"],len(ctx["x"]),rng)*param["sf"]
    elif mech=="P2_Seasonal_GARCH_t_Jump":
        y=simulate_garch_t_jump(param["garch_t"],len(ctx["x"]),rng,param["marks"],param["pbar"])*param["sf"]
    elif mech=="P3_Seasonal_FIGARCH_t":
        y=simulate_model(param["figarch_t"],len(ctx["x"]),rng)*param["sf"]
    else: raise ValueError(mech)
    st=all_statistics(np.asarray(y,float),specs,min_pairs)
    if st is None: return None
    return {"Mechanism":mech,"Replicate":rep+1,**st,**diag}


def run_mechanism(mech,freq,ctx,specs,min_pairs,param,reps,force=False):
    cp=_cp(mech,freq)
    done=pd.read_pickle(cp) if cp.exists() and not force else pd.DataFrame()
    got=set(done.get("Replicate",pd.Series(dtype=int)).astype(int).tolist())
    missing=[i for i in range(reps) if i+1 not in got]
    print(f"  [{mech} {freq}m] completed={len(got)}/{reps}; missing={len(missing)}",flush=True)
    for k in range(0,len(missing),BATCH):
        ids=missing[k:k+BATCH]
        rows=Parallel(n_jobs=min(N_JOBS,len(ids)),backend="loky")(
            delayed(mechanism_one)(mech,i,ctx,specs,min_pairs,param,
                RNG_BASE+freq*1_000_000+_stable_int(mech)+i) for i in ids)
        rows=[r for r in rows if r is not None]
        if rows:
            done=pd.concat([done,pd.DataFrame(rows)],ignore_index=True)
            done=done.drop_duplicates("Replicate",keep="last").sort_values("Replicate")
            pd.to_pickle(done,cp)
        print(f"    checkpoint={len(done)}/{reps}",flush=True)
    return done.reset_index(drop=True)


def summarize_tests(original, surrogates):
    rows=[]
    endpoints=["MFDFA_h_range","Pooled_MeanDeltaD_30d","MFDFA_Delta_alpha",
               "Pooled_Oldham_30d","Pooled_Log_SDratio_30d","Mean_TminusP"]
    for (mech,freq),g in surrogates.groupby(["Mechanism","Frequency_min"]):
        oo=original[original.Frequency_min==freq].iloc[0]
        for ep in endpoints:
            vals=pd.to_numeric(g[ep],errors="coerce").dropna().to_numpy(float)
            obs=float(oo[ep]);
            if not len(vals): continue
            rows.append({
                "Mechanism":mech,"Frequency_min":freq,"Endpoint":ep,"N_surrogates":len(vals),
                "Observed":obs,"Null_mean":float(np.mean(vals)),"Null_median":float(np.median(vals)),
                "Null_q025":float(np.quantile(vals,.025)),"Null_q975":float(np.quantile(vals,.975)),
                "Observed_percentile":float((np.sum(vals<=obs)+0.5)/(len(vals)+1)),
                "p_two_sided":_two_sided_emp(vals,obs),
                "p_lower":float((np.sum(vals<=obs)+1)/(len(vals)+1)),
                "p_upper":float((np.sum(vals>=obs)+1)/(len(vals)+1)),
                "Inside95":bool(np.quantile(vals,.025)<=obs<=np.quantile(vals,.975)),
                "Z_vs_null":float((obs-np.mean(vals))/np.std(vals,ddof=1)) if np.std(vals,ddof=1)>0 else np.nan,
            })
    tests=pd.DataFrame(rows); tests["q_BH"]=np.nan; tests["FDR_Family"]="secondary_descriptive"
    for ep,fam in [("MFDFA_h_range","Primary_MFDFA_h_range_all_mechanisms_freqs"),
                   ("Pooled_MeanDeltaD_30d","Primary_Pooled_MeanDeltaD_all_mechanisms_freqs")]:
        m=tests.Endpoint.eq(ep)
        if m.any():
            tests.loc[m,"q_BH"]=multipletests(tests.loc[m,"p_two_sided"].to_numpy(float),method="fdr_bh")[1]
            tests.loc[m,"FDR_Family"]=fam
    return tests


def adjudicate(tests):
    rows=[]; compat={}
    for mech in MECHANISMS:
        gm=tests[(tests.Mechanism==mech)&tests.Endpoint.isin(PRIMARY_ENDPOINTS)]
        ok=True; reasons=[]
        for freq in FREQUENCIES:
            for ep in PRIMARY_ENDPOINTS:
                z=gm[(gm.Frequency_min==freq)&(gm.Endpoint==ep)]
                if len(z)!=1:
                    ok=False; reasons.append(f"missing {freq}m/{ep}"); continue
                r=z.iloc[0]
                if not (bool(r.Inside95) and float(r.q_BH)>=ALPHA):
                    ok=False; reasons.append(f"reject {freq}m/{ep}")
        compat[mech]=ok
        rows.append({"Decision":f"Compatible_primary::{mech}","Result":bool(ok),
                     "Detail":"; ".join(reasons) if reasons else "Original within 95% and q>=0.05 for both endpoints at 15m and 30m; compatibility, not equivalence."})

    iaaft=compat.get("S2_IAAFT",False); sign=compat.get("S3_SignShuffle_Slot",False)
    mag=compat.get("S4_MagnitudeShuffle_Slot",False); day=compat.get("S5_DayBlock_Shuffle",False)
    g=compat.get("P1_Seasonal_GARCH_t",False); gj=compat.get("P2_Seasonal_GARCH_t_Jump",False); fig=compat.get("P3_Seasonal_FIGARCH_t",False)
    rows += [
        {"Decision":"Linear_marginal_IAAFT_sufficient","Result":iaaft,
         "Detail":"True means only compatibility with both endpoints/frequencies; it does not prove causal sufficiency."},
        {"Decision":"Magnitude_path_with_randomized_signs_compatible","Result":sign,
         "Detail":"If True and IAAFT=False, points to magnitude/volatility structure not captured by linear dependence."},
        {"Decision":"Magnitude_clustering_candidate","Result":bool(sign and not mag),
         "Detail":"SignShuffle preserves |r_t|; MagnitudeShuffle destroys its ordering. A True pattern suggests magnitude order/clustering is necessary among these ablations."},
        {"Decision":"Interday_order_candidate","Result":bool(sign and not day),
         "Detail":"DayBlock preserves intraday structure but disrupts day order. A True pattern suggests interday organization of magnitudes."},
        {"Decision":"Standard_GARCHt_compatible","Result":g,"Detail":"Conditional volatility + Student-t benchmark."},
        {"Decision":"GARCHt_jump_compatible","Result":gj,"Detail":"Conditional volatility + Student-t + BNS-anchored jumps benchmark."},
        {"Decision":"FIGARCHt_compatible","Result":fig,"Detail":"Long-memory volatility + Student-t benchmark."},
        {"Decision":"No_tested_mechanism_jointly_compatible","Result":bool(not any(compat.values())),
         "Detail":"If True, residual structure remains for the tested endpoints; it does not automatically identify its mechanism."},
    ]
    return pd.DataFrame(rows)


def _fitrow(name,freq,fit):
    r={"Model":name,"Frequency_min":freq,"Converged":fit.get("converged"),"N_fit":fit.get("n_fit"),
       "AIC":fit.get("AIC"),"BIC":fit.get("BIC"),"Scale":fit.get("scale")}
    r.update({f"param_{k}":v for k,v in fit.get("params",{}).items()}); return r


def main(stage5_excel=None,stage7c_excel=None,force=False,smoke=False):
    global REPS,RUN_TAG
    if smoke: REPS=min(REPS,3)
    s1=load_checkpoint("stage1_data",strict=not smoke)
    stage5_excel=Path(stage5_excel) if stage5_excel else OUTPUT_DIR/"stage5_mechanism_v3_results.xlsx"
    if not stage5_excel.exists(): raise FileNotFoundError(stage5_excel)
    stage7c_excel=Path(stage7c_excel) if stage7c_excel else OUTPUT_DIR/"stage7c"/"stage7c_garcht_jump_results.xlsx"
    sig={"version":STAGE8_VERSION,"input_sha":s1.get("input_sha","NA"),"stage5":_file_sig(stage5_excel),
         "stage7c":_file_sig(stage7c_excel) if stage7c_excel.exists() else "MISSING",
         "reps":REPS,"horizon":PRIMARY_HORIZON,"q":MF_Q_VALUES,"iaaft_iter":IAAFT_MAX_ITER}
    RUN_TAG=hashlib.sha256(json.dumps(sig,sort_keys=True,default=str).encode()).hexdigest()[:12]
    print("="*100); print(f"STAGE 8 | {STAGE8_VERSION} | RUN_TAG={RUN_TAG} | REPS={REPS}")
    print("Temporal ablation; endpoints: MF-DFA h-range + Pooled MeanDeltaD 30d; 15m/30m.")

    pairs=load_pairs(stage5_excel)
    bns=pd.read_excel(stage5_excel,sheet_name="BNS_Daily_Jump_Test"); bns["Date"]=_normalize_dates(bns["Date"])
    realjump=bns[bns["Jump_BNS_FDR"].astype(bool) & pd.to_numeric(bns["p_BNS_one_sided"],errors="coerce").notna()].copy()
    if len(realjump)==0: raise RuntimeError("No BNS-FDR jumps available to calibrate P2")

    original_rows=[]; surrogate_rows=[]; fit_rows=[]; diag_rows=[]; mf_orig_sheets={}
    contexts={}
    for freq in FREQUENCIES:
        df,daily=_make_frequency_data(s1,freq)
        ctx=preprocess_structure(df); specs=build_window_specs(ctx["idx"],pairs,PRIMARY_HORIZON)
        min_pairs=max(3 if smoke else 10,int(math.ceil(MIN_PAIR_FRACTION*len(specs))))
        ost=all_statistics(ctx["x"],specs,min_pairs)
        if ost is None: raise RuntimeError(f"Original {freq}m produced no valid statistics")
        original_rows.append({"Frequency_min":freq,"N_specs":len(specs),**ost})
        mf_orig_sheets[freq]=mf_dfa(ctx["x"])

        factors,stab=estimate_seasonality(df)
        sf=np.array([factors.get(int(s),1.0) for s in ctx["slots"]],float)
        xfit=deseasonalized_returns(df,factors,True)
        print(f"[FIT {freq}m] Seasonal GARCH-t + FIGARCH-t | n={len(xfit):,}")
        gfit=fit_model(xfit,"garch_t"); ffit=fit_model(xfit,"figarch_t")
        marks,pbar,jdiag=calibrate_jump_marks(df,factors,gfit,bns,realjump,smoke=smoke)
        fit_rows += [_fitrow("Seasonal_GARCH_t",freq,gfit),_fitrow("Seasonal_FIGARCH_t",freq,ffit)]
        diag_rows.append({"Frequency_min":freq,"Diagnostic":"DayBlock_movable_fraction","Value":ctx["dayblock_movable_fraction"]})
        for k,v in jdiag.items(): diag_rows.append({"Frequency_min":freq,"Diagnostic":f"Jump_{k}","Value":v})
        param={"garch_t":gfit,"figarch_t":ffit,"sf":sf,"marks":marks,"pbar":pbar}
        contexts[freq]=(ctx,specs,min_pairs,param)

    original=pd.DataFrame(original_rows)
    print("\n[ORIGINAL]"); print(original[["Frequency_min","MFDFA_h_range","Pooled_MeanDeltaD_30d","Mean_TminusP","N_pairs"]].to_string(index=False))

    for freq in FREQUENCIES:
        ctx,specs,min_pairs,param=contexts[freq]
        for mech in MECHANISMS:
            print(f"\n[{freq}m] {mech}")
            rr=run_mechanism(mech,freq,ctx,specs,min_pairs,param,REPS,force=force)
            if len(rr)<max(2,int(.8*REPS)):
                warnings.warn(f"{mech}/{freq}m only {len(rr)}/{REPS} valid replicates")
            rr["Frequency_min"]=freq; surrogate_rows.append(rr)

    sur=pd.concat(surrogate_rows,ignore_index=True)
    tests=summarize_tests(original,sur)
    decisions=adjudicate(tests)

    # Summary by mechanism/frequency.
    summary=sur.groupby(["Mechanism","Frequency_min"]).agg(
        N=("Replicate","count"),
        MF_mean=("MFDFA_h_range","mean"), MF_sd=("MFDFA_h_range","std"),
        Delta_mean=("Pooled_MeanDeltaD_30d","mean"), Delta_sd=("Pooled_MeanDeltaD_30d","std"),
        TminusP_mean=("Mean_TminusP","mean"), AbsRet_ACF1_mean=("AbsRet_ACF1","mean")
    ).reset_index()

    params=pd.DataFrame([
        {"Parameter":"STAGE8_VERSION","Value":STAGE8_VERSION},
        {"Parameter":"RUN_TAG","Value":RUN_TAG},
        {"Parameter":"REPS_per_mechanism_frequency","Value":REPS},
        {"Parameter":"Primary_horizon_days","Value":PRIMARY_HORIZON},
        {"Parameter":"Frequencies","Value":str(FREQUENCIES)},
        {"Parameter":"Primary_endpoints","Value":str(PRIMARY_ENDPOINTS)},
        {"Parameter":"Minimum_pair_fraction","Value":MIN_PAIR_FRACTION},
        {"Parameter":"MF_q_values","Value":str(MF_Q_VALUES)},
        {"Parameter":"IAAFT_max_iter","Value":IAAFT_MAX_ITER},
        {"Parameter":"N_JOBS","Value":N_JOBS},
        {"Parameter":"Python","Value":platform.python_version()},
        {"Parameter":"Stage5_SHA256","Value":_file_sig(stage5_excel)},
        {"Parameter":"Stage7C_SHA256","Value":_file_sig(stage7c_excel) if stage7c_excel.exists() else "MISSING"},
    ])

    outfile=OUTDIR/"stage8_temporal_ablation_results.xlsx"
    with pd.ExcelWriter(outfile,engine="openpyxl") as w:
        original.to_excel(w,sheet_name="Observed_Endpoints",index=False)
        sur.to_excel(w,sheet_name="Surrogate_Replicates",index=False)
        summary.to_excel(w,sheet_name="Mechanism_Summary",index=False)
        tests.to_excel(w,sheet_name="Endpoint_Tests_FDR",index=False)
        decisions.to_excel(w,sheet_name="Preregistered_Decisions",index=False)
        pd.DataFrame(fit_rows).to_excel(w,sheet_name="Parametric_Fits",index=False)
        pd.DataFrame(diag_rows).to_excel(w,sheet_name="Diagnostics",index=False)
        params.to_excel(w,sheet_name="Stage8_Parameters",index=False)
        for freq,mf in mf_orig_sheets.items(): mf.to_excel(w,sheet_name=f"MFDFA_Original_{freq}m",index=False)
        if stage7c_excel.exists():
            try: pd.read_excel(stage7c_excel,sheet_name="Preregistered_Decisions").to_excel(w,sheet_name="Prior_7C_Decisions",index=False)
            except Exception: pass
    print("\nDECISIONS:"); print(decisions.to_string(index=False))
    print(f"\n{outfile}")
    return outfile


if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--stage5-excel")
    ap.add_argument("--stage7c-excel")
    ap.add_argument("--force",action="store_true")
    ap.add_argument("--smoke",action="store_true")
    a=ap.parse_args()
    main(a.stage5_excel,a.stage7c_excel,a.force,a.smoke)


In [ ]:
# Run all stages in order (resumable)
import os
import sys
import json
import shutil
import hashlib
import subprocess
import compileall
from pathlib import Path
from datetime import datetime, timezone

MODULE_NAMES = ["config.py", "stage1_data.py", "stage2_regime.py", "stage3_rolling.py",
                "stage4_multifractal.py", "stage5_mechanism.py", "stage6_dh_calibration.py",
                "stage6b_rolling_overlap.py", "stage7_structural_discrimination.py",
                "stage7b_jump_regime.py", "stage7c_garcht_jump.py", "stage8_temporal_ablation.py"]


def sha256_path(path):
    h = hashlib.sha256()
    with path.open("rb") as fp:
        for chunk in iter(lambda: fp.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


module_hashes = {m: sha256_path(PROJECT_DIR / m) for m in MODULE_NAMES}
assert compileall.compile_dir(str(PROJECT_DIR), quiet=1, maxlevels=0, force=True), "compilation error"

# Run fingerprint: stops old checkpoints from being mixed with changed code or data
signature = {
    "modules": module_hashes,
    "dataset": {k: {"sha256": v["sha256"], "mtime_ns": v["local_mtime_ns"]} for k, v in file_info.items()},
    "workers": int(os.environ["XAUUSD_N_JOBS"]),
    "environment": environment_snapshot,
}
RUN_ID = hashlib.sha256(json.dumps(signature, sort_keys=True).encode("utf-8")).hexdigest()
manifest_path = CONTROL_DIR / "run_fingerprint.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")).get("run_id") != RUN_ID:
        raise RuntimeError("The Drive checkpoints come from different code or data. "
                           "Use a different DRIVE_OUTPUT_DIR.")
else:
    if any(CHECKPOINTS.iterdir()) or any(RESULTS.iterdir()):
        raise RuntimeError("Drive already has results without a run fingerprint. Use a different folder.")
    manifest_path.write_text(json.dumps({"run_id": RUN_ID, "details": signature}, indent=2), encoding="utf-8")
print(f"[RUN] {RUN_ID}")

STAGE_SPECS = [
    ("stage1_data", "stage1_data.py", [], CHECKPOINTS / "stage1_data.pkl"),
    ("stage2_regime", "stage2_regime.py", [], CHECKPOINTS / "stage2_regime.pkl"),
    ("stage3_rolling", "stage3_rolling.py", [], CHECKPOINTS / "stage3_rolling.pkl"),
    ("stage4_multifractal", "stage4_multifractal.py", [], RESULTS / "statistical_results.xlsx"),
    ("stage5_mechanism", "stage5_mechanism.py", [], RESULTS / "stage5_mechanism_v3_results.xlsx"),
    ("stage6_dh_calibration", "stage6_dh_calibration.py",
     ["--stage5-excel", str(RESULTS / "stage5_mechanism_v3_results.xlsx")],
     RESULTS / "stage6" / "stage6_dh_calibration_results.xlsx"),
    ("stage6b_rolling_overlap", "stage6b_rolling_overlap.py", [],
     RESULTS / "stage6" / "stage6b_rolling_overlap_results.xlsx"),
    ("stage7_structural_discrimination", "stage7_structural_discrimination.py",
     ["--stage5-excel", str(RESULTS / "stage5_mechanism_v3_results.xlsx"),
      "--stage6-excel", str(RESULTS / "stage6" / "stage6_dh_calibration_results.xlsx")],
     RESULTS / "stage7" / "stage7_structural_discrimination_results.xlsx"),
    ("stage7b_jump_regime", "stage7b_jump_regime.py",
     ["--stage5-excel", str(RESULTS / "stage5_mechanism_v3_results.xlsx"),
      "--stage7-excel", str(RESULTS / "stage7" / "stage7_structural_discrimination_results.xlsx")],
     RESULTS / "stage7b" / "stage7b_jump_regime_results.xlsx"),
    ("stage7c_garcht_jump", "stage7c_garcht_jump.py",
     ["--stage5-excel", str(RESULTS / "stage5_mechanism_v3_results.xlsx"),
      "--stage7-excel", str(RESULTS / "stage7" / "stage7_structural_discrimination_results.xlsx"),
      "--stage7b-excel", str(RESULTS / "stage7b" / "stage7b_jump_regime_results.xlsx")],
     RESULTS / "stage7c" / "stage7c_garcht_jump_results.xlsx"),
    ("stage8_temporal_ablation", "stage8_temporal_ablation.py",
     ["--stage5-excel", str(RESULTS / "stage5_mechanism_v3_results.xlsx"),
      "--stage7c-excel", str(RESULTS / "stage7c" / "stage7c_garcht_jump_results.xlsx")],
     RESULTS / "stage8" / "stage8_temporal_ablation_results.xlsx"),
]

progress_file = CONTROL_DIR / "completed_stages.json"
progress = json.loads(progress_file.read_text(encoding="utf-8")) if progress_file.exists() else {"run_id": RUN_ID, "completed": {}}
if progress.get("run_id") != RUN_ID:
    raise RuntimeError("completed_stages.json belongs to a different run")


def free_ram_gib():
    for limit_path, used_path in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                                  ("/sys/fs/cgroup/memory/memory.limit_in_bytes",
                                   "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            limit = Path(limit_path).read_text().strip()
            if limit.isdigit() and 0 < int(limit) < 1 << 48:
                return max(0.0, (int(limit) - int(Path(used_path).read_text().strip())) / 1024**3)
        except (ValueError, OSError):
            pass
    return psutil.virtual_memory().available / 1024**3


for i, (stage, script, args, output) in enumerate(STAGE_SPECS, 1):
    print("\n" + "=" * 100)
    print(f"[{i}/{len(STAGE_SPECS)}] {stage}", flush=True)
    done = progress["completed"].get(stage)
    if done is not None:
        if not output.is_file() or sha256_path(output) != done["primary_sha256"]:
            raise RuntimeError(f"Result of {stage} is missing or was modified: {output}")
        print("already completed, skipping")
        continue

    free = free_ram_gib()
    print(f"Free RAM: ~{free:.1f} GiB", flush=True)
    if free < 2.5:
        raise RuntimeError(f"Low RAM before {stage}. Restart the runtime and run again to resume.")

    command = [sys.executable, "-u", str(PROJECT_DIR / script), *args]
    log_file = CONTROL_DIR / f"{stage}_execution.log"
    with log_file.open("a", encoding="utf-8") as log:
        log.write(f"\nSTART {datetime.now(timezone.utc).isoformat()} {command}\n")
        proc = subprocess.Popen(command, cwd=str(PROJECT_DIR), env=os.environ.copy(),
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
                log.write(line)
        except KeyboardInterrupt:
            proc.terminate()
            raise
        finally:
            proc.stdout.close()
        code = proc.wait()
        log.write(f"\nEXIT_CODE={code}\n")
    if code != 0:
        raise RuntimeError(f"{stage} failed (exit code {code}). Checkpoints stay in Drive; run again to resume.")
    if not output.is_file():
        raise RuntimeError(f"{stage} finished but did not produce {output}")

    if i == 4:
        for name in ("daily.csv", "rolling_df.csv"):
            shutil.copy2(PROJECT_DIR / name, RESULTS / name)

    progress["completed"][stage] = {"finished_utc": datetime.now(timezone.utc).isoformat(),
                                    "script": script, "primary_output": str(output),
                                    "primary_sha256": sha256_path(output)}
    tmp = progress_file.with_suffix(".json.partial")
    tmp.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    tmp.replace(progress_file)
    print(f"{stage} done", flush=True)

print("\nAll stages finished.")
print(f"Results: {RESULTS}")
print(f"Checkpoints: {CHECKPOINTS}")


## Notes

- Up to 3 worker processes are used, only if Colab provides 3 CPUs and at least 11 GiB of RAM; otherwise 2 or 1.
- Before each stage the notebook checks that at least 2.5 GiB of RAM is free.
- Input files are read from `DATA_FOLDER` in your Drive, or uploaded manually.
